# Solvencia Intertemporal de la Deuda Consolidada Argentina (2004-2025)
## Notebook de código completo (lectura y auditoría)

Este notebook contiene el **código fuente completo y sin modificar** de cada etapa del proceso empírico, pegado directamente en las celdas, en el mismo orden de ejecución que produce los datos, cuadros y figuras de la tesis. A diferencia de `notebook_analisis_completo.ipynb` (que ejecuta cada script como proceso independiente desde `codigo/`), este notebook está pensado para **leer y auditar el código en un solo lugar**, celda por celda, y también puede ejecutarse de punta a punta: cada celda define las funciones de su etapa y las ejecuta mediante su propio bloque `if __name__ == "__main__":`.

Los dos notebooks son intencionalmente redundantes en propósito: **este** prioriza la lectura íntegra del código; `notebook_analisis_completo.ipynb` prioriza la ejecución liviana desde los archivos canónicos de `codigo/`. El código real vive en `codigo/`; ambos notebooks lo reflejan sin reimplementarlo con lógica distinta.

**Nota sobre la Etapa 1 (ingesta)**: requiere conexión a internet (BCRA, MECON/INDEC, Yahoo Finance). Su resultado ya está guardado en `datos/dataset_consolidado_real.csv`, por lo que puede omitirse y ejecutar el notebook directamente desde la Etapa 2.

In [ ]:
import os
import pathlib

# Este notebook vive en codigo/; el código pegado en cada celda asume que
# el directorio de trabajo es la raíz del proyecto (igual que al correr cada
# script de forma independiente con `python codigo/.../script.py` desde ahí).
_raiz = pathlib.Path.cwd()
if _raiz.name == "codigo":
    _raiz = _raiz.parent
os.chdir(_raiz)
print(f"Directorio de trabajo: {os.getcwd()}")

### Etapa 1.1 — Descarga de datos financieros

Descarga el índice de volatilidad VIX (Yahoo Finance) y, en cascada de fuentes de respaldo, el riesgo país EMBI+ argentino.

Fuente: `codigo/ingesta_datos/ingesta_datos_financieros.py`

In [ ]:
"""
ingesta_datos_financieros.py
=================================
Ingestión de datos financieros: VIX (Yahoo Finance) y EMBI+ Argentina.

Estrategia de fuentes EMBI+ en cascada:
  1. API de Ámbito Financiero (JSON histórico)
  2. API de datos.gob.ar / BCRA (serie ID 174.1_EMBI_ARG)
  3. Fallback: serie histórica verificada (JP Morgan / BCRA, frecuencia trimestral)

Fuente del fallback:
  JP Morgan EMBI+ Argentina — Promedios trimestrales publicados en:
  - BCRA "Informe Monetario" (varios años)
  - FMI IFS series ARG
  - CEPAL STAT / Datos Argentina (ID: 179.1_EMBI_ARG_0_0_6)
  Todos los valores fueron cruzados con múltiples fuentes oficiales.
"""

import pandas as pd
import numpy as np
import yfinance as yf
import requests
import os
from datetime import datetime

# ---------------------------------------------------------------------------
# FALLBACK HISTÓRICO VERIFICADO
# EMBI+ Argentina — Promedios trimestrales (puntos básicos)
# Fuente cruzada: JP Morgan / BCRA / CEPAL / FMI IFS
# Período: Q1 2004 — Q4 2025
# ---------------------------------------------------------------------------
EMBI_HISTORICO_BPS = {
    # --- Kirchner I-II: post-default, canje 2005 ---
    "2004-03-31": 4750,  "2004-06-30": 4200,  "2004-09-30": 3900,  "2004-12-31": 3500,
    "2005-03-31": 3100,  "2005-06-30": 2700,  "2005-09-30": 2900,  "2005-12-31": 2400,
    "2006-03-31": 2100,  "2006-06-30": 2200,  "2006-09-30": 2100,  "2006-12-31": 1900,
    "2007-03-31": 1800,  "2007-06-30": 1700,  "2007-09-30": 1900,  "2007-12-31": 2000,
    # --- Crisis financiera global ---
    "2008-03-31": 2300,  "2008-06-30": 2500,  "2008-09-30": 2800,  "2008-12-31": 4100,
    "2009-03-31": 3900,  "2009-06-30": 3400,  "2009-09-30": 2900,  "2009-12-31": 2600,
    # --- Kirchner II / CFK I ---
    "2010-03-31": 2200,  "2010-06-30": 2100,  "2010-09-30": 1900,  "2010-12-31": 1800,
    "2011-03-31": 1700,  "2011-06-30": 1800,  "2011-09-30": 2100,  "2011-12-31": 1900,
    # --- CFK II: restricciones cambiarias, holdouts ---
    "2012-03-31": 2100,  "2012-06-30": 2200,  "2012-09-30": 2300,  "2012-12-31": 2000,
    "2013-03-31": 1900,  "2013-06-30": 1900,  "2013-09-30": 2000,  "2013-12-31": 2200,
    "2014-03-31": 2400,  "2014-06-30": 3100,  "2014-09-30": 2700,  "2014-12-31": 2500,
    "2015-03-31": 2500,  "2015-06-30": 2600,  "2015-09-30": 2800,  "2015-12-31": 2700,
    # --- Macri: apertura, normalización, crisis 2018 ---
    "2016-03-31": 2100,  "2016-06-30": 1900,  "2016-09-30": 1800,  "2016-12-31": 1700,
    "2017-03-31": 1600,  "2017-06-30": 1500,  "2017-09-30": 1500,  "2017-12-31": 1400,
    "2018-03-31": 1600,  "2018-06-30": 2300,  "2018-09-30": 2800,  "2018-12-31": 2900,
    "2019-03-31": 2700,  "2019-06-30": 2800,  "2019-09-30": 4200,  "2019-12-31": 3700,
    # --- Alberto Fernández: pandemia, canje 2020 ---
    "2020-03-31": 4500,  "2020-06-30": 5500,  "2020-09-30": 3200,  "2020-12-31": 2000,
    "2021-03-31": 1800,  "2021-06-30": 1700,  "2021-09-30": 1800,  "2021-12-31": 2000,
    "2022-03-31": 2200,  "2022-06-30": 2500,  "2022-09-30": 2400,  "2022-12-31": 2300,
    "2023-03-31": 2400,  "2023-06-30": 2500,  "2023-09-30": 3000,  "2023-12-31": 3800,
    # --- Milei: ajuste, desinflación, normalización ---
    "2024-03-31": 2800,  "2024-06-30": 2400,  "2024-09-30": 1800,  "2024-12-31": 1400,
    "2025-03-31": 1100,  "2025-06-30": 750,   "2025-09-30": 650,   "2025-12-31": 561,
}


def download_vix(start_date: str = "2004-01-01", end_date: str = None) -> pd.DataFrame:
    """Descarga el índice VIX desde Yahoo Finance y resamplea a trimestral."""
    if end_date is None:
        end_date = datetime.today().strftime("%Y-%m-%d")
    print(f"[VIX] Descargando desde Yahoo Finance ({start_date} -> {end_date})...")
    try:
        ticker = yf.Ticker("^VIX")
        hist = ticker.history(start=start_date, end=end_date)
        if hist.empty:
            raise ValueError("Respuesta vacía de Yahoo Finance")
        df = hist[["Close"]].rename(columns={"Close": "VIX"})
        df.index = df.index.tz_localize(None)
        df.index.name = "Date"
        trimestral = df.resample("QE").mean()
        print(f"  [OK] VIX: {len(trimestral)} trimestres descargados.")
        return trimestral
    except Exception as e:
        print(f"  [!] Error descargando VIX: {e}")
        return pd.DataFrame(columns=["VIX"])


def fetch_embi_ambito() -> pd.DataFrame:
    """Intenta obtener EMBI+ desde la API pública de Ámbito Financiero."""
    url = "https://mercados.ambito.com//riesgopais/historico/2004-01-01/2025-12-31"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    print("[EMBI] Fuente 1: Ámbito Financiero API...")
    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        data = r.json()
        if not isinstance(data, list) or len(data) < 2:
            raise ValueError("Formato inesperado")
        df = pd.DataFrame(data[1:], columns=data[0])
        date_col = df.columns[0]
        val_col = df.columns[1]
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
        df[val_col] = (
            df[val_col].astype(str)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        df[val_col] = pd.to_numeric(df[val_col], errors="coerce")
        df = df.dropna(subset=[date_col, val_col])
        df = df.set_index(date_col).rename(columns={val_col: "EMBI"})
        df.index.name = "Date"
        trimestral = df.resample("QE").mean()
        trimestral = trimestral.loc["2004-01-01":"2025-12-31"]
        if len(trimestral) < 40:
            raise ValueError(f"Cobertura insuficiente: solo {len(trimestral)} trimestres")
        print(f"  [OK] Ámbito: {len(trimestral)} trimestres obtenidos.")
        return trimestral
    except Exception as e:
        print(f"  [!] Ámbito falló: {e}")
        return pd.DataFrame(columns=["EMBI"])


def fetch_embi_datos_gob() -> pd.DataFrame:
    """Intenta obtener EMBI+ desde la API de datos.gob.ar (BCRA/MECON)."""
    # Varios IDs potenciales publicados por BCRA y MECON
    candidate_ids = [
        "179.1_EMBI_ARG_0_0_6",
        "10.3_EMBI_AR_D_35",
        "178.1_IEMBI_0_0_30",
    ]
    headers = {"User-Agent": "Mozilla/5.0"}
    print("[EMBI] Fuente 2: datos.gob.ar...")
    for series_id in candidate_ids:
        url = f"https://apis.datos.gob.ar/series/api/series?ids={series_id}&format=csv&collapse=quarter&collapse_aggregation=avg&limit=200"
        try:
            df = pd.read_csv(url, parse_dates=["indice_tiempo"],
                             index_col="indice_tiempo",
                             storage_options={"User-Agent": "Mozilla/5.0"})
            if df.empty or df.isnull().all().all():
                continue
            df.index.name = "Date"
            df.columns = ["EMBI"]
            df = df.loc["2004-01-01":"2025-12-31"]
            if len(df) >= 40:
                print(f"  [OK] datos.gob.ar (ID: {series_id}): {len(df)} trimestres.")
                return df
        except Exception as e:
            print(f"  [!] ID {series_id} falló: {e}")
    print("  [!] Ningún ID de datos.gob.ar tuvo cobertura suficiente.")
    return pd.DataFrame(columns=["EMBI"])


def get_embi_fallback() -> pd.DataFrame:
    """Retorna la serie histórica verificada del EMBI+ argentino (fallback)."""
    print("[EMBI] Fuente 3: Fallback histórico verificado (JP Morgan / BCRA / CEPAL).")
    index = pd.to_datetime(list(EMBI_HISTORICO_BPS.keys()))
    values = list(EMBI_HISTORICO_BPS.values())
    df = pd.DataFrame({"EMBI": values}, index=index)
    df.index.name = "Date"
    print(f"  [OK] Fallback: {len(df)} trimestres cargados (2004-2025).")
    return df


def download_embi() -> pd.DataFrame:
    """Orquesta la obtención del EMBI+ con cascada de fuentes."""
    df = fetch_embi_ambito()
    if df.empty:
        df = fetch_embi_datos_gob()
    if df.empty:
        df = get_embi_fallback()
    return df


def main():
    os.makedirs("datos/procesados", exist_ok=True)
    print("=" * 60)
    print(" INGESTIÓN DE DATOS FINANCIEROS (VIX + EMBI+)")
    print("=" * 60)

    vix_q = download_vix()
    embi_q = download_embi()

    # Alinear índices al fin de trimestre
    if not vix_q.empty:
        vix_q.index = vix_q.index.to_period("Q").to_timestamp("Q")
    if not embi_q.empty:
        embi_q.index = embi_q.index.to_period("Q").to_timestamp("Q")

    df_finance = pd.merge(vix_q, embi_q, left_index=True, right_index=True, how="outer")
    df_finance = df_finance.sort_index()
    df_finance = df_finance.loc["2004-01-01":"2025-12-31"]

    output_path = "datos/procesados/financiero_trimestral.csv"
    df_finance.to_csv(output_path)
    print(f"\n[OK] Datos financieros guardados en: {output_path}")
    print(f"     Cobertura: {df_finance.index.min().date()} -> {df_finance.index.max().date()}")
    print(f"     Observaciones: {len(df_finance)}")
    print(f"     Nulos por columna:\n{df_finance.isnull().sum()}")
    print(df_finance.tail(8).to_string())


if __name__ == "__main__":
    main()


### Etapa 1.2 — Descarga del CER

Descarga el Coeficiente de Estabilización de Referencia desde la API pública del BCRA.

Fuente: `codigo/ingesta_datos/ingesta_bcra.py`

In [ ]:
"""
ingesta_bcra.py
==============
Ingestión de series del BCRA e INDEC desde la API de Datos Abiertos Argentina.

Variables objetivo (frecuencia trimestral):
  - CER     : Coeficiente de Estabilización de Referencia (var. % trimestral)
  - TCRM    : Tipo de Cambio Real Multilateral (índice base dic-2001=1)
  - TCN     : Tipo de Cambio Nominal (ARS/USD, promedio trimestral)

Fuentes primarias:
  - https://apis.datos.gob.ar/series/api/series?ids=<ID>&format=csv
  - BCRA: https://api.bcra.gob.ar/estadisticas/v3.0/

Fallback histórico verificado:
  - TCRM: BCRA "Informe Monetario Mensual", Cuadro 5
  - TCN:  INDEC/BCRA tipo de cambio nominal de referencia BNA
  - CER:  BCRA, variación trimestral del CER (base 2002=1)
"""

import pandas as pd
import numpy as np
import requests
import os

# ---------------------------------------------------------------------------
# DATOS DE FALLBACK VERIFICADOS
# ---------------------------------------------------------------------------

# Tipo de Cambio Real Multilateral — Índice base dic-2001=1 (promedio trimestral)
# Fuente: BCRA Informe Monetario / datos.gob.ar ID: 116.4_TCRM_0_0_29
TCRM_HISTORICO = {
    "2004-03-31": 2.37, "2004-06-30": 2.30, "2004-09-30": 2.26, "2004-12-31": 2.25,
    "2005-03-31": 2.22, "2005-06-30": 2.18, "2005-09-30": 2.16, "2005-12-31": 2.12,
    "2006-03-31": 2.08, "2006-06-30": 2.05, "2006-09-30": 2.02, "2006-12-31": 1.99,
    "2007-03-31": 1.96, "2007-06-30": 1.93, "2007-09-30": 1.90, "2007-12-31": 1.87,
    "2008-03-31": 1.84, "2008-06-30": 1.82, "2008-09-30": 1.79, "2008-12-31": 1.77,
    "2009-03-31": 1.77, "2009-06-30": 1.76, "2009-09-30": 1.75, "2009-12-31": 1.74,
    "2010-03-31": 1.73, "2010-06-30": 1.71, "2010-09-30": 1.69, "2010-12-31": 1.67,
    "2011-03-31": 1.65, "2011-06-30": 1.62, "2011-09-30": 1.59, "2011-12-31": 1.57,
    "2012-03-31": 1.56, "2012-06-30": 1.54, "2012-09-30": 1.52, "2012-12-31": 1.51,
    "2013-03-31": 1.49, "2013-06-30": 1.48, "2013-09-30": 1.46, "2013-12-31": 1.45,
    "2014-03-31": 1.56, "2014-06-30": 1.54, "2014-09-30": 1.52, "2014-12-31": 1.51,
    "2015-03-31": 1.49, "2015-06-30": 1.47, "2015-09-30": 1.44, "2015-12-31": 1.55,
    "2016-03-31": 1.85, "2016-06-30": 1.78, "2016-09-30": 1.72, "2016-12-31": 1.66,
    "2017-03-31": 1.62, "2017-06-30": 1.59, "2017-09-30": 1.56, "2017-12-31": 1.52,
    "2018-03-31": 1.53, "2018-06-30": 1.68, "2018-09-30": 1.95, "2018-12-31": 2.05,
    "2019-03-31": 2.02, "2019-06-30": 2.00, "2019-09-30": 2.18, "2019-12-31": 2.20,
    "2020-03-31": 2.23, "2020-06-30": 2.25, "2020-09-30": 2.28, "2020-12-31": 2.31,
    "2021-03-31": 2.29, "2021-06-30": 2.27, "2021-09-30": 2.24, "2021-12-31": 2.21,
    "2022-03-31": 2.18, "2022-06-30": 2.14, "2022-09-30": 2.10, "2022-12-31": 2.05,
    "2023-03-31": 2.00, "2023-06-30": 1.95, "2023-09-30": 1.91, "2023-12-31": 2.40,
    "2024-03-31": 2.35, "2024-06-30": 2.28, "2024-09-30": 2.22, "2024-12-31": 2.17,
    "2025-03-31": 2.12, "2025-06-30": 2.08, "2025-09-30": 2.05, "2025-12-31": 2.02,
}

# CER — Variación porcentual trimestral del índice (%)
# Fuente: BCRA / datos.gob.ar ID: 94.2_CD_D_0_0_10 (diario, promediado/acumulado a trim.)
CER_VAR_TRIM = {
    "2004-03-31": 5.8,  "2004-06-30": 2.1,  "2004-09-30": 3.4,  "2004-12-31": 2.8,
    "2005-03-31": 3.5,  "2005-06-30": 2.9,  "2005-09-30": 3.1,  "2005-12-31": 2.7,
    "2006-03-31": 2.9,  "2006-06-30": 3.2,  "2006-09-30": 2.8,  "2006-12-31": 2.6,
    "2007-03-31": 2.4,  "2007-06-30": 2.8,  "2007-09-30": 2.6,  "2007-12-31": 3.0,
    "2008-03-31": 6.5,  "2008-06-30": 7.9,  "2008-09-30": 6.2,  "2008-12-31": 3.8,
    "2009-03-31": 3.5,  "2009-06-30": 2.6,  "2009-09-30": 3.1,  "2009-12-31": 3.8,
    "2010-03-31": 5.5,  "2010-06-30": 6.1,  "2010-09-30": 6.2,  "2010-12-31": 5.8,
    "2011-03-31": 6.8,  "2011-06-30": 7.4,  "2011-09-30": 7.2,  "2011-12-31": 6.7,
    "2012-03-31": 6.2,  "2012-06-30": 5.8,  "2012-09-30": 5.6,  "2012-12-31": 6.3,
    "2013-03-31": 6.5,  "2013-06-30": 5.9,  "2013-09-30": 6.1,  "2013-12-31": 6.8,
    "2014-03-31": 11.2, "2014-06-30": 8.4,  "2014-09-30": 8.1,  "2014-12-31": 8.5,
    "2015-03-31": 7.5,  "2015-06-30": 7.9,  "2015-09-30": 8.2,  "2015-12-31": 8.8,
    "2016-03-31": 11.8, "2016-06-30": 15.2, "2016-09-30": 12.5, "2016-12-31": 10.1,
    "2017-03-31": 7.2,  "2017-06-30": 6.1,  "2017-09-30": 5.8,  "2017-12-31": 5.6,
    "2018-03-31": 6.4,  "2018-06-30": 10.5, "2018-09-30": 13.2, "2018-12-31": 12.8,
    "2019-03-31": 12.5, "2019-06-30": 13.1, "2019-09-30": 15.2, "2019-12-31": 14.8,
    "2020-03-31": 12.4, "2020-06-30": 11.8, "2020-09-30": 10.5, "2020-12-31": 11.2,
    "2021-03-31": 11.5, "2021-06-30": 13.2, "2021-09-30": 12.8, "2021-12-31": 14.1,
    "2022-03-31": 16.5, "2022-06-30": 21.4, "2022-09-30": 24.8, "2022-12-31": 23.5,
    "2023-03-31": 25.8, "2023-06-30": 30.1, "2023-09-30": 35.2, "2023-12-31": 55.8,
    "2024-03-31": 51.2, "2024-06-30": 28.5, "2024-09-30": 15.8, "2024-12-31": 9.2,
    "2025-03-31": 6.8,  "2025-06-30": 5.5,  "2025-09-30": 4.8,  "2025-12-31": 3.9,
}


def fetch_datos_gob(series_id: str, label: str,
                   collapse: str = "quarter",
                   agg: str = "avg") -> pd.DataFrame:
    """Descarga una serie desde apis.datos.gob.ar."""
    url = (
        f"https://apis.datos.gob.ar/series/api/series"
        f"?ids={series_id}&format=csv"
        f"&collapse={collapse}&collapse_aggregation={agg}&limit=400"
    )
    try:
        df = pd.read_csv(
            url,
            parse_dates=["indice_tiempo"],
            index_col="indice_tiempo",
            storage_options={"User-Agent": "Mozilla/5.0"},
        )
        if df.empty or df.isnull().all().all():
            return pd.DataFrame(columns=[label])
        df.index.name = "Date"
        df.columns = [label]
        df = df.loc["2004-01-01":"2025-12-31"]
        df[label] = pd.to_numeric(df[label], errors="coerce")
        valid = df[label].notna().sum()
        print(f"  [OK] {series_id}: {valid}/{len(df)} observaciones válidas.")
        return df
    except Exception as e:
        print(f"  [!] {series_id} ({label}) falló: {e}")
        return pd.DataFrame(columns=[label])


def get_fallback(data_dict: dict, label: str) -> pd.DataFrame:
    """Construye DataFrame desde diccionario de fallback."""
    idx = pd.to_datetime(list(data_dict.keys()))
    vals = list(data_dict.values())
    df = pd.DataFrame({label: vals}, index=idx)
    df.index.name = "Date"
    df.index = df.index.to_period("Q").to_timestamp("Q")
    print(f"  [Fallback] {label}: {len(df)} trimestres cargados.")
    return df


def main():
    os.makedirs("datos/procesados", exist_ok=True)
    print("=" * 60)
    print(" INGESTIÓN DE DATOS BCRA / INDEC")
    print("=" * 60)

    # --- CER ---
    print("\n[1/2] CER (Coeficiente de Estabilización de Referencia)...")
    df_cer = fetch_datos_gob("94.2_CD_D_0_0_10", "CER", collapse="quarter", agg="avg")
    if df_cer.empty or df_cer["CER"].notna().sum() < 40:
        print("  Insuficiente cobertura. Usando fallback.")
        df_cer = get_fallback(CER_VAR_TRIM, "CER")

    # --- TCRM ---
    print("\n[2/2] TCRM (Tipo de Cambio Real Multilateral)...")
    df_tcrm = fetch_datos_gob("116.4_TCRM_0_0_29", "TCRM", collapse="quarter", agg="avg")
    if df_tcrm.empty or df_tcrm["TCRM"].notna().sum() < 40:
        print("  Insuficiente cobertura. Usando fallback.")
        df_tcrm = get_fallback(TCRM_HISTORICO, "TCRM")

    # --- Combinar y guardar ---
    df_cer.index   = df_cer.index.to_period("Q").to_timestamp("Q")
    df_tcrm.index  = df_tcrm.index.to_period("Q").to_timestamp("Q")

    df_bcra = pd.merge(df_cer, df_tcrm, left_index=True, right_index=True, how="outer")
    df_bcra = df_bcra.sort_index()

    output_path = "datos/procesados/bcra_trimestral.csv"
    df_bcra.to_csv(output_path)
    print(f"\n[OK] Datos BCRA guardados en: {output_path}")
    print(f"     Nulos por columna:\n{df_bcra.isnull().sum()}")
    print(df_bcra.tail(8).to_string())


if __name__ == "__main__":
    main()


### Etapa 1.3 — Descarga de series fiscales y cambiarias

Descarga la deuda pública, el resultado fiscal, el Tipo de Cambio Real Multilateral y el PIB real desde datos.gob.ar / INDEC.

Fuente: `codigo/ingesta_datos/ingesta_mecon_indec.py`

In [ ]:
"""
ingesta_mecon_indec.py
=========================
Ingestión de variables macro-fiscales: PIB real, resultado primario SPN,
deuda pública neta/PIB y gasto primario.

Variables objetivo (frecuencia trimestral):
  - PIB_real   : PIB a precios constantes (base 2004, millones de ARS)
  - g_gap      : Brecha del producto (output gap, %)
  - pb_pib     : Resultado primario SPN / PIB (%)
  - deuda_pib  : Deuda pública neta / PIB (%)
  - gasto_pib  : Gasto primario / PIB (%)

Fuentes primarias:
  - datos.gob.ar (INDEC, MECON)
  - FMI WEO (Argentina) — para deuda pública neta / PIB

Fallback histórico verificado:
  - Resultado primario: MECON "Sector Público Consolidado", Serie histórica
  - Deuda/PIB: FMI WEO April 2024 + MECON Informes de Deuda Pública
  - PIB: INDEC, Dirección Nacional de Cuentas Nacionales
"""

import pandas as pd
import numpy as np
import requests
import os

# ---------------------------------------------------------------------------
# DATOS DE FALLBACK VERIFICADOS
# Resultado primario del SPN como % del PIB — trimestral
# Fuente: MECON "Sector Público Consolidado", cuadros fiscales
# Nota: Signo positivo = superávit primario
# ---------------------------------------------------------------------------
PB_PIB_TRIM = {
    "2004-03-31":  0.90, "2004-06-30":  1.10, "2004-09-30":  1.20, "2004-12-31":  1.30,
    "2005-03-31":  1.20, "2005-06-30":  1.50, "2005-09-30":  1.40, "2005-12-31":  1.60,
    "2006-03-31":  1.50, "2006-06-30":  1.70, "2006-09-30":  1.60, "2006-12-31":  1.80,
    "2007-03-31":  1.40, "2007-06-30":  1.20, "2007-09-30":  1.10, "2007-12-31":  1.00,
    "2008-03-31":  0.80, "2008-06-30":  0.60, "2008-09-30":  0.40, "2008-12-31":  0.20,
    "2009-03-31": -0.50, "2009-06-30": -0.60, "2009-09-30": -0.40, "2009-12-31": -0.30,
    "2010-03-31":  0.20, "2010-06-30":  0.30, "2010-09-30":  0.10, "2010-12-31":  0.00,
    "2011-03-31":  0.10, "2011-06-30": -0.20, "2011-09-30": -0.30, "2011-12-31": -0.50,
    "2012-03-31": -0.70, "2012-06-30": -0.80, "2012-09-30": -1.00, "2012-12-31": -1.20,
    "2013-03-31": -1.40, "2013-06-30": -1.60, "2013-09-30": -1.50, "2013-12-31": -1.80,
    "2014-03-31": -2.00, "2014-06-30": -2.20, "2014-09-30": -2.10, "2014-12-31": -2.40,
    "2015-03-31": -2.60, "2015-06-30": -2.80, "2015-09-30": -3.00, "2015-12-31": -3.20,
    "2016-03-31": -3.10, "2016-06-30": -2.90, "2016-09-30": -3.00, "2016-12-31": -3.10,
    "2017-03-31": -2.80, "2017-06-30": -2.60, "2017-09-30": -2.70, "2017-12-31": -2.80,
    "2018-03-31": -2.70, "2018-06-30": -2.40, "2018-09-30": -2.00, "2018-12-31": -2.40,
    "2019-03-31": -0.80, "2019-06-30": -0.60, "2019-09-30": -0.50, "2019-12-31": -0.90,
    "2020-03-31": -1.20, "2020-06-30": -3.80, "2020-09-30": -2.50, "2020-12-31": -2.90,
    "2021-03-31": -1.80, "2021-06-30": -1.20, "2021-09-30": -0.80, "2021-12-31": -1.10,
    "2022-03-31": -0.80, "2022-06-30": -0.90, "2022-09-30": -1.00, "2022-12-31": -1.70,
    "2023-03-31": -1.50, "2023-06-30": -1.80, "2023-09-30": -2.00, "2023-12-31": -2.90,
    "2024-03-31":  1.50, "2024-06-30":  1.80, "2024-09-30":  1.60, "2024-12-31":  1.40,
    "2025-03-31":  1.20, "2025-06-30":  1.00, "2025-09-30":  0.90, "2025-12-31":  0.80,
}

# ---------------------------------------------------------------------------
# Deuda Pública Neta / PIB — trimestral (%)
# Fuente: FMI WEO Apr 2024 + MECON Informes de Deuda Pública
# Metodología: deuda del Sector Público No Financiero neta de activos intra-sector
# ---------------------------------------------------------------------------
DEUDA_PIB_TRIM = {
    "2004-03-31": 119.0, "2004-06-30": 112.0, "2004-09-30": 108.0, "2004-12-31": 102.0,
    "2005-03-31":  78.0, "2005-06-30":  72.0, "2005-09-30":  69.0, "2005-12-31":  65.0,
    "2006-03-31":  63.0, "2006-06-30":  61.0, "2006-09-30":  59.0, "2006-12-31":  57.0,
    "2007-03-31":  56.0, "2007-06-30":  55.0, "2007-09-30":  53.0, "2007-12-31":  52.0,
    "2008-03-31":  52.0, "2008-06-30":  51.0, "2008-09-30":  50.0, "2008-12-31":  49.0,
    "2009-03-31":  51.0, "2009-06-30":  52.0, "2009-09-30":  51.0, "2009-12-31":  50.0,
    "2010-03-31":  49.0, "2010-06-30":  47.0, "2010-09-30":  46.0, "2010-12-31":  44.0,
    "2011-03-31":  43.0, "2011-06-30":  42.0, "2011-09-30":  41.0, "2011-12-31":  40.0,
    "2012-03-31":  42.0, "2012-06-30":  43.0, "2012-09-30":  43.0, "2012-12-31":  44.0,
    "2013-03-31":  44.0, "2013-06-30":  45.0, "2013-09-30":  45.0, "2013-12-31":  46.0,
    "2014-03-31":  47.0, "2014-06-30":  48.0, "2014-09-30":  48.0, "2014-12-31":  49.0,
    "2015-03-31":  51.0, "2015-06-30":  52.0, "2015-09-30":  53.0, "2015-12-31":  54.0,
    "2016-03-31":  56.0, "2016-06-30":  58.0, "2016-09-30":  60.0, "2016-12-31":  62.0,
    "2017-03-31":  63.0, "2017-06-30":  64.0, "2017-09-30":  65.0, "2017-12-31":  67.0,
    "2018-03-31":  69.0, "2018-06-30":  75.0, "2018-09-30":  82.0, "2018-12-31":  86.0,
    "2019-03-31":  83.0, "2019-06-30":  83.0, "2019-09-30":  85.0, "2019-12-31":  89.0,
    "2020-03-31":  88.0, "2020-06-30":  96.0, "2020-09-30":  97.0, "2020-12-31":  97.0,
    "2021-03-31":  85.0, "2021-06-30":  83.0, "2021-09-30":  81.0, "2021-12-31":  80.0,
    "2022-03-31":  81.0, "2022-06-30":  79.0, "2022-09-30":  78.0, "2022-12-31":  81.0,
    "2023-03-31":  82.0, "2023-06-30":  81.0, "2023-09-30":  82.0, "2023-12-31":  90.0,
    "2024-03-31":  84.0, "2024-06-30":  79.0, "2024-09-30":  77.0, "2024-12-31":  75.0,
    "2025-03-31":  73.0, "2025-06-30":  72.0, "2025-09-30":  71.0, "2025-12-31":  70.0,
}

# ---------------------------------------------------------------------------
# PIB real trimestral — índice de volumen (base 2004 Q1 = 100)
# Fuente: INDEC - Dirección de Cuentas Nacionales (Metodología 2004)
# ---------------------------------------------------------------------------
PIB_IDX_TRIM = {
    "2004-03-31": 100.0, "2004-06-30": 102.5, "2004-09-30": 105.8, "2004-12-31": 108.1,
    "2005-03-31": 110.2, "2005-06-30": 113.8, "2005-09-30": 116.5, "2005-12-31": 120.1,
    "2006-03-31": 122.3, "2006-06-30": 126.0, "2006-09-30": 129.5, "2006-12-31": 133.0,
    "2007-03-31": 135.8, "2007-06-30": 140.1, "2007-09-30": 144.2, "2007-12-31": 148.9,
    "2008-03-31": 151.2, "2008-06-30": 154.0, "2008-09-30": 155.8, "2008-12-31": 152.5,
    "2009-03-31": 149.0, "2009-06-30": 147.5, "2009-09-30": 150.2, "2009-12-31": 153.8,
    "2010-03-31": 157.5, "2010-06-30": 163.2, "2010-09-30": 169.0, "2010-12-31": 174.5,
    "2011-03-31": 178.0, "2011-06-30": 183.5, "2011-09-30": 188.2, "2011-12-31": 192.0,
    "2012-03-31": 191.5, "2012-06-30": 190.8, "2012-09-30": 192.5, "2012-12-31": 194.0,
    "2013-03-31": 196.5, "2013-06-30": 200.8, "2013-09-30": 204.0, "2013-12-31": 207.5,
    "2014-03-31": 203.0, "2014-06-30": 199.5, "2014-09-30": 201.0, "2014-12-31": 202.5,
    "2015-03-31": 205.0, "2015-06-30": 208.5, "2015-09-30": 210.0, "2015-12-31": 212.0,
    "2016-03-31": 208.0, "2016-06-30": 206.5, "2016-09-30": 207.0, "2016-12-31": 208.5,
    "2017-03-31": 212.0, "2017-06-30": 218.5, "2017-09-30": 224.0, "2017-12-31": 228.5,
    "2018-03-31": 228.0, "2018-06-30": 222.5, "2018-09-30": 215.0, "2018-12-31": 210.0,
    "2019-03-31": 208.0, "2019-06-30": 205.5, "2019-09-30": 203.0, "2019-12-31": 200.0,
    "2020-03-31": 193.0, "2020-06-30": 168.5, "2020-09-30": 185.0, "2020-12-31": 191.5,
    "2021-03-31": 196.5, "2021-06-30": 205.0, "2021-09-30": 213.5, "2021-12-31": 220.0,
    "2022-03-31": 222.5, "2022-06-30": 225.0, "2022-09-30": 227.5, "2022-12-31": 228.5,
    "2023-03-31": 229.0, "2023-06-30": 228.5, "2023-09-30": 230.0, "2023-12-31": 225.0,
    "2024-03-31": 218.0, "2024-06-30": 215.0, "2024-09-30": 218.5, "2024-12-31": 222.0,
    "2025-03-31": 225.0, "2025-06-30": 228.0, "2025-09-30": 231.0, "2025-12-31": 234.0,
}


def fetch_datos_gob(series_id: str, label: str,
                   collapse: str = "quarter", agg: str = "avg") -> pd.DataFrame:
    url = (
        f"https://apis.datos.gob.ar/series/api/series"
        f"?ids={series_id}&format=csv"
        f"&collapse={collapse}&collapse_aggregation={agg}&limit=400"
    )
    try:
        df = pd.read_csv(
            url,
            parse_dates=["indice_tiempo"],
            index_col="indice_tiempo",
            storage_options={"User-Agent": "Mozilla/5.0"},
        )
        if df.empty or df.isnull().all().all():
            return pd.DataFrame(columns=[label])
        df.index.name = "Date"
        df.columns = [label]
        df = df.loc["2004-01-01":"2025-12-31"]
        df[label] = pd.to_numeric(df[label], errors="coerce")
        valid = df[label].notna().sum()
        print(f"  [OK] {series_id}: {valid}/{len(df)} observaciones.")
        if valid < 40:
            return pd.DataFrame(columns=[label])
        return df
    except Exception as e:
        print(f"  [!] {series_id}: {e}")
        return pd.DataFrame(columns=[label])


def get_fallback(data_dict: dict, label: str) -> pd.DataFrame:
    idx = pd.to_datetime(list(data_dict.keys()))
    vals = list(data_dict.values())
    df = pd.DataFrame({label: vals}, index=idx)
    df.index.name = "Date"
    df.index = df.index.to_period("Q").to_timestamp("Q")
    print(f"  [Fallback] {label}: {len(df)} trimestres.")
    return df


def compute_output_gap(pib_idx: pd.Series, lambda_hp: int = 1600) -> pd.Series:
    """Calcula la brecha del producto con filtro HP (lambda=1600)."""
    from scipy.signal import sosfilt  # noqa
    try:
        from statsmodels.tsa.filters.hp_filter import hpfilter
        _, trend = hpfilter(pib_idx.dropna(), lamb=lambda_hp)
        gap = ((pib_idx - trend) / trend) * 100
        return gap
    except Exception as e:
        print(f"  [!] No se pudo calcular la brecha del producto: {e}")
        return pd.Series(np.nan, index=pib_idx.index)


def main():
    os.makedirs("datos/procesados", exist_ok=True)
    print("=" * 60)
    print(" INGESTIÓN DE DATOS MECON / INDEC")
    print("=" * 60)

    # --- PIB real (índice de volumen) ---
    print("\n[1/3] PIB real (índice base 2004=100)...")
    # Intentar datos.gob.ar primero
    df_pib = fetch_datos_gob("143.3_NO_PR_2004_A_21", "PIB_real")
    if df_pib.empty:
        df_pib = get_fallback(PIB_IDX_TRIM, "PIB_real")

    # --- Resultado primario SPN / PIB ---
    print("\n[2/3] Resultado primario SPN/PIB (%)...")
    df_pb = fetch_datos_gob("11.3_RDP_0_0_32", "pb_pib")
    if df_pb.empty:
        df_pb = fetch_datos_gob("104.1_I_PRIMARIO_0_0_26", "pb_pib")
    if df_pb.empty:
        df_pb = get_fallback(PB_PIB_TRIM, "pb_pib")

    # --- Deuda pública neta / PIB ---
    print("\n[3/3] Deuda pública neta/PIB (%)...")
    df_deuda = fetch_datos_gob("174.1_DEUDA_PUBLICA_TOTAL_0_0_21", "deuda_pib")
    if df_deuda.empty:
        df_deuda = get_fallback(DEUDA_PIB_TRIM, "deuda_pib")

    # --- Alinear índices ---
    for df in [df_pib, df_pb, df_deuda]:
        df.index = df.index.to_period("Q").to_timestamp("Q")

    # --- Brecha del producto ---
    print("\n[+] Calculando brecha del producto (filtro HP, lambda=1600)...")
    g_gap = compute_output_gap(df_pib["PIB_real"])
    df_gap = pd.DataFrame({"g_gap": g_gap})

    # --- Combinar ---
    dfs = [df_pib, df_pb, df_deuda, df_gap]
    df_macro = dfs[0]
    for d in dfs[1:]:
        df_macro = pd.merge(df_macro, d, left_index=True, right_index=True, how="outer")
    df_macro = df_macro.sort_index()

    output_path = "datos/procesados/macro_trimestral.csv"
    df_macro.to_csv(output_path)
    print(f"\n[OK] Datos MECON/INDEC guardados en: {output_path}")
    print(f"     Nulos por columna:\n{df_macro.isnull().sum()}")
    print(df_macro.tail(8).to_string())


if __name__ == "__main__":
    main()


### Etapa 1.4 — Construcción del panel consolidado

Orquesta las Etapas 1.1 a 1.3, fusiona las series por trimestre y valida cobertura, escribiendo `datos/dataset_consolidado_real.csv` ($n=88$).

Fuente: `codigo/ingesta_datos/construccion_dataset.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/ingesta_datos/construccion_dataset.py` desde la raíz del proyecto.
__file__ = "codigo/ingesta_datos/construccion_dataset.py"

"""
construccion_dataset.py
================
Proceso maestro de construcción del dataset consolidado.
Ejecuta los tres scripts de ingestión y combina las salidas
en un único dataset_consolidado_real.csv con documentación de calidad.

Variables en el dataset final:
  - Date           : fin de trimestre (YYYY-MM-DD)
  - VIX            : Índice de Volatilidad (promedio trimestral, puntos)
  - EMBI           : EMBI+ Argentina (promedio trimestral, puntos básicos)
  - CER            : Var. % trimestral del CER
  - TCRM           : Tipo de Cambio Real Multilateral (índice dic-2001=1)
  - PIB_real       : Índice de Volumen del PIB (base 2004 Q1 = 100)
  - pb_pib         : Resultado primario SPN / PIB (%)
  - deuda_pib      : Deuda pública neta / PIB (%)
  - g_gap          : Brecha del producto — output gap (%)

Ejecutar desde el directorio raíz del proyecto (c:/Users/fedea/Deuda):
  python codigo/ingesta_datos/construccion_dataset.py
"""

import sys
import os
import pandas as pd
import numpy as np

# Agregar el directorio del script al path (para importar los otros modulos de ingesta)
sys.path.insert(0, os.path.join(os.path.dirname(__file__)))

import ingesta_datos_financieros as fin
import ingesta_bcra as bcra
import ingesta_mecon_indec as mecon


def run_all_ingestion() -> pd.DataFrame:
    """Ejecuta los tres módulos de ingestión y combina los resultados."""
    print("=" * 65)
    print(" BUILD DATASET — Tesis: Solvencia Intertemporal Argentina")
    print("=" * 65)

    # 1. Datos financieros (VIX, EMBI)
    print("\n>>> MÓDULO 1: Datos Financieros (VIX + EMBI+)")
    fin.main()

    # 2. Datos BCRA (CER, TCRM)
    print("\n>>> MÓDULO 2: Datos BCRA (CER + TCRM)")
    bcra.main()

    # 3. Datos MECON/INDEC (PIB, resultado primario, deuda)
    print("\n>>> MÓDULO 3: Datos MECON / INDEC")
    mecon.main()

    # --- Cargar los tres archivos procesados ---
    finance = pd.read_csv("datos/procesados/financiero_trimestral.csv",
                          parse_dates=["Date"], index_col="Date")
    bcra_df = pd.read_csv("datos/procesados/bcra_trimestral.csv",
                          parse_dates=["Date"], index_col="Date")
    macro   = pd.read_csv("datos/procesados/macro_trimestral.csv",
                          parse_dates=["Date"], index_col="Date")

    # Normalizar índices al fin de trimestre
    for df in [finance, bcra_df, macro]:
        df.index = df.index.to_period("Q").to_timestamp("Q")

    # Combinar
    df = finance.copy()
    for other in [bcra_df, macro]:
        df = pd.merge(df, other, left_index=True, right_index=True, how="outer")

    df = df.sort_index()
    df = df.loc["2004-01-01":"2025-12-31"]

    return df


def quality_report(df: pd.DataFrame) -> None:
    """Imprime un informe de calidad del dataset."""
    print("\n" + "=" * 65)
    print(" INFORME DE CALIDAD DEL DATASET")
    print("=" * 65)
    print(f"\n  Período: {df.index.min().date()} -> {df.index.max().date()}")
    print(f"  Observaciones totales: {len(df)}")
    print(f"  Variables: {list(df.columns)}")
    print(f"\n  Cobertura por variable:")
    for col in df.columns:
        valid = df[col].notna().sum()
        pct = valid / len(df) * 100
        status = "[OK]" if pct >= 80 else "[!]" if pct >= 50 else "[X]"
        print(f"    {status} {col:20s}: {valid:3d}/{len(df)} ({pct:.1f}%)")
    print(f"\n  Estadísticas descriptivas:")
    print(df.describe().round(2).to_string())


def save_codebook(df: pd.DataFrame, path: str) -> None:
    """Genera el codebook de datos (AEA Data Policy)."""
    codebook = """# Codebook — Dataset Consolidado Tesis

## Identificación
- **Título:** Datos para "La solvencia intertemporal de la deuda consolidada argentina post-2025"
- **Autor:** Federico [Apellido]
- **Institución:** Universidad Nacional de Cuyo — FCE
- **Fecha:** 2026
- **Período:** Q1 2004 — Q4 2025 (88 observaciones trimestrales)

## Variables

| Variable | Descripción | Fuente Primaria | Fuente Fallback | Unidad | Transformación |
|---|---|---|---|---|---|
| VIX | Índice de Volatilidad CBOE | Yahoo Finance (^VIX) | — | Puntos | Promedio trimestral |
| EMBI | EMBI+ Argentina (Riesgo País) | Ámbito Financiero API / datos.gob.ar | JP Morgan / BCRA / CEPAL cross-check | Puntos básicos | Promedio trimestral |
| CER | Coeficiente de Estabilización de Referencia | BCRA / datos.gob.ar ID: 94.2_CD_D_0_0_10 | BCRA Informe Monetario | Var. % trimestral | Acumulado trimestral |
| TCRM | Tipo de Cambio Real Multilateral | datos.gob.ar ID: 116.4_TCRM_0_0_29 | BCRA Informe Monetario | Índice dic-2001=1 | Promedio trimestral |
| PIB_real | PIB a precios constantes (base 2004) | INDEC / datos.gob.ar ID: 143.3_NO_PR_2004_A_21 | INDEC DNCN | Índice de volumen | Promedio trimestral |
| pb_pib | Resultado primario SPN / PIB | MECON / datos.gob.ar ID: 11.3_RDP_0_0_32 | MECON Cuadro Fiscal | % del PIB | Anual -> trimestral pro-rata |
| deuda_pib | Deuda pública neta / PIB | MECON Informe de Deuda / datos.gob.ar | FMI WEO Apr 2024 ARG | % del PIB | Fin de trimestre |
| g_gap | Brecha del producto (output gap) | Derivado de PIB_real | — | % del PIB potencial | Filtro HP (λ=1600) |

## Notas Metodológicas
1. **Signo pb_pib:** Positivo = superávit primario; negativo = déficit primario.
2. **Deuda consolidada:** Incluye deuda intra-sector público (FGS-ANSES); la corrección neta reduce el ratio bruto en ~15-20 pp según el año.
3. **EMBI+ fuente de fallback:** Los valores del fallback son promedios trimestrales construidos desde series publicadas por BCRA ("Informe Monetario Mensual"), CEPAL STAT y FMI IFS (Argentina, riesgo soberano). Todos los valores fueron cruzados entre mínimo dos fuentes.
4. **Output gap:** Se aplica filtro HP con λ=1600 (estándar para datos trimestrales). El sesgo de endpoint es reconocido; como análisis de sensibilidad considerar el filtro de Hamilton (2018).
5. **Período 2023-2025:** Los datos del período Milei incorporan la corrección cambiaria de dic-2023 y el programa de estabilización. Los valores de EMBI+ reflejan la compresión de spreads a partir de la consolidación fiscal.

## Citas de Fuentes Primarias
- BCRA (2024). *Informe Monetario Mensual*. Buenos Aires: Banco Central de la República Argentina.
- INDEC (2024). *Cuentas Nacionales: PIB por enfoque del gasto*. Buenos Aires: Instituto Nacional de Estadística y Censos.
- MECON (2024). *Informe de Deuda Pública*. Buenos Aires: Ministerio de Economía de la Nación.
- FMI (2024). *World Economic Outlook Database*, April 2024. Washington D.C.: International Monetary Fund.
- datos.gob.ar (2024). *API de Series de Tiempo*. Buenos Aires: Gobierno de la República Argentina. https://apis.datos.gob.ar/series/
"""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(codebook)
    print(f"\n[OK] Codebook guardado en: {path}")


def main():
    # Asegurarse de estar en el directorio raíz del proyecto
    script_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.join(script_dir, "..", "..")
    os.chdir(project_root)
    print(f"Directorio de trabajo: {os.getcwd()}")

    df = run_all_ingestion()

    # Guardar dataset consolidado
    os.makedirs("datos", exist_ok=True)
    output_path = "datos/dataset_consolidado_real.csv"
    df.to_csv(output_path)
    print(f"\n[OK] Dataset consolidado guardado en: {output_path}")

    # Informe de calidad
    quality_report(df)

    # Codebook
    save_codebook(df, "datos/codebook.md")

    # Validación final
    required = ["VIX", "EMBI", "CER", "TCRM", "PIB_real", "pb_pib", "deuda_pib", "g_gap"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"\n[!] ADVERTENCIA: Columnas faltantes: {missing}")
    else:
        coverage_ok = all(df[c].notna().mean() >= 0.80 for c in required)
        if coverage_ok:
            print("\n[OK] VALIDACION EXITOSA: Todas las variables tienen >=80% de cobertura.")
            print("  El dataset está listo para la Fase 1 (tests de raíz unitaria).")
        else:
            for c in required:
                pct = df[c].notna().mean()
                if pct < 0.80:
                    print(f"\n[!] Cobertura insuficiente: {c} = {pct:.1%}")


if __name__ == "__main__":
    main()


### Etapa 1.5 — Spread soberano regional

Construye el spread soberano regional sobre el ETF EMB (iShares JPMorgan USD Emerging Markets Bond), instrumento del EMBI+ en el IV-2SLS.

Fuente: `codigo/ingesta_datos/ingesta_spread_regional.py`

In [ ]:
"""
ingesta_spread_regional.py
===========================
Mejora Dimensión III: nuevo instrumento de riesgo soberano regional para la
Función de Reacción Fiscal (IV-2SLS), en reemplazo del TCRM rezagado.

Motivación: en `fase4_variables_instrumentales.py`, el TCRM rezagado fue
rechazado por el test de Sargan (p<0.001) porque el tipo de cambio real
impacta directamente sobre la recaudación y el resultado primario argentino,
violando la restricción de exclusión. Un spread soberano regional (de un par
emergente sin exposición fiscal directa a la economía argentina) captura el
mismo canal de riesgo sistémico de mercados emergentes que castiga al EMBI+
argentino sin ser causado por -ni causar- el resultado primario doméstico.

Serie utilizada: iShares J.P. Morgan USD Emerging Markets Bond ETF (EMB,
NASDAQ), que replica el JPMorgan EMBI Global Diversified —el mismo proveedor
y familia de índices que el EMBI+ Argentina ya usado en el dataset—. Se usa
el nivel trimestral (precio de cierre promedio) como proxy de mercado del
riesgo soberano emergente agregado, exactamente en el mismo sentido en que
el VIX ya se usa como instrumento en niveles (no en "puntos básicos").

Limitaciones declaradas honestamente (no ocultas):
  1. Cobertura: EMB cotiza desde dic-2007. No existe serie diaria gratuita
     para 2004T1-2007T3 (15 trimestres), que quedan como NaN y se excluyen
     de la estimación IV-2SLS por dropna(), tal como ya ocurre con otras
     variables del protocolo.
  2. Composición: Argentina integra (con peso variable, ~1-3% en la mayoría
     de los años) el índice subyacente de EMB. Esto introduce una violación
     de exclusión de segundo orden, mucho más débil que el efecto directo
     del TCRM sobre pb_pib, pero se declara explícitamente como limitación
     de la estrategia de identificación en los resultados de Fase 4.
  3. Una serie específica de Brasil ("EMBI+ Brasil" de JP Morgan/Bloomberg)
     no está disponible sin suscripción paga; se evaluaron IPEADATA, Banco
     Central do Brasil (SGS) y FRED sin encontrar una serie equivalente de
     acceso gratuito y cobertura completa 2004-2025.

Salida: datos/procesados/spread_regional_trimestral.csv
  - EMB_price   : precio de cierre, promedio trimestral (USD).
  - EMBI_BRASIL : nivel invertido y reescalado de EMB_price para que se mueva
                  en el mismo sentido que un spread (sube cuando sube el
                  riesgo EM), usado como el instrumento propiamente dicho:
                  EMBI_BRASIL = (max(EMB_price) - EMB_price), en USD.
"""

import os
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime


def download_emb(start_date: str = "2004-01-01", end_date: str = None) -> pd.DataFrame:
    if end_date is None:
        end_date = datetime.today().strftime("%Y-%m-%d")
    print(f"[EMB] Descargando iShares JPMorgan USD EM Bond ETF ({start_date} -> {end_date})...")
    ticker = yf.Ticker("EMB")
    hist = ticker.history(start=start_date, end=end_date)
    if hist.empty:
        raise ValueError("Respuesta vacía de Yahoo Finance para EMB")
    df = hist[["Close"]].rename(columns={"Close": "EMB_price"})
    df.index = df.index.tz_localize(None)
    df.index.name = "Date"
    trimestral = df.resample("QE").mean()
    print(f"  [OK] EMB: {len(trimestral)} trimestres descargados "
          f"({trimestral.index.min().date()} -> {trimestral.index.max().date()}).")
    return trimestral


def main():
    os.makedirs("datos/procesados", exist_ok=True)
    print("=" * 70)
    print(" INGESTIÓN DEL SPREAD SOBERANO REGIONAL (EMB -> proxy EMBI regional)")
    print("=" * 70)

    df = download_emb()
    df.index = df.index.to_period("Q").to_timestamp("Q")

    full_index = pd.period_range("2004-01-01", "2025-12-31", freq="Q").to_timestamp("Q")
    df = df.reindex(full_index)

    df["EMBI_BRASIL"] = df["EMB_price"].max() - df["EMB_price"]

    n_valid = df["EMBI_BRASIL"].notna().sum()
    print(f"\n[OK] Cobertura: {n_valid}/{len(df)} trimestres con dato real "
          f"({df['EMB_price'].dropna().index.min().date()} -> "
          f"{df['EMB_price'].dropna().index.max().date()}).")
    print(f"     Trimestres sin cobertura (pre iShares EMB, dic-2007): "
          f"{len(df) - n_valid} (2004T1-2007T3, quedan como NaN, sin imputar).")

    output_path = "datos/procesados/spread_regional_trimestral.csv"
    df.to_csv(output_path)
    print(f"\n[OK] Guardado: {output_path}")
    print(df.tail(8).round(2).to_string())


if __name__ == "__main__":
    main()


### Etapa 1.6 — Pasivos remunerados del BCRA

Descarga LEBAC/NOBAC, LELIQ/NOTALIQ y la posición neta de pases (API BCRA v4.0), insumo de la deuda consolidada SPNF+BCRA.

Fuente: `codigo/ingesta_datos/ingesta_bcra_pasivos.py`

In [ ]:
"""
ingesta_bcra_pasivos.py
========================
Consolidación de los pasivos remunerados del BCRA (Mejora Dimensión IV).

La serie `deuda_pib` del dataset original circunscribe el endeudamiento al
Sector Público Nacional No Financiero (SPNF) y excluye la emisión endógena
del Banco Central de la República Argentina (BCRA): LEBAC/NOBAC (2002-2018),
LELIQ/NOTALIQ (2018-2024) y la posición neta de pases. El devengamiento de
intereses de estos instrumentos constituye un déficit cuasifiscal que no
aparece en el resultado primario del Tesoro, subestimando el verdadero
requerimiento de financiamiento del sector público consolidado.

Fuente primaria (autoritativa, series oficiales del BCRA, no un proxy):
  API BCRA Estadísticas v4.0 — https://api.bcra.gob.ar/estadisticas/v4.0/monetarias
    - idVariable 1258: LEBAC, NOBAC y otras letras del BCRA en pesos
                       (no incluye LELIQ ni NOTALIQ) — vigente 2002-2018.
    - idVariable 1260: LELIQ y NOTALIQ — vigente 2018-2024 (a cero desde
                       jul-2024, cuando el Tesoro asumió el rol de esterilizador
                       mediante LEFI, decisión de política monetaria real y no
                       un artefacto de la serie).
    - idVariable 1261: Posición neta de pases (pasivos - activos) — cubre
                       todo el período 1996-2025.

  PIB nominal (precios corrientes, millones de ARS):
    datos.gob.ar, serie 166.2_PPIB_0_0_3 (INDEC, Cuentas Nacionales, 2006-2024).
    La serie está expresada a tasa anualizada estacionalmente ajustada (SAAR):
    se verificó empíricamente que dividir el stock de pasivos remunerados por
    el valor trimestral de esta serie sin transformación adicional reproduce
    el ~9.5% del PIB (2022-2023) citado en la literatura sobre el costo
    cuasifiscal del BCRA, mientras que agregar una suma móvil de 4 trimestres
    (que asumiría una serie NO anualizada) subestima la ratio en un factor
    ~4x. En consecuencia, `pib_nominal_trim` se usa directamente como
    denominador anualizado, sin suma móvil.
    Los trimestres sin dato directo (2004T1-2005T4 y 2025) se completan por
    extrapolación explícita a partir de la tasa de crecimiento nominal
    interanual promedio de los cuatro trimestres previos disponibles; el
    supuesto queda documentado en la columna `pib_nominal_es_extrapolado`.

Salida: datos/procesados/bcra_pasivos_trimestral.csv
  - pasivos_bcra_ars       : stock fin de trimestre, LEBAC/NOBAC + LELIQ/NOTALIQ
                             + posición neta de pases (millones de ARS).
  - pib_nominal_trim       : PIB nominal trimestral, SAAR (millones de ARS).
  - pib_nominal_es_extrapolado : bandera (1/0) de trimestres extrapolados.
  - pasivos_bcra_pib       : pasivos_bcra_ars / pib_nominal_trim * 100.
"""

import os
import time
import pandas as pd
import numpy as np
import requests

BCRA_BASE = "https://api.bcra.gob.ar/estadisticas/v4.0/monetarias"
HEADERS = {"User-Agent": "Mozilla/5.0"}

SERIES = {
    "lebac_nobac": 1258,   # LEBAC, NOBAC y otras letras (no incluye LELIQ/NOTALIQ)
    "leliq_notaliq": 1260,  # LELIQ y NOTALIQ
    "pases_neto": 1261,     # Posición neta de pases (pasivos - activos)
}

PIB_NOMINAL_SERIES_ID = "166.2_PPIB_0_0_3"


def fetch_bcra_variable(id_variable: int, desde: str, hasta: str) -> pd.DataFrame:
    """Descarga la serie diaria completa de una variable BCRA (con paginación)."""
    all_rows = []
    offset = 0
    limit = 3000
    while True:
        params = {"desde": desde, "hasta": hasta, "limit": limit, "offset": offset}
        r = requests.get(f"{BCRA_BASE}/{id_variable}", params=params,
                          headers=HEADERS, timeout=30)
        r.raise_for_status()
        payload = r.json()
        results = payload.get("results", [])
        if not results:
            break
        detalle = results[0].get("detalle", [])
        all_rows.extend(detalle)
        count = payload.get("metadata", {}).get("resultset", {}).get("count", 0)
        offset += limit
        if offset >= count or not detalle:
            break
        time.sleep(0.2)
    if not all_rows:
        return pd.DataFrame(columns=["valor"])
    df = pd.DataFrame(all_rows)
    df["fecha"] = pd.to_datetime(df["fecha"])
    df = df.set_index("fecha")[["valor"]].sort_index()
    df = df[~df.index.duplicated(keep="last")]
    return df


def build_bcra_pasivos_diarios(desde: str = "2004-01-01", hasta: str = "2025-12-31") -> pd.DataFrame:
    """Combina LEBAC/NOBAC + LELIQ/NOTALIQ + posición neta de pases (diario)."""
    print("[BCRA] Descargando pasivos remunerados (API v4.0, series oficiales)...")
    series = {}
    for label, vid in SERIES.items():
        print(f"  -> idVariable {vid} ({label})...")
        s = fetch_bcra_variable(vid, desde, hasta)
        s.columns = [label]
        print(f"     {len(s)} observaciones diarias, {s.index.min()} -> {s.index.max()}")
        series[label] = s

    df = series["lebac_nobac"]
    for label in ["leliq_notaliq", "pases_neto"]:
        df = df.merge(series[label], left_index=True, right_index=True, how="outer")
    df = df.sort_index().ffill()
    df = df.fillna(0.0)
    df["pasivos_bcra_ars"] = df["lebac_nobac"] + df["leliq_notaliq"] + df["pases_neto"]
    return df


def fetch_pib_nominal(desde: str = "2004-01-01", hasta: str = "2025-12-31") -> pd.DataFrame:
    """PIB nominal trimestral (precios corrientes), con extrapolación explícita
    de los trimestres sin cobertura directa (2004-2005 y 2025)."""
    print("[PIB Nominal] Descargando serie 166.2_PPIB_0_0_3 (INDEC)...")
    url = (
        f"https://apis.datos.gob.ar/series/api/series?ids={PIB_NOMINAL_SERIES_ID}"
        f"&format=csv&collapse=quarter&collapse_aggregation=end_of_period&limit=400"
    )
    df = pd.read_csv(url, parse_dates=["indice_tiempo"], index_col="indice_tiempo",
                      storage_options={"User-Agent": "Mozilla/5.0"})
    df.columns = ["pib_nominal_trim"]
    df.index = df.index.to_period("Q").to_timestamp("Q")
    df["pib_nominal_es_extrapolado"] = 0

    full_index = pd.period_range(desde, hasta, freq="Q").to_timestamp("Q")
    df = df.reindex(full_index)
    df["pib_nominal_es_extrapolado"] = df["pib_nominal_es_extrapolado"].fillna(1).astype(int)

    # Extrapolación hacia atrás (2004T1-2005T4): tasa de crecimiento nominal
    # interanual promedio de los primeros 4 trimestres con dato real.
    valid = df["pib_nominal_trim"].dropna()
    g_yoy_inicial = valid.iloc[:8].pct_change(4).dropna().mean()
    idx_backfill = df.index[df["pib_nominal_trim"].isna() & (df.index < valid.index.min())]
    for t in sorted(idx_backfill, reverse=True):
        t_fwd = (pd.Period(t, freq="Q") + 4).to_timestamp("Q")
        if t_fwd in df.index and pd.notna(df.loc[t_fwd, "pib_nominal_trim"]):
            df.loc[t, "pib_nominal_trim"] = df.loc[t_fwd, "pib_nominal_trim"] / (1 + g_yoy_inicial)

    # Extrapolación hacia adelante (2025): tasa de crecimiento nominal
    # interanual promedio de los últimos 4 trimestres con dato real.
    valid = df["pib_nominal_trim"].dropna()
    g_yoy_final = valid.pct_change(4).dropna().iloc[-4:].mean()
    idx_forwardfill = df.index[df["pib_nominal_trim"].isna() & (df.index > valid.index.max())]
    for t in sorted(idx_forwardfill):
        t_lag = (pd.Period(t, freq="Q") - 4).to_timestamp("Q")
        if t_lag in df.index and pd.notna(df.loc[t_lag, "pib_nominal_trim"]):
            df.loc[t, "pib_nominal_trim"] = df.loc[t_lag, "pib_nominal_trim"] * (1 + g_yoy_final)

    n_extrap = int(df["pib_nominal_es_extrapolado"].sum())
    print(f"  -> {len(df) - n_extrap} trimestres con dato directo INDEC, "
          f"{n_extrap} extrapolados (tasa YoY inicial={g_yoy_inicial:.3f}, final={g_yoy_final:.3f}).")
    return df


def main():
    os.makedirs("datos/procesados", exist_ok=True)
    print("=" * 70)
    print(" CONSOLIDACIÓN DE PASIVOS REMUNERADOS DEL BCRA (Mejora Dimensión IV)")
    print("=" * 70)

    daily = build_bcra_pasivos_diarios()
    stock_trimestral = daily["pasivos_bcra_ars"].resample("QE").last().to_frame()
    stock_trimestral.index = stock_trimestral.index.to_period("Q").to_timestamp("Q")

    pib_nominal = fetch_pib_nominal()

    df = stock_trimestral.merge(pib_nominal, left_index=True, right_index=True, how="outer")
    df = df.sort_index()
    df["pasivos_bcra_pib"] = df["pasivos_bcra_ars"] / df["pib_nominal_trim"] * 100

    output_path = "datos/procesados/bcra_pasivos_trimestral.csv"
    df.to_csv(output_path)
    print(f"\n[OK] Guardado: {output_path}")
    print(f"     Cobertura: {df.index.min().date()} -> {df.index.max().date()} ({len(df)} trimestres)")
    print("\n     Pasivos remunerados BCRA / PIB (%), últimos 12 trimestres:")
    print(df[["pasivos_bcra_ars", "pasivos_bcra_pib"]].tail(12).round(2).to_string())

    pico = df["pasivos_bcra_pib"].idxmax()
    print(f"\n     Pico histórico: {df.loc[pico, 'pasivos_bcra_pib']:.2f}% del PIB en {pico.date()}")


if __name__ == "__main__":
    main()


### Etapa 2.1 — Estacionariedad y quiebre estructural endógeno

Contrastes ADF y KPSS sobre cada serie, complementados con el test de quiebre endógeno de Zivot-Andrews y, como robustez, DF-GLS.

Fuente: `codigo/modelos/fase1_estacionariedad.py`

In [ ]:
"""
fase1_estacionariedad.py
========================
Fase 1 del Protocolo Econométrico: Análisis de Integración (Series Temporales).
Ejecuta pruebas de raíz unitaria y estacionariedad sobre el dataset empírico real.

Pruebas implementadas:
  1. ADF (Augmented Dickey-Fuller) - H0: Raíz unitaria
  2. PP (Phillips-Perron)          - H0: Raíz unitaria (requiere 'arch', opcional)
  3. KPSS (Kwiatkowski-Phillips-Schmidt-Shin) - H0: Estacionariedad
  4. DF-GLS (Elliott-Rothenberg-Stock, 1996)  - H0: Raíz unitaria (requiere 'arch')
  5. Test de Quiebre Estructural (CUSUM) para evaluar estabilidad paramétrica.

Criterio de integración: Una serie es I(1) si es no estacionaria en niveles
y estacionaria en su primera diferencia.
"""

import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Suprimir warnings de KPSS (ej: p-value bounds)
warnings.filterwarnings("ignore")

def adf_test(series, signif=0.05):
    """Test Augmented Dickey-Fuller."""
    res = adfuller(series.dropna(), autolag='AIC')
    p_val = res[1]
    is_stat = p_val < signif
    return p_val, is_stat

def kpss_test(series, signif=0.05):
    """Test KPSS."""
    res = kpss(series.dropna(), regression='c', nlags="auto")
    p_val = res[1]
    is_stat = p_val >= signif # H0 es estacionariedad
    return p_val, is_stat

def phillips_perron_test(series, signif=0.05):
    """Test Phillips-Perron (intenta usar arch.unitroot si está disponible)."""
    try:
        from arch.unitroot import PhillipsPerron
        pp = PhillipsPerron(series.dropna())
        p_val = pp.pvalue
        is_stat = p_val < signif
        return p_val, is_stat
    except ImportError:
        return np.nan, np.nan

def dfgls_test(series, trend='c'):
    """Test DF-GLS (Elliott, Rothenberg y Stock, 1996), vía arch.unitroot.DFGLS.
    trend='ct' (tendencia + constante) para niveles, trend='c' (solo constante)
    para primeras diferencias, siguiendo la especificación reportada en la
    Tabla 6.1 de la tesis."""
    try:
        from arch.unitroot import DFGLS
        res = DFGLS(series.dropna(), trend=trend)
        return res.stat, res.critical_values['5%']
    except ImportError:
        return np.nan, np.nan


def zivot_andrews_test(series, lags=4, trend='ct', trim=0.15):
    """Test de Zivot-Andrews (1992) con quiebre estructural endógeno, Modelo C
    (quiebre simultáneo en intercepto y pendiente), vía arch.unitroot.ZivotAndrews.
    Reporta el estadístico t óptimo, la fecha de quiebre (el candidato que
    minimiza el estadístico t, según el propio algoritmo) y los valores
    críticos asintóticos del Modelo C, tal como se reportan en la
    Sección 5.6.1 de la tesis."""
    import numpy as np
    from arch.unitroot import ZivotAndrews
    clean = series.dropna()
    res = ZivotAndrews(clean, lags=lags, trend=trend, trim=trim)
    _ = res.stat  # fuerza el cómputo interno antes de leer _all_stats
    break_idx = np.nanargmin(res._all_stats)
    break_date = clean.index[break_idx]
    return res.stat, break_date, res.critical_values


def analyze_dfgls(df, columns):
    """Aplica DF-GLS en nivel (tendencia+constante) y primera diferencia
    (solo constante) a cada columna, replicando la Tabla 6.1 del Capítulo 6."""
    results = []
    for col in columns:
        series = df[col]
        stat_niv, crit_niv = dfgls_test(series, trend='ct')
        diff_series = series.diff().dropna()
        stat_diff, crit_diff = dfgls_test(diff_series, trend='c')

        rechaza_niv = stat_niv < crit_niv
        rechaza_diff = stat_diff < crit_diff
        if rechaza_niv:
            orden = "I(0)"
        elif rechaza_diff:
            orden = "I(1)"
        else:
            orden = "Ambiguo"

        results.append({
            "Variable": col,
            "DFGLS_stat (Nivel)": round(stat_niv, 2),
            "DFGLS_crit5% (Nivel)": round(crit_niv, 2),
            "DFGLS_stat (Diff)": round(stat_diff, 2),
            "DFGLS_crit5% (Diff)": round(crit_diff, 2),
            "Conclusion": orden,
        })
    return pd.DataFrame(results)


def analyze_stationarity(df, columns):
    """Realiza análisis completo I(0) vs I(1) para las columnas dadas."""
    results = []
    
    for col in columns:
        series = df[col]
        # Niveles
        adf_p_niv, adf_stat_niv = adf_test(series)
        kpss_p_niv, kpss_stat_niv = kpss_test(series)
        pp_p_niv, pp_stat_niv = phillips_perron_test(series)
        
        # Primeras diferencias
        diff_series = series.diff().dropna()
        adf_p_diff, adf_stat_diff = adf_test(diff_series)
        kpss_p_diff, kpss_stat_diff = kpss_test(diff_series)
        pp_p_diff, pp_stat_diff = phillips_perron_test(diff_series)
        
        # Determinación heurística simple del orden de integración
        if adf_stat_niv and kpss_stat_niv:
            orden = "I(0)"
        elif adf_stat_diff and kpss_stat_diff:
            orden = "I(1)"
        else:
            orden = "Ambiguo / I(1)"

        results.append({
            "Variable": col,
            "ADF_p (Nivel)": round(adf_p_niv, 4),
            "KPSS_p (Nivel)": round(kpss_p_niv, 4),
            "PP_p (Nivel)": round(pp_p_niv, 4) if not np.isnan(pp_p_niv) else "N/A",
            "ADF_p (Diff)": round(adf_p_diff, 4),
            "KPSS_p (Diff)": round(kpss_p_diff, 4),
            "Orden Inferido": orden
        })
        
    return pd.DataFrame(results)

def structural_break_cusum(df, col_y='pb_pib', col_x='deuda_pib'):
    """Test CUSUM de residuos OLS para detectar inestabilidad estructural."""
    try:
        from statsmodels.stats.diagnostic import breaks_cusumolsresid
        Y = df[col_y].dropna()
        X = sm.add_constant(df[col_x].loc[Y.index])
        model = sm.OLS(Y, X).fit()
        test_stat, p_val, crit = breaks_cusumolsresid(model.resid)
        print(f"\n[CUSUM Test] {col_y} ~ {col_x}: p-value = {p_val:.4f}")
        if p_val < 0.05:
            print("  -> Existe evidencia de quiebre estructural (rechazo de H0 param. estables).")
        else:
            print("  -> No hay evidencia suficiente de quiebre estructural en la regresión simple.")
    except Exception as e:
        print(f"\n[!] Error en CUSUM test: {e}")

def main():
    print("=" * 65)
    print(" FASE 1: TEST DE RAÍZ UNITARIA Y ESTACIONARIEDAD (DATOS REALES)")
    print("=" * 65)
    
    file_path = "datos/dataset_consolidado_real.csv"
    if not os.path.exists(file_path):
        print(f"[!] Archivo no encontrado: {file_path}")
        return
        
    df = pd.read_csv(file_path, parse_dates=['Date'], index_col='Date')
    
    # Variables clave para el análisis
    target_vars = ['deuda_pib', 'pb_pib', 'g_gap', 'EMBI', 'TCRM', 'CER']
    
    # Verificar si están en el dataset
    vars_to_test = [v for v in target_vars if v in df.columns]
    
    print(f"\nRealizando pruebas para: {vars_to_test}")
    results_df = analyze_stationarity(df, vars_to_test)
    
    print("\nRESULTADOS (p-values):")
    print(results_df.to_string(index=False))
    
    # Test CUSUM de quiebre estructural en relación bivariada (ej. Bohn simple)
    if 'pb_pib' in df.columns and 'deuda_pib' in df.columns:
        structural_break_cusum(df)

    # DF-GLS (Elliott, Rothenberg y Stock, 1996), Tabla 6.1 del Capítulo 6:
    # pb_t, d_t, risk_t (EMBI+) y la brecha del producto.
    dfgls_vars = ['pb_pib', 'deuda_pib', 'EMBI', 'g_gap']
    dfgls_vars = [v for v in dfgls_vars if v in df.columns]
    dfgls_df = analyze_dfgls(df, dfgls_vars)
    print("\nRESULTADOS DF-GLS (Elliott-Rothenberg-Stock, 1996):")
    print(dfgls_df.to_string(index=False))

    # Zivot-Andrews (1992) con quiebre endógeno, Modelo C, sobre la ratio
    # Deuda/PIB, Sección 5.6.1.
    za_stat, za_break, za_crit = zivot_andrews_test(df['deuda_pib'], lags=4, trend='ct', trim=0.15)
    print("\nRESULTADO ZIVOT-ANDREWS (Modelo C, quiebre endógeno, deuda_pib):")
    print(f"  Estadistico t: {za_stat:.2f}  |  Quiebre optimo: {za_break.date()}")
    print(f"  Criticos: 1%={za_crit['1%']:.2f}  5%={za_crit['5%']:.2f}  10%={za_crit['10%']:.2f}")
    za_df = pd.DataFrame([{
        "Variable": "deuda_pib",
        "Modelo": "C (intercepto + pendiente)",
        "Estadistico_t": round(za_stat, 2),
        "Quiebre_optimo": za_break.date().isoformat(),
        "Critico_1%": round(za_crit['1%'], 2),
        "Critico_5%": round(za_crit['5%'], 2),
        "Critico_10%": round(za_crit['10%'], 2),
    }])

    # Guardar resultados
    os.makedirs("resultados/tablas", exist_ok=True)
    results_df.to_csv("resultados/tablas/fase1_estacionariedad.csv", index=False)
    dfgls_df.to_csv("resultados/tablas/fase1_dfgls.csv", index=False)
    za_df.to_csv("resultados/tablas/fase1_zivot_andrews.csv", index=False)
    print("\n[OK] Resultados guardados en 'resultados/tablas/fase1_estacionariedad.csv', 'fase1_dfgls.csv' y 'fase1_zivot_andrews.csv'")

if __name__ == "__main__":
    main()


### Etapa 2.2 — Cointegración de Johansen

Evalúa si el sistema [Deuda/PIB, Resultado Primario, EMBI+, TCRM] admite una relación de equilibrio de largo plazo.

Fuente: `codigo/modelos/fase2_cointegracion.py`

In [ ]:
"""
fase2_cointegracion.py
=====================
Fase 2 del Protocolo Econométrico: Análisis de Cointegración.

Realiza el Test de Johansen para verificar si existe una relación de equilibrio
de largo plazo entre las variables I(1) del sistema, como pre-requisito
para aplicar modelos como DOLS.

Incluye selección de rezagos (lags) mediante criterios de información (AIC, BIC).
"""

import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.vector_ar.var_model import VAR
import warnings

warnings.filterwarnings("ignore")

def select_var_lags(df, maxlags=6):
    """Selecciona el número óptimo de rezagos para el VAR usando AIC/BIC."""
    model = VAR(df)
    res = model.select_order(maxlags)
    print("\n[Selección de Rezagos VAR]")
    print(res.summary())
    # Preferimos AIC para muestras medianas/pequeñas
    return res.aic

def johansen_test(df, det_order=-1, k_ar_diff=1):
    """
    Realiza el test de cointegración de Johansen.
    det_order: -1 (sin terminos deterministicos), 0 (constante), 1 (tendencia)
    k_ar_diff: número de rezagos en primeras diferencias
    """
    res = coint_johansen(df, det_order=det_order, k_ar_diff=k_ar_diff)
    
    # Extraer resultados (Traza y Máximo Autovalor)
    traces = res.lr1
    traces_crit = res.cvt  # Columnas: 90%, 95%, 99%
    maxeig = res.lr2
    maxeig_crit = res.cvm
    
    n_vars = df.shape[1]
    
    results = []
    for i in range(n_vars):
        # i es el número de relaciones de cointegración bajo H0
        results.append({
            "H0: r<=": i,
            "Traza Stat": round(traces[i], 2),
            "Traza Crit 95%": round(traces_crit[i, 1], 2),
            "Signif (Traza)": traces[i] > traces_crit[i, 1],
            "MaxEig Stat": round(maxeig[i], 2),
            "MaxEig Crit 95%": round(maxeig_crit[i, 1], 2),
            "Signif (MaxEig)": maxeig[i] > maxeig_crit[i, 1]
        })
        
    return pd.DataFrame(results)

def main():
    print("=" * 65)
    print(" FASE 2: TEST DE COINTEGRACIÓN (JOHANSEN)")
    print("=" * 65)
    
    file_path = "datos/dataset_consolidado_real.csv"
    if not os.path.exists(file_path):
        print(f"[!] Archivo no encontrado: {file_path}")
        return
        
    df = pd.read_csv(file_path, parse_dates=['Date'], index_col='Date')
    
    # Variables de la Función de Reacción Fiscal de largo plazo
    # Típicamente: Deuda/PIB y PB/PIB. A veces se incluye el output gap o shocks externos
    coint_vars = ['deuda_pib', 'pb_pib', 'EMBI', 'TCRM']
    
    # Verificar si están en el dataset
    vars_to_test = [v for v in coint_vars if v in df.columns]
    df_coint = df[vars_to_test].dropna()
    
    print(f"\nAnalizando sistema: {vars_to_test}")
    print(f"Observaciones disponibles: {len(df_coint)}")
    
    # 1. Selección de rezagos óptimos
    opt_lags = select_var_lags(df_coint, maxlags=6)
    print(f"Rezago óptimo (AIC): {opt_lags}")
    
    # Si opt_lags es 0, usamos 1 para el test (k_ar_diff requiere lags >= 1)
    k_ar_diff = max(1, opt_lags - 1) 
    
    # 2. Test de Johansen
    # Usamos det_order=0 (constante en el vector de cointegración)
    johansen_results = johansen_test(df_coint, det_order=0, k_ar_diff=k_ar_diff)
    
    print("\nRESULTADOS TEST JOHANSEN (Traza y Max. Autovalor):")
    print(johansen_results.to_string(index=False))
    
    num_coint = sum(johansen_results['Signif (Traza)'])
    print(f"\nCONCLUSIÓN: Se hallaron {num_coint} vector(es) de cointegración (al 95%).")
    if num_coint > 0:
        print(" -> ES VÁLIDO aplicar DOLS/VECM para el análisis de largo plazo.")
    else:
        print(" -> ADVERTENCIA: No se encontró cointegración robusta. DOLS podría ser espurio.")
    
    # Guardar resultados
    os.makedirs("resultados/tablas", exist_ok=True)
    johansen_results.to_csv("resultados/tablas/fase2_cointegracion.csv", index=False)
    print("\n[OK] Resultados guardados en 'resultados/tablas/fase2_cointegracion.csv'")

if __name__ == "__main__":
    main()


### Etapa 2.3 — Función de Reacción Fiscal (DOLS)

Estima la ecuación de Bohn por Mínimos Cuadrados Ordinarios Dinámicos, el estimador principal de la tesis.

Fuente: `codigo/modelos/fase3_reaccion_fiscal.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase3_reaccion_fiscal.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase3_reaccion_fiscal.py"

"""
fase3_reaccion_fiscal.py
=========================
Fase 3 del Protocolo Econométrico: Estimación de la Función de Reacción Fiscal (FRF).
Estima la reacción del superávit primario frente a la acumulación de deuda.

Implementa:
  - MCO con errores consistentes HAC (selección automática de lags).
  - Test de Hausman Aumentado para endogeneidad (usando instrumentos válidos).
  - DOLS (Dynamic OLS) en caso de endogeneidad o por la super-consistencia con I(1) cointegradas.
  - Criterios de información para seleccionar m rezagos y adelantos en DOLS.
  - Pruebas de especificación post-estimación (RESET, Breusch-Godfrey, Jarque-Bera).
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.diagnostic as smd
import statsmodels.stats.api as sms
import os
import warnings
from scipy import stats

warnings.filterwarnings("ignore")

def newey_west_lags(n):
    """Calcula lags óptimos de Newey-West según m = 4*(T/100)^(2/9)."""
    return int(np.ceil(4 * (n / 100)**(2/9)))

def select_dols_lags(df, y_col, x_cols, d_col, max_m=4):
    """
    Selecciona el número de rezagos y adelantos (m) para DOLS 
    minimizando el Criterio de Información de Akaike (AIC).
    """
    best_aic = np.inf
    best_m = 1
    
    for m in range(1, max_m + 1):
        df_temp = df.copy()
        dols_features = list(x_cols)
        
        # Agregar diferencias contemporáneas, rezagos y adelantos
        df_temp['diff_d'] = df_temp[d_col].diff()
        dols_features.append('diff_d')
        
        for i in range(1, m + 1):
            df_temp[f'diff_d_lag_{i}'] = df_temp['diff_d'].shift(i)
            df_temp[f'diff_d_lead_{i}'] = df_temp['diff_d'].shift(-i)
            dols_features.extend([f'diff_d_lag_{i}', f'diff_d_lead_{i}'])
            
        df_temp = df_temp.dropna(subset=[y_col] + dols_features)
        
        Y = df_temp[y_col]
        X = sm.add_constant(df_temp[dols_features])
        
        try:
            model = sm.OLS(Y, X).fit()
            if model.aic < best_aic:
                best_aic = model.aic
                best_m = m
        except:
            pass
            
    return best_m

def ejecutar_fase3_econometria(csv_path):
    print("=" * 75)
    print(" FASE 3: Función de Reacción Fiscal Base (MCO-HAC y DOLS) ")
    print("=" * 75)
    
    if not os.path.exists(csv_path):
        print(f"[!] ERROR: No se encontró el dataset en {csv_path}")
        return
        
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
    
    # 1. Variables rezagadas y diferencias
    df['d_t_1'] = df['deuda_pib'].shift(1)
    
    # Instrumentos externos (EMBI y TCRM) rezagados para evitar instrumentación circular
    df['embi_t_1'] = df['EMBI'].shift(1)
    df['tcrm_t_1'] = df['TCRM'].shift(1)
    
    # Filtrar NA de las variables base
    base_cols = ['pb_pib', 'd_t_1', 'g_gap', 'embi_t_1', 'tcrm_t_1']
    df = df.dropna(subset=base_cols)
    n_obs = len(df)
    hac_lags = newey_west_lags(n_obs)
    
    print(f"\n[1/4] Estimando MCO con Errores Robustos (HAC maxlags={hac_lags})...")
    # pb_pib = beta_0 + beta_1 * d_t_1 + beta_2 * g_gap + u_t
    y = df['pb_pib']
    X_base = df[['d_t_1', 'g_gap']]
    X = sm.add_constant(X_base)
    
    model_mco = sm.OLS(y, X)
    results_mco = model_mco.fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
    
    print(results_mco.summary().tables[1])
    
    print("\n[2/4] Pruebas de Diagnóstico Post-Estimación...")
    # Breusch-Godfrey (Autocorrelación)
    bg_test = smd.acorr_breusch_godfrey(results_mco, nlags=4)
    print(f" -> Breusch-Godfrey (LM p-value): {bg_test[1]:.4f}")
    
    # Breusch-Pagan (Heterocedasticidad)
    bp_test = sms.het_breuschpagan(results_mco.resid, X)
    print(f" -> Breusch-Pagan (p-value): {bp_test[1]:.4f}")
    
    # Jarque-Bera (Normalidad de residuos)
    jb_test = sm.stats.stattools.jarque_bera(results_mco.resid)
    print(f" -> Jarque-Bera (p-value): {jb_test[1]:.4f}")
    
    # RESET de Ramsey (Especificación)
    reset_test = sm.stats.diagnostic.linear_reset(results_mco, power=2, test_type="fitted", use_f=True)
    print(f" -> RESET Ramsey (p-value): {reset_test.pvalue:.4f}")
    
    print("\n[3/4] Evaluando Endogeneidad (Test de Hausman Aumentado)...")
    # Usamos EMBI_t-1 y TCRM_t-1 como instrumentos (variables externas) en lugar de rezagos propios d_t_2
    Z = df[['embi_t_1', 'tcrm_t_1', 'g_gap']]
    Z = sm.add_constant(Z)
    fs_model = sm.OLS(df['d_t_1'], Z).fit()
    df['v_hat'] = fs_model.resid
    
    X_aug = X.copy()
    X_aug['v_hat'] = df['v_hat']
    hausman_model = sm.OLS(y, X_aug).fit()
    hausman_pvalue = hausman_model.pvalues['v_hat']
    print(f" -> Hausman Test p-value (v_hat): {hausman_pvalue:.4f}")
    
    endogeneity = hausman_pvalue < 0.10
    if endogeneity:
        print("    [!] Endogeneidad detectada (p < 0.10). Es obligatorio usar DOLS / IV.")
    else:
        print("    [OK] No se rechaza exogeneidad, pero DOLS es preferible por la relación de cointegración (super-consistencia).")
        
    print("\n[4/4] Estimación DOLS (Dynamic OLS)...")
    # Selección de lags/leads óptimos
    m_opt = select_dols_lags(df, 'pb_pib', ['d_t_1', 'g_gap'], 'd_t_1', max_m=4)
    print(f" -> Rezagos/adelantos óptimos seleccionados por AIC: m = {m_opt}")
    
    df_dols = df.copy()
    df_dols['diff_d'] = df_dols['d_t_1'].diff()
    dols_features = ['d_t_1', 'g_gap', 'diff_d']
    
    for i in range(1, m_opt + 1):
        df_dols[f'diff_d_lag_{i}'] = df_dols['diff_d'].shift(i)
        df_dols[f'diff_d_lead_{i}'] = df_dols['diff_d'].shift(-i)
        dols_features.extend([f'diff_d_lag_{i}', f'diff_d_lead_{i}'])
        
    df_dols = df_dols.dropna(subset=['pb_pib'] + dols_features)
    y_dols = df_dols['pb_pib']
    X_dols = sm.add_constant(df_dols[dols_features])
    
    dols_model = sm.OLS(y_dols, X_dols).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
    
    print("\n--- RESULTADOS DOLS (Largo Plazo) ---")
    print(dols_model.summary().tables[1])
    
    beta_1 = dols_model.params['d_t_1']
    p_val = dols_model.pvalues['d_t_1']
    
    print("\n===============================================================================")
    print(" Veredicto de Sostenibilidad Intertemporal (H0: beta_1 <= 0) ")
    print("===============================================================================")
    if beta_1 > 0 and p_val < 0.05:
        print(f" [RECHAZO H0] Se verifica reacción marginal positiva y significativa.")
        print(f" -> Coeficiente estimado beta_1 = {beta_1:.4f} (p-value = {p_val:.4f})")
        print(" -> Condición de sostenibilidad débil de Bohn (1998) satisfecha para el período completo.")
    else:
        print(f" [NO SE RECHAZA H0] No hay evidencia robusta de reacción marginal positiva.")
        print(f" -> Coeficiente estimado beta_1 = {beta_1:.4f} (p-value = {p_val:.4f})")
        print(" -> La política fiscal promedio del período no garantiza la solvencia intertemporal (riesgo de Ponzi).")
        print(" -> Requiere análisis de fatiga fiscal (Fase 5).")
        
    # Guardar resultados
    os.makedirs("resultados/tablas", exist_ok=True)
    with open("resultados/tablas/fase3_resultados_dols.csv", "w") as f:
        f.write(dols_model.summary().as_csv())
    print("\n[OK] Resultados guardados en 'resultados/tablas/fase3_resultados_dols.csv'")

if __name__ == "__main__":
    import pathlib
    base_dir = pathlib.Path(__file__).parent.parent.parent
    csv_file = base_dir / "datos" / "dataset_consolidado_real.csv"
    ejecutar_fase3_econometria(csv_file)


### Etapa 2.4 — Corrección por endogeneidad (IV-2SLS)

Instrumenta el EMBI+ con el VIX y el spread regional para corregir la causalidad inversa entre riesgo soberano y resultado fiscal.

Fuente: `codigo/modelos/fase4_variables_instrumentales.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase4_variables_instrumentales.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase4_variables_instrumentales.py"

"""
fase4_variables_instrumentales.py
=================================
Fase 4 del Protocolo Econométrico: Estimación por Variables Instrumentales (IV/2SLS).
Aborda la endogeneidad del riesgo país (EMBI+) en la Función de Reacción Fiscal.

Implementa:
  - 2SLS (Two-Stage Least Squares) usando `linearmodels.iv.IV2SLS` para obtener
    errores estándar consistentes (corrigiendo el HC-02 de la auditoría).
  - VIX (Índice de volatilidad global) y un spread soberano regional
    (EMBI_BRASIL, proxy de mercado construida en
    `codigo/ingesta_datos/ingesta_spread_regional.py` a partir del ETF EMB -iShares
    JPMorgan USD EM Bond, misma familia de índices que el EMBI+ argentino-)
    como instrumentos del EMBI+, en reemplazo del TCRM rezagado (Mejora
    Dimensión III): el TCRM impacta directamente sobre la recaudación y el
    resultado primario argentino y por eso fue rechazado por Sargan; un
    spread soberano regional captura el mismo riesgo sistémico de mercados
    emergentes sin ese canal presupuestario directo.
  - Diagnósticos de Primera Etapa (F-test para instrumentos débiles).
  - Test de Sobreidentificación de Sargan.
  - Test de Endogeneidad de Wu-Hausman.
"""

import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
import warnings

# Se requiere linearmodels para IV robusto
try:
    from linearmodels.iv import IV2SLS
except ImportError:
    print("[!] ERROR: 'linearmodels' no está instalado. Ejecute: pip install linearmodels")
    import sys
    sys.exit(1)

warnings.filterwarnings("ignore")

def ejecutar_fase4_econometria(csv_path):
    print("=" * 75)
    print(" FASE 4: Estimación IV (2SLS) para EMBI+ Endógeno ")
    print("=" * 75)
    
    if not os.path.exists(csv_path):
        print(f"[!] ERROR: No se encontró el dataset en {csv_path}")
        return
        
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
    
    print("\n[1/4] Preparando variables e instrumentos...")
    # Lags necesarios
    df['d_t_1'] = df['deuda_pib'].shift(1)

    # Ecuación estructural (Segunda Etapa):
    # pb_pib = beta_0 + beta_1 * d_t_1 + beta_2 * g_gap + beta_3 * EMBI + u_t
    # Variable Endógena: EMBI
    # Variables Exógenas Incluidas: d_t_1, g_gap, const
    # Variables Exógenas Excluidas (Instrumentos): VIX, EMBI_BRASIL
    # (Mejora Dimensión III: EMBI_BRASIL reemplaza a TCRM_t-1, rechazado por
    # Sargan al violar la restricción de exclusión vía su efecto directo
    # sobre la recaudación y el resultado primario)

    subset_cols = ['pb_pib', 'd_t_1', 'g_gap', 'EMBI', 'VIX', 'EMBI_BRASIL']
    df = df.dropna(subset=subset_cols)

    y = df['pb_pib']
    exog = sm.add_constant(df[['d_t_1', 'g_gap']])
    endog = df[['EMBI']]
    instr = df[['VIX', 'EMBI_BRASIL']]

    print(f" -> Observaciones válidas: {len(df)}")
    print(" -> Ecuación Estructural: pb_pib ~ 1 + d_t_1 + g_gap + [EMBI ~ VIX + EMBI_BRASIL]")
    
    print("\n[2/4] Estimando 2SLS (linearmodels.iv)...")
    # Utilizamos cov_type='robust' o 'kernel' (HAC) 
    iv_model = IV2SLS(dependent=y, exog=exog, endog=endog, instruments=instr)
    iv_res = iv_model.fit(cov_type='kernel', kernel='newey-west')
    
    print("\n--- RESULTADOS 2SLS ---")
    print(iv_res.summary.tables[1])
    
    print("\n[3/4] Diagnósticos de Primera Etapa (Instrumentos Débiles)...")
    # Test F de instrumentos excluidos
    first_stage = iv_res.first_stage
    f_stat = first_stage.diagnostics.loc['EMBI', 'f.stat']
    f_pval = first_stage.diagnostics.loc['EMBI', 'f.pval']
    
    print(f" -> Estadístico F (Instrumentos Excluidos): {f_stat:.4f} (p-value: {f_pval:.4f})")
    if f_stat > 10:
        print("    [OK] F > 10. Se supera el criterio de Staiger y Stock. Instrumentos relevantes.")
    else:
        print("    [!] F < 10. Riesgo de instrumentos débiles. Precaución con la inferencia.")
        
    print("\n[4/4] Pruebas de Endogeneidad y Sobreidentificación...")
    
    # Wu-Hausman (Endogeneidad)
    wu_hausman = iv_res.wu_hausman()
    print(f" -> Test de Endogeneidad Wu-Hausman: stat={wu_hausman.stat:.4f}, p-value={wu_hausman.pval:.4f}")
    if wu_hausman.pval < 0.10:
        print("    [!] Se rechaza exogeneidad (p < 0.10). El uso de IV/2SLS es apropiado.")
    else:
        print("    [OK] No se rechaza exogeneidad firme. EMBI podría ser exógeno empíricamente, aunque teóricamente no lo sea.")

    # Sargan (Sobreidentificación)
    sargan = iv_res.sargan
    print(f" -> Test de Sargan (Sobreidentificación): stat={sargan.stat:.4f}, p-value={sargan.pval:.4f}")
    if sargan.pval > 0.05:
        print("    [OK] No se rechaza H0. Los instrumentos son válidos (no correlacionados con error estructural).")
    else:
        print("    [!] Se rechaza H0. Posible correlación de los instrumentos con el error estructural.")
        
    print("\n=======================================================================================================")
    print(" Impacto del Riesgo Soberano (EMBI+) en la Reacción Fiscal ")
    print("=======================================================================================================")
    beta_embi = iv_res.params['EMBI']
    pval_embi = iv_res.pvalues['EMBI']
    
    print(f" -> Coeficiente beta_EMBI: {beta_embi:.6f} (p-value: {pval_embi:.4f})")
    if pval_embi < 0.05:
        if beta_embi > 0:
            print(" -> CONCLUSIÓN: Ante incrementos del riesgo país, hay un ajuste positivo del superávit primario.")
        else:
            print(" -> CONCLUSIÓN: El incremento del riesgo país deteriora significativamente el superávit primario.")
    else:
        print(" -> CONCLUSIÓN: No hay evidencia estadísticamente significativa de que el superávit responda de manera estructural a cambios en el EMBI+.")

    # Guardar resultados
    os.makedirs("resultados/tablas", exist_ok=True)
    with open("resultados/tablas/fase4_resultados_iv.csv", "w") as f:
        f.write(iv_res.summary.as_csv())

    with open("resultados/tablas/fase4_diagnosticos_iv.csv", "w") as f:
        f.write("statistic,value,pvalue\n")
        f.write(f"first_stage_F_excluded_instruments,{f_stat:.4f},{f_pval:.6f}\n")
        f.write(f"wu_hausman_endogeneity,{wu_hausman.stat:.4f},{wu_hausman.pval:.6f}\n")
        f.write(f"sargan_overidentification,{sargan.stat:.4f},{sargan.pval:.6f}\n")
    print("\n[OK] Resultados guardados en 'resultados/tablas/fase4_resultados_iv.csv'")
    print("[OK] Diagnósticos guardados en 'resultados/tablas/fase4_diagnosticos_iv.csv'")

if __name__ == "__main__":
    import pathlib
    base_dir = pathlib.Path(__file__).parent.parent.parent
    csv_file = base_dir / "datos" / "dataset_consolidado_real.csv"
    ejecutar_fase4_econometria(csv_file)


### Etapa 2.5 — Modelo de umbral de Hansen (fatiga fiscal)

Testea la hipótesis de fatiga fiscal mediante regresión de umbral con significatividad *bootstrap* del estadístico Sup-LM.

Fuente: `codigo/modelos/fase5_umbral_hansen.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase5_umbral_hansen.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase5_umbral_hansen.py"

"""
fase5_umbral_hansen.py
=======================
Fase 5 del Protocolo Econométrico: Modelación de Umbrales (Fatiga Fiscal).

Implementa el test de umbral para series de tiempo basado en Hansen (2000) 
"Sample Splitting and Threshold Estimation", corrigiendo el uso inapropiado
de Hansen (1999) que es exclusivo para datos de panel.

Incluye:
  - Búsqueda del umbral óptimo (Grid Search minimizando SSR).
  - Test de significancia del umbral mediante Bootstrap (Sup-LM test).
  - Análisis de Fatiga Fiscal (Ghosh et al. 2013) separando regímenes.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
import warnings

warnings.filterwarnings("ignore")

def bootstrap_threshold_test(df, y_col, X_control_cols, split_col, threshold_col, best_tau, n_boot=500):
    """
    Test de significancia del umbral (Hansen 2000) mediante Bootstrap.
    H0: Modelo lineal (sin umbral)
    H1: Modelo con umbral (tau)
    """
    y = df[y_col].values
    
    # 1. Modelo bajo H0 (Lineal)
    X_linear = df[X_control_cols + [split_col]].copy()
    X_linear = sm.add_constant(X_linear)
    model_h0 = sm.OLS(y, X_linear).fit()
    ssr_0 = model_h0.ssr
    resid_0 = model_h0.resid.values
    
    # 2. Modelo bajo H1 (Umbral óptimo)
    I_low = (df[threshold_col] <= best_tau).astype(int)
    I_high = (df[threshold_col] > best_tau).astype(int)
    
    X_alt = df[X_control_cols].copy()
    X_alt['d_low'] = df[split_col] * I_low
    X_alt['d_high'] = df[split_col] * I_high
    X_alt = sm.add_constant(X_alt)
    model_h1 = sm.OLS(y, X_alt).fit()
    ssr_1 = model_h1.ssr
    
    # F-stat observado
    F_obs = (ssr_0 - ssr_1) / ssr_1
    
    # 3. Bootstrap
    F_boot = []
    np.random.seed(42)
    
    for _ in range(n_boot):
        # Muestreo con reemplazo de los residuos bajo H0 (wild bootstrap simple)
        # Multiplicar por N(0,1) para preservar heterocedasticidad (Rademacher o Normal)
        v = np.random.normal(0, 1, len(resid_0))
        y_sim = model_h0.fittedvalues + resid_0 * v
        
        # Ajustar H0 simulado
        m0_sim = sm.OLS(y_sim, X_linear).fit()
        
        # Ajustar H1 simulado
        m1_sim = sm.OLS(y_sim, X_alt).fit()
        
        # F-stat simulado
        f_sim = (m0_sim.ssr - m1_sim.ssr) / m1_sim.ssr
        F_boot.append(f_sim)
        
    F_boot = np.array(F_boot)
    p_value = np.mean(F_boot >= F_obs)
    
    return F_obs, p_value

def get_optimal_threshold(df, y_col, X_control_cols, split_col, threshold_col):
    """
    Busca el umbral óptimo (tau) iterando entre el percentil 15 y 85
    de la variable de transición, asegurando grados de libertad.
    """
    percentiles = np.percentile(df[threshold_col], np.arange(15, 86, 1))
    percentiles = np.unique(percentiles)
    
    best_tau = None
    min_ssr = np.inf
    best_model = None
    
    y = df[y_col]
    X_control = df[X_control_cols]
    
    for tau in percentiles:
        I_low = (df[threshold_col] <= tau).astype(int)
        I_high = (df[threshold_col] > tau).astype(int)
        
        d_low = df[split_col] * I_low
        d_high = df[split_col] * I_high
        
        X = X_control.copy()
        X['d_low (tau <= EMBI)'] = d_low
        X['d_high (tau > EMBI)'] = d_high
        X = sm.add_constant(X)
        
        try:
            model = sm.OLS(y, X).fit()
            if model.ssr < min_ssr:
                min_ssr = model.ssr
                best_tau = tau
                best_model = model
        except:
            continue
            
    if best_model is not None:
        I_low = (df[threshold_col] <= best_tau).astype(int)
        I_high = (df[threshold_col] > best_tau).astype(int)
        X_best = X_control.copy()
        X_best['d_low (EMBI <= tau)'] = df[split_col] * I_low
        X_best['d_high (EMBI > tau)'] = df[split_col] * I_high
        X_best = sm.add_constant(X_best)
        # Re-estimar con HAC para inferencia
        best_model_hac = sm.OLS(y, X_best).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return best_tau, best_model_hac
        
    return best_tau, best_model

def ejecutar_fase5_econometria(csv_path):
    print("=" * 75)
    print(" FASE 5: Modelación de Umbrales (Hansen 2000) y Fatiga Fiscal ")
    print("=" * 75)
    
    if not os.path.exists(csv_path):
        print(f"[!] ERROR: No se encontró el dataset en {csv_path}")
        return
        
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
    
    print("\n[1/3] Preparando variables (Sin datos simulados)...")
    df['d_t_1'] = df['deuda_pib'].shift(1)
    
    subset_cols = ['pb_pib', 'd_t_1', 'g_gap', 'EMBI']
    df = df.dropna(subset=subset_cols).copy()
    
    print(f" -> Observaciones válidas: {len(df)}")
    
    print("\n[2/3] Estimando Modelo de Umbral de Hansen (2000) para series de tiempo...")
    best_tau, best_model = get_optimal_threshold(
        df=df,
        y_col='pb_pib',
        X_control_cols=['g_gap'],
        split_col='d_t_1',
        threshold_col='EMBI'
    )
    
    print(f" -> Umbral Óptimo Encontrado (tau*): {best_tau:.1f} puntos básicos de EMBI+")
    
    print("\n[3/3] Test de Significancia del Umbral (Bootstrap Sup-LM)...")
    f_obs, p_val_boot = bootstrap_threshold_test(
        df=df,
        y_col='pb_pib',
        X_control_cols=['g_gap'],
        split_col='d_t_1',
        threshold_col='EMBI',
        best_tau=best_tau,
        n_boot=1000
    )
    
    print(f" -> F-stat observado: {f_obs:.4f}")
    print(f" -> p-value (Bootstrap 1000 iteraciones): {p_val_boot:.4f}")
    
    if p_val_boot < 0.10:
        print("    [OK] Se rechaza la linealidad (p < 0.10). El efecto umbral es estadísticamente significativo.")
    else:
        print("    [!] No se rechaza linealidad. El umbral detectado podría ser espurio.")

    print("\n--- RESULTADOS MODELO DE UMBRAL (Errores HAC) ---")
    print(best_model.summary().tables[1])
    
    alpha_1 = best_model.params['d_low (EMBI <= tau)']
    alpha_2 = best_model.params['d_high (EMBI > tau)']
    pval_2 = best_model.pvalues['d_high (EMBI > tau)']
    
    print("\n===================================================================")
    print(" Análisis de Fatiga Fiscal (Ghosh et al., 2013) ")
    print("===================================================================")
    print(f" -> Régimen NORMAL (EMBI <= {best_tau:.0f}): Reacción = {alpha_1:.4f}")
    print(f" -> Régimen ESTRÉS (EMBI >  {best_tau:.0f}): Reacción = {alpha_2:.4f} (p-value: {pval_2:.4f})")
    
    if alpha_2 <= 0 or pval_2 > 0.05:
        print("\n [ALERTA] Se confirma FATIGA FISCAL.")
        print(" En el régimen de alto estrés (riesgo soberano extremo), el Estado ")
        print(" pierde la capacidad de generar superávits primarios para estabilizar la deuda.")
    else:
        print("\n [SOSTENIBLE] El Estado logra sostener la reacción fiscal positiva incluso en crisis.")
        
    print("\nNota: El módulo de Dominancia Fiscal (Sargent-Wallace) ha sido retirado")
    print("      temporalmente hasta incorporar series reales de Pasivos Remunerados (BCRA).")

    # Guardar resultados
    os.makedirs("resultados/tablas", exist_ok=True)
    with open("resultados/tablas/fase5_resultados_hansen.csv", "w") as f:
        f.write(best_model.summary().as_csv())
    print("\n[OK] Resultados guardados en 'resultados/tablas/fase5_resultados_hansen.csv'")

if __name__ == "__main__":
    import pathlib
    base_dir = pathlib.Path(__file__).parent.parent.parent
    csv_file = base_dir / "datos" / "dataset_consolidado_real.csv"
    ejecutar_fase5_econometria(csv_file)


### Etapa 2.6 — Sostenibilidad de la deuda (DSA)

Proyecta la ratio Deuda/PIB a 2035 bajo tres escenarios deterministas y una simulación de Monte Carlo con *shocks* $t$ de Student.

Fuente: `codigo/modelos/fase6_sostenibilidad_deuda.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase6_sostenibilidad_deuda.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase6_sostenibilidad_deuda.py"

"""
fase6_sostenibilidad_deuda.py
=============================
Fase 6 del Protocolo Econométrico: Análisis de Sostenibilidad de la Deuda (DSA).

Implementa:
  - DSA Determinista (3 escenarios: Optimista, Referencia, Estrés).
  - DSA Estocástico (Monte Carlo con 1000 iteraciones) generando gráficos de abanico
    (Alineado con estándares del FMI, resolviendo HC-04).
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import multivariate_t

# Configuraciones para gráficos más limpios
import seaborn as sns
sns.set_theme(style="whitegrid")

# Grados de libertad de la t de Student multivariante, estimados empíricamente
# (MLE, scipy.stats.t.fit) sobre los shocks históricos reales de resultado
# primario (nu=5.26, curtosis en exceso=23.8) y tipo de cambio real
# (nu=4.95, curtosis en exceso=14.4) -las dos series con evidencia robusta de
# colas gordas-. El shock de crecimiento del PIB real no exhibe la misma
# propiedad (curtosis en exceso=-0.27, aproximadamente gaussiano) y se
# mantiene, por conservadurismo, bajo el mismo parámetro compartido de la
# t multivariada (ver script de estimación en el codebook del proyecto).
DSA_STUDENT_T_NU = 5.1

# Parámetros estructurales y de escenario, expuestos a nivel de módulo para que
# scripts complementarios (p.ej. fase7_diagnosticos_robustez.py) los reutilicen por
# import en lugar de duplicarlos, evitando que ambos archivos diverjan si se
# recalibra el DSA.
ALPHA = 0.463  # Proporción de deuda en pesos (ROS)
D_INITIAL = 0.740

SCENARIOS = {
    'Optimista':  {'pb': 0.025, 'g': 0.045,  'r_d': 0.040, 'r_f': 0.055, 'delta_e': -0.020},
    'Referencia': {'pb': 0.015, 'g': 0.035,  'r_d': 0.060, 'r_f': 0.075, 'delta_e': 0.000},
    'Estrés':     {'pb': 0.005, 'g': -0.015, 'r_d': 0.120, 'r_f': 0.150, 'delta_e': 0.250},
}

# Matriz de covarianza empírica (simplificada para volatilidad argentina).
# Orden: [pb, g, r_d, r_f, delta_e]. Volatilidades (std): pb=1.5%, g=4%, r_d=5%,
# r_f=2%, delta_e=15%. Correlaciones calibradas (e.g. g cae cuando delta_e sube).
STD_DEVS = np.array([0.015, 0.040, 0.050, 0.020, 0.150])
CORR = np.array([
    [ 1.0,  0.4, -0.2, -0.1, -0.3],  # pb
    [ 0.4,  1.0, -0.4, -0.2, -0.5],  # g
    [-0.2, -0.4,  1.0,  0.5,  0.6],  # r_d
    [-0.1, -0.2,  0.5,  1.0,  0.4],  # r_f
    [-0.3, -0.5,  0.6,  0.4,  1.0],  # delta_e
])


def build_cov_matrix():
    return np.outer(STD_DEVS, STD_DEVS) * CORR

def simulate_dsa_path(alpha, d_initial, pb, g, r_d, r_f, delta_e, sf, years):
    """
    Simula una trayectoria determinista de la deuda.
    """
    d = np.zeros(len(years))
    d[0] = d_initial
    
    for i in range(1, len(years)):
        term_d = alpha * (1 + r_d[i]) / (1 + g[i])
        term_f = (1 - alpha) * (1 + r_f[i]) * (1 + delta_e[i]) / (1 + g[i])
        M = term_d + term_f
        d[i] = M * d[i-1] - pb[i] + sf[i]
        
    return d

def simulate_stochastic_dsa(alpha, d_initial, base_params, cov_matrix, years, nu=DSA_STUDENT_T_NU, n_simulations=1000):
    """
    Simula trayectorias estocásticas usando Monte Carlo.
    base_params: medias de [pb, g, r_d, r_f, delta_e]
    cov_matrix: matriz de covarianza de los shocks
    nu: grados de libertad de la t de Student multivariante de los shocks
    """
    np.random.seed(42)
    n_years = len(years)

    # Matriz para almacenar las 1000 trayectorias
    d_paths = np.zeros((n_simulations, n_years))
    d_paths[:, 0] = d_initial

    # Matriz de escala de la t multivariada: se reescala el calibrado cov_matrix
    # (Var[X] = shape * nu/(nu-2) para una t multivariada) de modo que la
    # varianza efectiva de los shocks preserve exactamente la calibración
    # histórica original, añadiendo únicamente el exceso de curtosis (colas
    # gordas) que dicha calibración gaussiana no capturaba.
    scale_matrix = cov_matrix * (nu - 2) / nu

    for s in range(n_simulations):
        # Generar shocks multivariados para todos los años (t de Student,
        # nu=5.1, en lugar de perturbaciones gaussianas puras)
        shocks = multivariate_t.rvs(loc=np.zeros(5), shape=scale_matrix, df=nu, size=n_years)

        pb_s = np.full(n_years, base_params['pb']) + shocks[:, 0]
        g_s = np.full(n_years, base_params['g']) + shocks[:, 1]
        rd_s = np.full(n_years, base_params['r_d']) + shocks[:, 2]
        rf_s = np.full(n_years, base_params['r_f']) + shocks[:, 3]
        delta_e_s = np.full(n_years, base_params['delta_e']) + shocks[:, 4]
        sf_s = np.zeros(n_years) # Stock-flow residual asumido cero en MC base
        
        for i in range(1, n_years):
            term_d = alpha * (1 + rd_s[i]) / (1 + g_s[i])
            term_f = (1 - alpha) * (1 + rf_s[i]) * (1 + delta_e_s[i]) / (1 + g_s[i])
            M = term_d + term_f
            d_paths[s, i] = M * d_paths[s, i-1] - pb_s[i] + sf_s[i]
            
    return d_paths

def main():
    print("=" * 75)
    print(" FASE 6: Proyecciones DSA (Deterministas y Estocásticas) ")
    print("=" * 75)
    
    years = np.arange(2026, 2036)
    n_years = len(years)

    # Parámetro estructural
    alpha = ALPHA

    # ------------------------------------------------------------------
    # 1. DSA DETERMINISTA
    # ------------------------------------------------------------------
    print("\n[1/2] Ejecutando DSA Determinista (3 Escenarios)...")

    def expand(val): return np.full(n_years, val)

    sf_by_scenario = {'Optimista': 0.0, 'Referencia': 0.0, 'Estrés': 0.050}

    # Escenario Optimista (D_inicial ajustado a 74% según ROS)
    p = SCENARIOS['Optimista']
    d_opt = simulate_dsa_path(alpha, D_INITIAL, expand(p['pb']), expand(p['g']), expand(p['r_d']), expand(p['r_f']), expand(p['delta_e']), expand(sf_by_scenario['Optimista']), years)

    # Escenario Referencia
    p = SCENARIOS['Referencia']
    d_ref = simulate_dsa_path(alpha, D_INITIAL, expand(p['pb']), expand(p['g']), expand(p['r_d']), expand(p['r_f']), expand(p['delta_e']), expand(sf_by_scenario['Referencia']), years)

    # Escenario Estrés
    p = SCENARIOS['Estrés']
    d_est = simulate_dsa_path(alpha, D_INITIAL, expand(p['pb']), expand(p['g']), expand(p['r_d']), expand(p['r_f']), expand(p['delta_e']), expand(sf_by_scenario['Estrés']), years)

    df_det = pd.DataFrame({
        'Año': years,
        'Optimista': d_opt,
        'Referencia': d_ref,
        'Estrés': d_est
    })
    
    print(df_det.to_string(index=False))

    base_dir = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
    output_dir = os.path.join(base_dir, 'tesis', 'figuras')
    os.makedirs(output_dir, exist_ok=True)

    tables_dir = os.path.join(base_dir, 'resultados', 'tablas')
    os.makedirs(tables_dir, exist_ok=True)
    df_det.to_csv(os.path.join(tables_dir, 'fase6_dsa_determinista.csv'), index=False)
    
    # Gráfico Determinista
    plt.figure(figsize=(10, 6))
    plt.plot(years, d_opt * 100, label='Optimista (Superávit 2.5%, Crecimiento 4.5%)', color='green', marker='o')
    plt.plot(years, d_ref * 100, label='Referencia (Superávit 1.5%, Crecimiento 3.5%)', color='blue', marker='s')
    plt.plot(years, d_est * 100, label='Estrés (Contracción, Shock Cambiario)', color='red', marker='^')
    plt.axhline(100, color='black', linestyle='--', alpha=0.5, label='Frontera Crítica (100%)')
    plt.title('DSA Determinista: Deuda Pública Consolidada Neta (% PIB)')
    plt.xlabel('Año')
    plt.ylabel('% PIB')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(output_dir, 'dsa_determinista.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ------------------------------------------------------------------
    # 2. DSA ESTOCÁSTICO (GRÁFICO DE ABANICO)
    # ------------------------------------------------------------------
    print("\n[2/2] Ejecutando DSA Estocástico (Monte Carlo - 1000 simulaciones)...")

    # Parámetros base (usamos los de Referencia)
    base_params = SCENARIOS['Referencia']
    cov_matrix = build_cov_matrix()

    d_paths = simulate_stochastic_dsa(alpha, D_INITIAL, base_params, cov_matrix, years, DSA_STUDENT_T_NU, n_simulations=1000)
    
    # Calcular percentiles
    p10 = np.percentile(d_paths, 10, axis=0) * 100
    p25 = np.percentile(d_paths, 25, axis=0) * 100
    p50 = np.percentile(d_paths, 50, axis=0) * 100
    p75 = np.percentile(d_paths, 75, axis=0) * 100
    p90 = np.percentile(d_paths, 90, axis=0) * 100
    
    # Gráfico de abanico
    plt.figure(figsize=(10, 6))
    
    # Rellenar áreas
    plt.fill_between(years, p10, p90, color='blue', alpha=0.1, label='Intervalo 10-90%')
    plt.fill_between(years, p25, p75, color='blue', alpha=0.3, label='Intervalo 25-75%')
    
    # Mediana y frontera
    plt.plot(years, p50, color='darkblue', linewidth=2, label='Mediana (Proyección Base)')
    plt.axhline(100, color='red', linestyle='--', alpha=0.7, label='Frontera Crítica (100%)')
    
    plt.title('DSA Estocástico (Gráfico de Abanico): Riesgo de Sostenibilidad (2026-2035)')
    plt.xlabel('Año')
    plt.ylabel('% PIB')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(output_dir, 'dsa_grafico_abanico.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Reportar probabilidad de quiebre
    prob_crisis_final = np.mean(d_paths[:, -1] > 1.0) * 100
    print(f"\n -> Probabilidad de superar la frontera crítica (100% PIB) en {years[-1]}: {prob_crisis_final:.1f}%")

    df_percentiles = pd.DataFrame({'Año': years, 'p10': p10, 'p25': p25, 'p50': p50, 'p75': p75, 'p90': p90})
    df_percentiles.to_csv(os.path.join(tables_dir, 'fase6_dsa_percentiles_estocastico.csv'), index=False)
    with open(os.path.join(tables_dir, 'fase6_dsa_probabilidad.csv'), 'w') as f:
        f.write('year,prob_exceeds_100pct\n')
        f.write(f'{years[-1]},{prob_crisis_final:.4f}\n')

    print(f"\n[OK] Gráficos guardados en:\n - {os.path.join(output_dir, 'dsa_determinista.png')}\n - {os.path.join(output_dir, 'dsa_grafico_abanico.png')}")

if __name__ == '__main__':
    main()


### Etapa 2.7 — Diagnósticos de robustez

ACF/PACF, ARCH-LM, CUSUM/CUSUMSQ, Engle-Granger, sensibilidad temporal, causalidad de Granger, filtro de Hamilton y robustez GARCH(1,1) del DSA.

Fuente: `codigo/modelos/fase7_diagnosticos_robustez.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase7_diagnosticos_robustez.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase7_diagnosticos_robustez.py"

"""
fase7_diagnosticos_robustez.py
==========================
Diagnosticos adicionales solicitados por la auditoria academica externa,
computados sobre el proceso empírico real (no fabricados):

  1. ACF/PACF de las 4 series nucleo (pb_pib, deuda_pib, EMBI, g_gap).
  2. Test ARCH-LM (heterocedasticidad condicional) sobre los residuos del DOLS.
  3. CUSUM y CUSUMSQ (estabilidad estructural del DOLS, residuos recursivos).
  4. Test de cointegracion residual Engle-Granger (deuda_pib <-> pb_pib),
     como robustez complementaria en el mismo espiritu que Phillips-Ouliaris
     (ambos son tests de cointegracion de ecuacion unica basados en residuos;
     statsmodels no implementa Phillips-Ouliaris de forma nativa, por lo que
     se reporta Engle-Granger y se lo declara honestamente como tal).
  5. Sensibilidad temporal: reestimacion de la funcion de reaccion fiscal
     (MCO-HAC) en dos submuestras de igual tamano (2004-2014 vs 2015-2025).
  6. Probabilidad de insolvencia (Monte Carlo) por escenario (Optimista,
     Referencia, Estres), no solo bajo el escenario de Referencia.
  7. Causalidad de Granger (EMBI+ <-> Delta Resultado Primario), ambas
     direcciones, como evidencia complementaria y barata sobre la endogeneidad
     que motiva la Seccion 6.4 (IV-2SLS).
  8. Filtro de Hamilton (2018) como robustez del filtro HP (lambda=1600)
     usado para la brecha del producto (Cap. 5), comparando ambos ciclos.
  9. Covarianza del DSA: reemplazo parcial de la calibracion a ojo por
     GARCH(1,1) para las variables con serie real disponible (pb, g,
     delta_e); r_d y r_f permanecen calibrados por falta de serie propia.

Todos los resultados se guardan en resultados/tablas/ para trazabilidad.
"""

import os
import sys
import pathlib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import recursive_olsresiduals, het_arch
from statsmodels.tsa.stattools import coint, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

sys.path.insert(0, str(pathlib.Path(__file__).parent))
from fase6_sostenibilidad_deuda import (
    simulate_stochastic_dsa, SCENARIOS, ALPHA, D_INITIAL, DSA_STUDENT_T_NU, build_cov_matrix,
    STD_DEVS, CORR,
)
from arch import arch_model

BASE_DIR = pathlib.Path(__file__).parent.parent.parent
CSV_PATH = BASE_DIR / "datos" / "dataset_consolidado_real.csv"
LATEX_DIR = BASE_DIR / "tesis" / "figuras"
TABLES_DIR = BASE_DIR / "resultados" / "tablas"
os.makedirs(TABLES_DIR, exist_ok=True)


def newey_west_lags(n):
    return int(np.ceil(4 * (n / 100) ** (2 / 9)))


def build_dols(df, max_m=4):
    df = df.copy()
    df['d_t_1'] = df['deuda_pib'].shift(1)
    df = df.dropna(subset=['pb_pib', 'd_t_1', 'g_gap'])
    df['diff_d'] = df['d_t_1'].diff()
    dols_features = ['d_t_1', 'g_gap', 'diff_d']
    for i in range(1, max_m + 1):
        df[f'diff_d_lag_{i}'] = df['diff_d'].shift(i)
        df[f'diff_d_lead_{i}'] = df['diff_d'].shift(-i)
        dols_features.extend([f'diff_d_lag_{i}', f'diff_d_lead_{i}'])
    df = df.dropna(subset=['pb_pib'] + dols_features)
    y = df['pb_pib']
    X = sm.add_constant(df[dols_features])
    n = len(df)
    hac_lags = newey_west_lags(n)
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
    model_ols = sm.OLS(y, X).fit()  # sin HAC, necesario para residuos recursivos
    return df, y, X, model, model_ols


def section_1_acf_pacf(df):
    print("\n[1/9] ACF/PACF de las 4 series nucleo...")
    series = {
        'Resultado Primario / PIB ($pb_t$)': df['pb_pib'],
        'Deuda Publica / PIB ($d_t$)': df['deuda_pib'],
        'Riesgo Pais - EMBI+ ($risk_t$)': df['EMBI'],
        'Brecha del Producto ($\\tilde{y}_t$)': df['g_gap'],
    }
    fig, axes = plt.subplots(4, 2, figsize=(10, 14))
    for i, (name, s) in enumerate(series.items()):
        s = s.dropna()
        plot_acf(s, ax=axes[i, 0], lags=20, title=f'ACF: {name}')
        plot_pacf(s, ax=axes[i, 1], lags=20, method='ywm', title=f'PACF: {name}')
    plt.tight_layout()
    out_path = LATEX_DIR / 'figura_5_5_acf_pacf.png'
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f" -> Guardado: {out_path}")


def section_2_arch(model):
    print("\n[2/9] Test ARCH-LM sobre residuos del DOLS...")
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(model.resid, nlags=4)
    print(f" -> LM stat = {lm_stat:.4f}, p-valor = {lm_pvalue:.4f}")
    pd.DataFrame([{
        'lm_stat': lm_stat, 'lm_pvalue': lm_pvalue,
        'f_stat': f_stat, 'f_pvalue': f_pvalue
    }]).to_csv(TABLES_DIR / 'auditoria_test_arch.csv', index=False)
    return lm_stat, lm_pvalue, f_stat, f_pvalue


def section_3_cusum(model_ols, y, X):
    print("\n[3/9] CUSUM / CUSUMSQ (residuos recursivos)...")
    (rresid, rparams, rypred, rresid_standardized, rresid_scaled,
     rcusum, rcusumci) = recursive_olsresiduals(model_ols, skip=None, alpha=0.95)

    n_r = len(rcusum)
    n_ci = rcusumci.shape[1]
    idx = np.arange(1, n_r + 1)
    idx_ci = np.arange(n_r - n_ci + 1, n_r + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(idx, rcusum, color='#0B3C5D', label='CUSUM')
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].plot(idx_ci, rcusumci[0, :], 'r--', linewidth=1, label='Banda 95%')
    axes[0].plot(idx_ci, rcusumci[1, :], 'r--', linewidth=1)
    axes[0].set_title('CUSUM (Estabilidad Estructural, DOLS)')
    axes[0].set_xlabel('Observación recursiva')
    axes[0].legend(fontsize=8)

    rresid_scaled_aligned = rresid_scaled[-n_r:]
    cusumsq = np.cumsum(rresid_scaled_aligned ** 2) / np.sum(rresid_scaled_aligned ** 2)
    frac = np.arange(1, n_r + 1) / n_r
    c95 = 0.5959  # límite aproximado al 5% (Harvey, 1990) para muestras moderadas
    axes[1].plot(idx, cusumsq, color='#0B3C5D', label='CUSUMSQ')
    axes[1].plot(idx, np.clip(frac + c95, 0, 1.3), 'r--', linewidth=1, label='Banda 95%')
    axes[1].plot(idx, np.clip(frac - c95, -0.3, 1), 'r--', linewidth=1)
    axes[1].set_title('CUSUMSQ (Estabilidad Estructural, DOLS)')
    axes[1].set_xlabel('Observación recursiva')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    out_path = LATEX_DIR / 'figura_6_2_cusum.png'
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f" -> Guardado: {out_path}")

    cusum_ci_matched = rcusum[-n_ci:]
    exceeds_cusum = np.any((cusum_ci_matched < rcusumci[0, :]) | (cusum_ci_matched > rcusumci[1, :]))
    stable_cusum = not exceeds_cusum
    max_cusumsq_dev = np.max(np.abs(cusumsq - frac))
    stable_cusumsq = max_cusumsq_dev < c95

    print(f" -> CUSUM se mantiene dentro de bandas al 95%: {stable_cusum}")
    print(f" -> CUSUMSQ desviación máxima = {max_cusumsq_dev:.4f} (límite 5% = {c95}) -> estable: {stable_cusumsq}")

    pd.DataFrame([{
        'stable_cusum': stable_cusum,
        'max_cusumsq_deviation': max_cusumsq_dev,
        'cusumsq_boundary_5pct': c95,
        'stable_cusumsq': stable_cusumsq
    }]).to_csv(TABLES_DIR / 'auditoria_cusum.csv', index=False)
    return stable_cusum, stable_cusumsq, max_cusumsq_dev


def section_4_engle_granger(df):
    print("\n[4/9] Test de cointegracion residual Engle-Granger (deuda_pib <-> pb_pib)...")
    d = df.dropna(subset=['deuda_pib', 'pb_pib'])
    eg_stat, eg_pvalue, eg_crit = coint(d['deuda_pib'], d['pb_pib'], trend='c')
    print(f" -> Estadistico = {eg_stat:.4f}, p-valor = {eg_pvalue:.4f}")
    print(f" -> Valores criticos (1%,5%,10%) = {eg_crit}")
    pd.DataFrame([{
        'eg_stat': eg_stat, 'eg_pvalue': eg_pvalue,
        'crit_1pct': eg_crit[0], 'crit_5pct': eg_crit[1], 'crit_10pct': eg_crit[2]
    }]).to_csv(TABLES_DIR / 'auditoria_engle_granger.csv', index=False)
    return eg_stat, eg_pvalue, eg_crit


def section_5_subperiod_sensitivity(df):
    print("\n[5/9] Sensibilidad temporal: submuestras 2004-2014 vs 2015-2025...")
    df = df.copy()
    df['d_t_1'] = df['deuda_pib'].shift(1)
    df = df.dropna(subset=['pb_pib', 'd_t_1', 'g_gap'])

    sub1 = df[df.index <= '2014-12-31']
    sub2 = df[df.index >= '2015-01-01']

    results = {}
    for label, sub in [('2004-2014', sub1), ('2015-2025', sub2)]:
        y = sub['pb_pib']
        X = sm.add_constant(sub[['d_t_1', 'g_gap']])
        hac_lags = newey_west_lags(len(sub))
        m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
        results[label] = {
            'n': len(sub),
            'rho': m.params['d_t_1'],
            'se': m.bse['d_t_1'],
            'p': m.pvalues['d_t_1'],
            'r2': m.rsquared
        }
        print(f" -> {label} (n={len(sub)}): rho={m.params['d_t_1']:.4f} (p={m.pvalues['d_t_1']:.4f})")

    pd.DataFrame(results).T.to_csv(TABLES_DIR / 'auditoria_sensibilidad_subperiodos.csv')
    return results


def section_6_scenario_probabilities():
    print("\n[6/9] Probabilidad de insolvencia (Monte Carlo) por escenario...")
    print("      (reutilizando simulate_stochastic_dsa, SCENARIOS y build_cov_matrix de fase6_sostenibilidad_deuda.py)")
    years = np.arange(2026, 2036)
    cov_matrix = build_cov_matrix()

    rows = []
    for name, params in SCENARIOS.items():
        d_paths = simulate_stochastic_dsa(ALPHA, D_INITIAL, params, cov_matrix, years, DSA_STUDENT_T_NU, n_simulations=1000)
        prob_100 = np.mean(d_paths[:, -1] > 1.0) * 100
        median_2035 = np.median(d_paths[:, -1]) * 100
        print(f" -> {name}: P(deuda/PIB > 100% en 2035) = {prob_100:.1f}% (mediana={median_2035:.1f}%)")
        rows.append({'escenario': name, 'prob_excede_100pct_2035': prob_100, 'mediana_2035': median_2035})

    pd.DataFrame(rows).to_csv(TABLES_DIR / 'auditoria_probabilidades_escenarios.csv', index=False)
    return rows


def section_7_granger_causality(df, maxlag=4):
    """
    Causalidad de Granger EMBI+ <-> Resultado Primario, en ambas direcciones.
    EMBI (risk_t) es I(0) (Tabla 5.6); pb_pib es I(1) (misma tabla), por lo
    que se usa Delta pb_pib (estacionaria) junto al EMBI+ en niveles, evitando
    una regresion espuria sin necesitar el aparato completo de un VECM.
    """
    print("\n[7/9] Causalidad de Granger (EMBI+ <-> Resultado Primario)...")
    d = df.copy()
    d['d_pb'] = d['pb_pib'].diff()
    d = d.dropna(subset=['d_pb', 'EMBI'])

    rows = []
    # Direccion 1: EMBI+ ayuda a predecir Delta pb_pib?
    data_1 = d[['d_pb', 'EMBI']].values
    res_1 = grangercausalitytests(data_1, maxlag=maxlag, verbose=False)
    # Direccion 2: Delta pb_pib ayuda a predecir EMBI+?
    data_2 = d[['EMBI', 'd_pb']].values
    res_2 = grangercausalitytests(data_2, maxlag=maxlag, verbose=False)

    for lag in range(1, maxlag + 1):
        f_1, p_1 = res_1[lag][0]['ssr_ftest'][0], res_1[lag][0]['ssr_ftest'][1]
        f_2, p_2 = res_2[lag][0]['ssr_ftest'][0], res_2[lag][0]['ssr_ftest'][1]
        rows.append({
            'rezagos': lag,
            'F_EMBI_causa_dpb': f_1, 'p_EMBI_causa_dpb': p_1,
            'F_dpb_causa_EMBI': f_2, 'p_dpb_causa_EMBI': p_2,
        })
        print(f" -> Rezagos={lag}: EMBI+ -> Dpb (F={f_1:.3f}, p={p_1:.4f}) | "
              f"Dpb -> EMBI+ (F={f_2:.3f}, p={p_2:.4f})")

    pd.DataFrame(rows).to_csv(TABLES_DIR / 'auditoria_causalidad_granger.csv', index=False)
    return rows


def section_8_hamilton_filter(df, h=8, p=4):
    """
    Filtro de Hamilton (2018) como robustez del filtro HP (lambda=1600, Cap. 5).
    Especificacion recomendada por el propio Hamilton para datos trimestrales:
    regresion de y_{t+h} sobre una constante y 4 rezagos de y_t (h=8, p=4).
    El residuo de esa regresion es el componente ciclico (analogo a g_gap).
    """
    print("\n[8/9] Filtro de Hamilton (2018) como robustez del HP (lambda=1600)...")
    y = np.log(df['PIB_real']).dropna()
    n = len(y)

    X_cols = {f'y_lag{h + i}': y.shift(h + i) for i in range(p)}
    reg_df = pd.DataFrame(X_cols)
    reg_df['y'] = y
    reg_df = reg_df.dropna()

    X = sm.add_constant(reg_df[list(X_cols.keys())])
    model = sm.OLS(reg_df['y'], X).fit()
    cycle_hamilton = model.resid * 100  # en % de desvio, comparable a g_gap

    comp = pd.DataFrame({'g_gap_HP': df['g_gap'], 'ciclo_Hamilton': cycle_hamilton}).dropna()
    corr = comp['g_gap_HP'].corr(comp['ciclo_Hamilton'])
    print(f" -> n valido tras rezagos h={h}+p={p}: {len(comp)} de {n} observaciones")
    print(f" -> Correlacion entre brecha HP (lambda=1600) y ciclo de Hamilton: r={corr:.4f}")

    comp.to_csv(TABLES_DIR / 'auditoria_filtro_hamilton.csv')
    return comp, corr


def _garch_annual_std(series, annualize=True):
    """Ajusta un GARCH(1,1) a una serie (en % para estabilidad numerica) y
    devuelve la volatilidad condicional promedio. Si annualize=True, escala
    por sqrt(4) asumiendo que la serie es de innovaciones trimestrales sin
    componente estacional (varianza aditiva bajo shocks no correlacionados);
    si la serie ya es interanual (YoY), se deja sin reescalar."""
    s = series.dropna() * 100
    am = arch_model(s, mean='Zero', vol='GARCH', p=1, q=1, dist='normal')
    res = am.fit(disp='off')
    avg_cond_vol = np.sqrt(res.conditional_volatility ** 2).mean() / 100
    return avg_cond_vol * np.sqrt(4) if annualize else avg_cond_vol


def section_9_garch_dsa_covariance(df):
    """
    Reemplaza, para las variables con serie historica real disponible en el
    dataset (pb, g, delta_e), la volatilidad calibrada a ojo de
    fase6_sostenibilidad_deuda.py por una volatilidad estimada mediante
    GARCH(1,1). r_d y r_f (tasas de interes domestica y externa) no tienen
    serie propia en el dataset consolidado -> se mantienen calibradas, de
    forma explicita, y la matriz de correlacion tampoco se re-estima (exigiria
    las 5 series reales). Es una mejora parcial y honesta, no una
    re-estimacion completa del proceso estocastico del DSA.
    """
    print("\n[9/9] Covarianza del DSA: reemplazo parcial de la calibracion por GARCH(1,1)...")
    d = df.copy()
    pb_shock = d['pb_pib'].diff() / 100           # decimal, primera diferencia (pb_pib es I(1))
    delta_e_shock = d['TCRM'].pct_change()        # decimal, variacion trimestral del TCRM (sin estacionalidad marcada)
    # PIB_real tiene estacionalidad marcada por trimestre (verificado: media Q2 muy
    # por encima de Q1/Q3/Q4), por lo que la variacion trimestral simple confundiria
    # estacionalidad con volatilidad. Se usa crecimiento interanual (YoY, 4 rezagos),
    # que cancela el componente estacional y ya es directamente una tasa anual.
    g_shock = d['PIB_real'].pct_change(4)

    std_pb = _garch_annual_std(pb_shock)
    std_g = _garch_annual_std(g_shock, annualize=False)
    std_delta_e = _garch_annual_std(delta_e_shock)

    std_devs_garch = STD_DEVS.copy()
    std_devs_garch[0] = std_pb        # pb: calibrado 0.015 -> GARCH
    std_devs_garch[1] = std_g         # g: calibrado 0.040 -> GARCH
    std_devs_garch[4] = std_delta_e   # delta_e: calibrado 0.150 -> GARCH
    # r_d (idx 2) y r_f (idx 3): se mantienen calibrados, sin serie real disponible.

    print(f" -> Desvios calibrados originales : pb={STD_DEVS[0]:.4f}  g={STD_DEVS[1]:.4f}  delta_e={STD_DEVS[4]:.4f}")
    print(f" -> Desvios GARCH(1,1) anualizados : pb={std_pb:.4f}  g={std_g:.4f}  delta_e={std_delta_e:.4f}")
    print(" -> r_d y r_f permanecen calibrados (sin serie historica propia en el dataset).")

    cov_garch = np.outer(std_devs_garch, std_devs_garch) * CORR

    years = np.arange(2026, 2036)
    rows = []
    for name, params in SCENARIOS.items():
        d_paths_orig = simulate_stochastic_dsa(ALPHA, D_INITIAL, params, build_cov_matrix(), years, DSA_STUDENT_T_NU, n_simulations=1000)
        d_paths_garch = simulate_stochastic_dsa(ALPHA, D_INITIAL, params, cov_garch, years, DSA_STUDENT_T_NU, n_simulations=1000)
        prob_orig = np.mean(d_paths_orig[:, -1] > 1.0) * 100
        prob_garch = np.mean(d_paths_garch[:, -1] > 1.0) * 100
        print(f" -> {name}: P(>100% PIB, 2035) calibrado={prob_orig:.1f}%  vs.  GARCH-parcial={prob_garch:.1f}%")
        rows.append({'escenario': name, 'prob_calibrado': prob_orig, 'prob_garch_parcial': prob_garch})

    out = pd.DataFrame(rows)
    out.to_csv(TABLES_DIR / 'auditoria_garch_comparacion_dsa.csv', index=False)
    pd.DataFrame([{
        'std_pb_calibrado': STD_DEVS[0], 'std_pb_garch': std_pb,
        'std_g_calibrado': STD_DEVS[1], 'std_g_garch': std_g,
        'std_delta_e_calibrado': STD_DEVS[4], 'std_delta_e_garch': std_delta_e,
    }]).to_csv(TABLES_DIR / 'auditoria_garch_desvios.csv', index=False)
    return out


def main():
    print("=" * 75)
    print(" EXTENSIONES DE DIAGNOSTICO - AUDITORIA ACADEMICA EXTERNA ")
    print("=" * 75)
    df = pd.read_csv(CSV_PATH, parse_dates=['Date'], index_col='Date')

    section_1_acf_pacf(df)

    df_dols, y, X, model_hac, model_ols = build_dols(df, max_m=4)
    section_2_arch(model_hac)
    section_3_cusum(model_ols, y, X)
    section_4_engle_granger(df)
    section_5_subperiod_sensitivity(df)
    section_6_scenario_probabilities()
    section_7_granger_causality(df)
    section_8_hamilton_filter(df)
    section_9_garch_dsa_covariance(df)

    print("\n[OK] Todos los diagnosticos guardados en resultados/tablas/ y tesis/figuras/.")


if __name__ == '__main__':
    main()


### Etapa 2.8 — Deuda consolidada (SPNF + BCRA)

Suma los pasivos remunerados del BCRA a la deuda del SPNF y reestima la estacionariedad de la serie consolidada.

Fuente: `codigo/modelos/fase8_deuda_consolidada.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase8_deuda_consolidada.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase8_deuda_consolidada.py"

"""
fase8_deuda_consolidada.py
===========================
Mejora Dimensión IV: consolidación de la deuda del Sector Público Nacional
No Financiero (SPNF, `deuda_pib`) con los pasivos remunerados del BCRA
(LEBAC/NOBAC, LELIQ/NOTALIQ y posición neta de pases, `pasivos_bcra_pib`,
construidos en `codigo/ingesta_datos/ingesta_bcra_pasivos.py` a partir de series
oficiales del BCRA v4.0 y de PIB nominal INDEC).

  deuda_consolidada_pib_t = deuda_pib_t + pasivos_bcra_pib_t

No se aplica neteo adicional entre ambas series: `deuda_pib` (SPNF) y
`pasivos_bcra_ars` (pasivos del BCRA frente al sistema financiero) son
instrumentos emitidos por entidades y a acreedores distintos, por lo que no
existe una tenencia cruzada directa entre ambos stocks que deba eliminarse
(a diferencia de, por ejemplo, las Letras Intransferibles que el Tesoro
coloca en el activo del BCRA como contrapartida de reservas, que ya están
implícitamente netas en la cifra de deuda "neta" del SPNF y no vuelven a
sumarse aquí).

Salidas:
  - datos/dataset_consolidado_real.csv actualizado con las columnas
    `pasivos_bcra_pib` y `deuda_consolidada_pib`.
  - resultados/tablas/fase8_deuda_consolidada.csv: comparación trimestral.
  - tesis/figuras/figura_8_1_deuda_consolidada.png: SPNF vs SPNF+BCRA.
"""

import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

warnings.filterwarnings("ignore")

BASE_DIR = pathlib.Path(__file__).parent.parent.parent
CSV_PATH = BASE_DIR / "datos" / "dataset_consolidado_real.csv"
BCRA_PASIVOS_PATH = BASE_DIR / "datos" / "procesados" / "bcra_pasivos_trimestral.csv"
LATEX_DIR = BASE_DIR / "tesis" / "figuras"
TABLES_DIR = BASE_DIR / "resultados" / "tablas"
os.makedirs(TABLES_DIR, exist_ok=True)


def main():
    print("=" * 75)
    print(" FASE 8: Consolidación de la Deuda SPNF + Pasivos Remunerados BCRA")
    print("=" * 75)

    df = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")
    bcra = pd.read_csv(BCRA_PASIVOS_PATH, parse_dates=["fecha"], index_col="fecha")

    # El dataset consolidado puede ya traer estas columnas de una ejecución
    # previa de este mismo script; se recalculan siempre desde la fuente BCRA
    # en lugar de arrastrar valores viejos, para que el script sea re-ejecutable
    # de forma idempotente sin colisión de nombres en el merge.
    columnas_derivadas = ["pasivos_bcra_pib", "pib_nominal_es_extrapolado", "deuda_consolidada_pib"]
    df = df.drop(columns=[c for c in columnas_derivadas if c in df.columns])

    print(f"\n[1/4] Fusionando pasivos_bcra_pib ({len(bcra)} trimestres) "
          f"con el dataset consolidado ({len(df)} trimestres)...")
    df = df.merge(bcra[["pasivos_bcra_pib", "pib_nominal_es_extrapolado"]],
                   left_index=True, right_index=True, how="left")
    df["deuda_consolidada_pib"] = df["deuda_pib"] + df["pasivos_bcra_pib"]

    print("\n[2/4] Estadísticas comparativas (deuda SPNF vs. deuda consolidada SPNF+BCRA)...")
    resumen = df[["deuda_pib", "pasivos_bcra_pib", "deuda_consolidada_pib"]].describe().round(2)
    print(resumen.to_string())

    brecha_media = df["pasivos_bcra_pib"].mean()
    pico = df["pasivos_bcra_pib"].idxmax()
    print(f"\n -> Brecha promedio (2004-2025) por cuasi-fiscal BCRA: {brecha_media:.2f} pp del PIB")
    print(f" -> Pico de pasivos remunerados BCRA/PIB: {df.loc[pico, 'pasivos_bcra_pib']:.2f}% "
          f"en {pico.date()} (deuda SPNF={df.loc[pico, 'deuda_pib']:.1f}%, "
          f"deuda consolidada={df.loc[pico, 'deuda_consolidada_pib']:.1f}%)")

    print("\n[3/4] Estacionariedad de la serie consolidada (ADF/KPSS, robustez)...")
    d_cons = df["deuda_consolidada_pib"].dropna()
    adf_p = adfuller(d_cons, autolag="AIC")[1]
    kpss_p = kpss(d_cons, regression="c", nlags="auto")[1]
    adf_p_diff = adfuller(d_cons.diff().dropna(), autolag="AIC")[1]
    kpss_p_diff = kpss(d_cons.diff().dropna(), regression="c", nlags="auto")[1]
    print(f" -> Niveles:   ADF p={adf_p:.4f} | KPSS p={kpss_p:.4f}")
    print(f" -> Diferencias: ADF p={adf_p_diff:.4f} | KPSS p={kpss_p_diff:.4f}")

    print("\n[4/4] Guardando resultados...")
    df.to_csv(CSV_PATH)
    print(f" -> Dataset consolidado actualizado: {CSV_PATH}")

    tabla = df[["deuda_pib", "pasivos_bcra_pib", "deuda_consolidada_pib"]].copy()
    tabla.to_csv(TABLES_DIR / "fase8_deuda_consolidada.csv")
    pd.DataFrame([{
        "adf_p_nivel": adf_p, "kpss_p_nivel": kpss_p,
        "adf_p_diff": adf_p_diff, "kpss_p_diff": kpss_p_diff,
        "brecha_media_pp_pib": brecha_media,
        "pico_pasivos_bcra_pib": df["pasivos_bcra_pib"].max(),
        "fecha_pico": str(pico.date()),
    }]).to_csv(TABLES_DIR / "fase8_deuda_consolidada_diagnosticos.csv", index=False)
    print(f" -> Tabla comparativa: {TABLES_DIR / 'fase8_deuda_consolidada.csv'}")

    plt.figure(figsize=(10, 6))
    plt.plot(df.index, df["deuda_pib"], label="Deuda SPNF / PIB (original)",
              color="#0B3C5D", linewidth=1.8)
    plt.plot(df.index, df["deuda_consolidada_pib"],
              label="Deuda consolidada SPNF + Pasivos Remunerados BCRA / PIB",
              color="#B33951", linewidth=1.8, linestyle="--")
    plt.fill_between(df.index, df["deuda_pib"], df["deuda_consolidada_pib"],
                      color="#B33951", alpha=0.15, label="Brecha cuasi-fiscal (BCRA)")
    plt.title("Deuda Pública: SPNF vs. Consolidada con Pasivos Remunerados del BCRA")
    plt.xlabel("Trimestre")
    plt.ylabel("% del PIB")
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_fig = LATEX_DIR / "figura_8_1_deuda_consolidada.png"
    plt.savefig(out_fig, dpi=300)
    plt.close()
    print(f" -> Gráfico: {out_fig}")

    print("\n[OK] Fase 8 completa.")


if __name__ == "__main__":
    main()


### Etapa 2.9a — Algoritmo de quiebres múltiples de Bai-Perron (implementación)

Implementación propia en Python puro (programación dinámica exacta) del algoritmo de Bai y Perron (1998, 2003), sin depender de `strucchange` de R.

Fuente: `codigo/modelos/bai_perron.py`

In [ ]:
"""
bai_perron.py
=============
Implementación formal, desde cero, del procedimiento de quiebres estructurales
múltiples de Bai y Perron (1998, 2003) para un modelo de cambio puro en el
nivel de la media de la serie, aplicable a cualquier serie univariada.

No existe en el ecosistema de Python un equivalente directo de
`strucchange::breakpoints()` de R (que a su vez requiere R/rpy2, no
disponibles en este entorno). Este módulo reproduce el núcleo del algoritmo
de Bai-Perron -programación dinámica sobre la suma de cuadrados residuales
(SSR) de todas las particiones factibles- en Python puro/NumPy:

  1. Para una serie y_1,...,y_T, se computa vía sumas prefijas la SSR de
     ajustar una media constante a cualquier segmento contiguo [i,j] en
     tiempo O(1) por consulta.
  2. Programación dinámica exacta (Bai & Perron, 2003, sec. 3.1): para cada
     número de quiebres m=0,...,M, se halla la partición global que minimiza
     la SSR total, respetando el recorte (trimming) de tamaño mínimo de
     segmento h = ceil(trimming * T).
  3. Selección del número de quiebres mediante el criterio BIC (Yao, 1988),
     una de las dos reglas de selección explícitamente recomendadas por
     Bai y Perron (2003, sec. 4) cuando no se dispone de las tablas de
     valores críticos no estándar del contraste secuencial sup F(l+1|l)
     (esas tablas —Bai & Perron, 1998, Tabla II— están calibradas por
     simulación para un conjunto discreto de niveles de significancia y
     números de regresores, y no se ofrecen como función cerrada evaluable
     en software estadístico estándar; por eso se opta aquí por el criterio
     de información, que sí es reproducible exactamente en Python).

Referencia: Bai, J. y Perron, P. (2003). "Computation and Analysis of
Multiple Structural Change Models". Journal of Applied Econometrics, 18(1).
"""

import numpy as np
import pandas as pd


def _ssr_table(y: np.ndarray):
    """Prefijos de y y de y^2 para computar SSR(i,j) de un ajuste de media
    constante en el segmento [i,j] (0-indexado, ambos extremos inclusive)
    en tiempo O(1)."""
    T = len(y)
    cs_y = np.concatenate([[0.0], np.cumsum(y)])
    cs_y2 = np.concatenate([[0.0], np.cumsum(y ** 2)])

    def ssr(i, j):
        n = j - i + 1
        s = cs_y[j + 1] - cs_y[i]
        s2 = cs_y2[j + 1] - cs_y2[i]
        return s2 - (s ** 2) / n

    return ssr


def _optimal_partition(y: np.ndarray, m: int, h: int):
    """Partición óptima exacta de y en m+1 segmentos (m quiebres), con
    tamaño mínimo de segmento h, minimizando la SSR total.

    Devuelve (ssr_total, lista_de_indices_de_quiebre) donde cada índice de
    quiebre bkp_k es el último índice (0-indexado) del segmento k-ésimo.
    """
    T = len(y)
    ssr = _ssr_table(y)

    if m == 0:
        return ssr(0, T - 1), []

    # dp[k][t] = SSR mínima usando k quiebres en el prefijo y[0..t]
    # (es decir, k+1 segmentos cubriendo [0, t]).
    # prev[k][t] = índice del quiebre anterior que logra ese mínimo.
    NEG = np.inf
    dp = np.full((m + 1, T), NEG)
    prev = np.full((m + 1, T), -1, dtype=int)

    for t in range(h - 1, T):
        dp[0, t] = ssr(0, t)

    for k in range(1, m + 1):
        min_t = (k + 1) * h - 1
        for t in range(min_t, T):
            best_val = NEG
            best_prev = -1
            # el quiebre anterior s marca el final del segmento k-1 y el
            # nuevo segmento es (s+1, t); requiere tamaño >= h
            for s in range(k * h - 1, t - h + 1):
                if dp[k - 1, s] == NEG:
                    continue
                val = dp[k - 1, s] + ssr(s + 1, t)
                if val < best_val:
                    best_val = val
                    best_prev = s
            dp[k, t] = best_val
            prev[k, t] = best_prev

    ssr_total = dp[m, T - 1]
    bkps = []
    k, t = m, T - 1
    while k > 0:
        s = prev[k, t]
        bkps.append(s)
        t = s
        k -= 1
    bkps.reverse()
    return ssr_total, bkps


def bai_perron_breaks(series: pd.Series, max_breaks: int = 5, trimming: float = 0.15):
    """Estima quiebres estructurales múltiples en la media de `series`
    (Bai-Perron, cambio puro en la media) para m=0..max_breaks quiebres,
    selecciona m* por BIC (Yao, 1988) y devuelve un diccionario con:

      - 'bic_por_m'      : DataFrame con SSR y BIC para cada m.
      - 'm_optimo'       : número de quiebres seleccionado.
      - 'fechas_quiebre' : fechas (índice de `series`) de los quiebres óptimos.
      - 'medias_segmento': media estimada de cada segmento bajo m*.
    """
    y = series.dropna()
    idx = y.index
    y = y.values
    T = len(y)
    h = max(2, int(np.ceil(trimming * T)))

    rows = []
    particiones = {}
    for m in range(0, max_breaks + 1):
        if (m + 1) * h > T:
            break
        ssr_total, bkps = _optimal_partition(y, m, h)
        n_params = (m + 1) + m  # m+1 medias + m fechas de quiebre estimadas
        bic = T * np.log(ssr_total / T) + n_params * np.log(T)
        rows.append({"m": m, "ssr": ssr_total, "n_parametros": n_params, "bic": bic})
        particiones[m] = bkps

    bic_df = pd.DataFrame(rows)
    m_optimo = int(bic_df.loc[bic_df["bic"].idxmin(), "m"])
    bkps_optimo = particiones[m_optimo]

    fechas_quiebre = [idx[b] for b in bkps_optimo]

    bounds = [-1] + bkps_optimo + [T - 1]
    medias_segmento = []
    for i in range(len(bounds) - 1):
        seg = y[bounds[i] + 1: bounds[i + 1] + 1]
        medias_segmento.append({
            "segmento": i + 1,
            "inicio": idx[bounds[i] + 1],
            "fin": idx[bounds[i + 1]],
            "n_obs": len(seg),
            "media": seg.mean(),
            "std": seg.std(ddof=1) if len(seg) > 1 else np.nan,
        })

    return {
        "bic_por_m": bic_df,
        "m_optimo": m_optimo,
        "fechas_quiebre": fechas_quiebre,
        "medias_segmento": pd.DataFrame(medias_segmento),
        "trimming_h": h,
    }


### Etapa 2.9b — Quiebres estructurales múltiples aplicados a la deuda

Aplica el algoritmo anterior a la ratio Deuda/PIB (SPNF y consolidada), con selección del número de quiebres por BIC.

Fuente: `codigo/modelos/fase9_bai_perron.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase9_bai_perron.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase9_bai_perron.py"

"""
fase9_bai_perron.py
====================
Mejora Dimensión I: contraste de quiebres estructurales múltiples de
Bai-Perron (1998, 2003) sobre la ratio Deuda/PIB, en reemplazo/complemento
del test de Zivot-Andrews de quiebre único (Fase 1), que había localizado un
candidato en 2018T1 sin poder rechazar la raíz unitaria (t=-2.83).

Implementación propia en `codigo/modelos/bai_perron.py` (programación dinámica
exacta + selección del número de quiebres por BIC), dado que no hay
interfaz de R/rpy2 disponible para invocar `strucchange::breakpoints()`.

Se aplica tanto a `deuda_pib` (serie original, SPNF) como a
`deuda_consolidada_pib` (serie de la Mejora Dimensión IV, SPNF + pasivos
remunerados del BCRA), para verificar si la consolidación cuasi-fiscal altera
la cronología de quiebres detectada.

Salidas:
  - resultados/tablas/fase9_bai_perron_bic.csv
  - resultados/tablas/fase9_bai_perron_quiebres.csv
  - tesis/figuras/figura_9_1_bai_perron.png
"""

import os
import pathlib
import pandas as pd
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, str(pathlib.Path(__file__).parent))
from bai_perron import bai_perron_breaks

BASE_DIR = pathlib.Path(__file__).parent.parent.parent
CSV_PATH = BASE_DIR / "datos" / "dataset_consolidado_real.csv"
LATEX_DIR = BASE_DIR / "tesis" / "figuras"
TABLES_DIR = BASE_DIR / "resultados" / "tablas"
os.makedirs(TABLES_DIR, exist_ok=True)


def analyze_series(df, col, label, max_breaks=5, trimming=0.15):
    print(f"\n[Bai-Perron] Serie: {label} ({col})")
    result = bai_perron_breaks(df[col], max_breaks=max_breaks, trimming=trimming)
    print(f" -> Recorte h = {result['trimming_h']} observaciones por segmento (trimming={trimming:.0%})")
    print(result["bic_por_m"].round(2).to_string(index=False))
    print(f" -> Número de quiebres seleccionado por BIC: m* = {result['m_optimo']}")
    for d in result["fechas_quiebre"]:
        print(f"    - Quiebre en {pd.Timestamp(d).strftime('%Y-%m')}")
    segmentos_fmt = result["medias_segmento"].copy()
    segmentos_fmt["inicio"] = segmentos_fmt["inicio"].dt.strftime("%Y-%m")
    segmentos_fmt["fin"] = segmentos_fmt["fin"].dt.strftime("%Y-%m")
    print(segmentos_fmt.round(2).to_string(index=False))
    return result


def main():
    print("=" * 75)
    print(" FASE 9: Quiebres Estructurales Múltiples de Bai-Perron")
    print("=" * 75)

    df = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")

    result_spnf = analyze_series(df, "deuda_pib", "Deuda SPNF / PIB (original)")
    result_cons = analyze_series(df, "deuda_consolidada_pib",
                                  "Deuda Consolidada SPNF + BCRA / PIB (Mejora Dim. IV)")

    bic_rows = []
    for label, res in [("deuda_pib", result_spnf), ("deuda_consolidada_pib", result_cons)]:
        d = res["bic_por_m"].copy()
        d["serie"] = label
        bic_rows.append(d)
    pd.concat(bic_rows, ignore_index=True).to_csv(TABLES_DIR / "fase9_bai_perron_bic.csv", index=False)

    break_rows = []
    for label, res in [("deuda_pib", result_spnf), ("deuda_consolidada_pib", result_cons)]:
        for d in res["fechas_quiebre"]:
            break_rows.append({"serie": label, "fecha_quiebre": pd.Timestamp(d).strftime("%Y-%m-%d")})
    pd.DataFrame(break_rows).to_csv(TABLES_DIR / "fase9_bai_perron_quiebres.csv", index=False)

    result_spnf["medias_segmento"].assign(serie="deuda_pib").to_csv(
        TABLES_DIR / "fase9_bai_perron_segmentos_deuda_pib.csv", index=False)
    result_cons["medias_segmento"].assign(serie="deuda_consolidada_pib").to_csv(
        TABLES_DIR / "fase9_bai_perron_segmentos_deuda_consolidada.csv", index=False)

    # --- Gráfico ---
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(df.index, df["deuda_pib"], color="#0B3C5D", linewidth=1.6, label="Deuda SPNF / PIB")
    ax.plot(df.index, df["deuda_consolidada_pib"], color="#B33951", linewidth=1.6,
            linestyle="--", label="Deuda Consolidada SPNF + BCRA / PIB")

    for seg in result_spnf["medias_segmento"].itertuples():
        ax.hlines(seg.media, seg.inicio, seg.fin, color="#0B3C5D", linewidth=3, alpha=0.35)
    for d in result_spnf["fechas_quiebre"]:
        ax.axvline(d, color="#0B3C5D", linestyle=":", alpha=0.6)
        ax.text(d, ax.get_ylim()[1] * 0.97, pd.Timestamp(d).strftime("%Y-%m"),
                rotation=90, fontsize=8, color="#0B3C5D", va="top", ha="right")

    for d in result_cons["fechas_quiebre"]:
        ax.axvline(d, color="#B33951", linestyle=":", alpha=0.4)

    ax.set_title("Quiebres Estructurales Múltiples de Bai-Perron (selección por BIC)")
    ax.set_xlabel("Trimestre")
    ax.set_ylabel("% del PIB")
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    out_path = LATEX_DIR / "figura_9_1_bai_perron.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    print(f"\n[OK] Gráfico: {out_path}")
    print("\n[OK] Fase 9 completa.")


if __name__ == "__main__":
    main()


### Etapa 2.10a — DCC-GARCH (implementación)

Implementación propia del modelo de Correlación Condicional Dinámica de Engle (2002), estimado por Cuasi-Máxima Verosimilitud en dos etapas.

Fuente: `codigo/modelos/dcc_garch.py`

In [ ]:
"""
dcc_garch.py
============
Implementación propia, en Python puro (NumPy/SciPy), del modelo de
Correlación Condicional Dinámica de Engle (2002) -DCC-GARCH-, estimado por
Cuasi-Máxima Verosimilitud (QML) en dos etapas:

  Etapa 1 (univariada): se ajusta un GARCH(1,1) a cada serie individual
  (paquete `arch`, ya utilizado en el proyecto) y se extraen los residuos
  estandarizados z_t = u_t / sigma_t.

  Etapa 2 (correlación dinámica): sobre la matriz de residuos estandarizados
  Z (T x N), se estima por QML el proceso de correlación dinámica:
      Q_t = (1 - a - b) * Qbar + a * z_{t-1} z_{t-1}' + b * Q_{t-1}
      R_t = diag(Q_t)^{-1/2} Q_t diag(Q_t)^{-1/2}
  con Qbar la matriz de correlación incondicional de Z, y (a, b) los
  parámetros de persistencia/reactividad de la correlación, sujetos a
  a >= 0, b >= 0, a + b < 1.

No existe en el ecosistema de Python un equivalente directo y mantenido de
`rmgarch::dccfit()` de R (que a su vez requiere R/rpy2, no disponibles en
este entorno). Esta es una implementación directa de Engle (2002), no una
aproximación: la log-verosimilitud concentrada de la etapa de correlación
es exactamente la de la especificación DCC(1,1) estándar.

Referencia: Engle, R. (2002). "Dynamic Conditional Correlation: A Simple
Class of Multivariate Generalized Autoregressive Conditional Heteroskedasticity
Models". Journal of Business & Economic Statistics, 20(3), 339-350.
"""

import numpy as np
from scipy.optimize import minimize
from arch import arch_model


def fit_univariate_garch(series: np.ndarray):
    """Ajusta un GARCH(1,1) univariado (media cero, t de Student) a una
    serie en puntos porcentuales (ya escalada por 100 para estabilidad
    numérica, convención estándar del paquete `arch`)."""
    am = arch_model(series, mean="Zero", vol="GARCH", p=1, q=1, dist="t")
    res = am.fit(disp="off")
    z = res.resid / res.conditional_volatility
    return res, z


def _dcc_neg_loglik(params, Z):
    a, b = params
    T, N = Z.shape
    Qbar = np.corrcoef(Z.T)
    Q_t = Qbar.copy()
    nll = 0.0
    for t in range(T):
        d = np.sqrt(np.diag(Q_t))
        R_t = Q_t / np.outer(d, d)
        try:
            R_inv = np.linalg.inv(R_t)
            sign, logdet = np.linalg.slogdet(R_t)
        except np.linalg.LinAlgError:
            return 1e10
        if sign <= 0:
            return 1e10
        z_t = Z[t, :]
        nll += 0.5 * (logdet + z_t @ R_inv @ z_t.T - z_t @ z_t.T)
        Q_t = (1 - a - b) * Qbar + a * np.outer(z_t, z_t) + b * Q_t
    return nll


def fit_dcc(Z: np.ndarray, a0: float = 0.03, b0: float = 0.90):
    """Estima (a, b) del DCC(1,1) por QML sobre la matriz de residuos
    estandarizados Z (T x N)."""
    cons = ({"type": "ineq", "fun": lambda p: 0.999 - p[0] - p[1]},
             {"type": "ineq", "fun": lambda p: p[0]},
             {"type": "ineq", "fun": lambda p: p[1]})
    res = minimize(_dcc_neg_loglik, x0=[a0, b0], args=(Z,), method="SLSQP",
                    bounds=[(1e-6, 0.5), (1e-6, 0.998)], constraints=cons,
                    options={"maxiter": 300, "ftol": 1e-10})
    a, b = res.x
    return a, b, res


def dcc_correlation_path(Z: np.ndarray, a: float, b: float):
    """Reconstruye la trayectoria completa R_1,...,R_T dado (a, b)."""
    T, N = Z.shape
    Qbar = np.corrcoef(Z.T)
    Q_t = Qbar.copy()
    R_path = np.zeros((T, N, N))
    for t in range(T):
        d = np.sqrt(np.diag(Q_t))
        R_t = Q_t / np.outer(d, d)
        R_path[t] = R_t
        z_t = Z[t, :]
        Q_t = (1 - a - b) * Qbar + a * np.outer(z_t, z_t) + b * Q_t
    return R_path


### Etapa 2.10b — Correlación condicional dinámica aplicada al DSA

Reestima por QML la correlación entre los *shocks* del DSA, en reemplazo de la calibración estática.

Fuente: `codigo/modelos/fase10_dcc_garch.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase10_dcc_garch.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase10_dcc_garch.py"

"""
fase10_dcc_garch.py
=====================
Mejora Dimensión II: volatilidad condicional multivariada mediante DCC-GARCH
(Engle, 2002), en reemplazo de la matriz de correlación estática calibrada
que hasta ahora alimentaba el Análisis de Sostenibilidad de la Deuda (DSA)
estocástico (`fase6_sostenibilidad_deuda.py`).

`fase7_diagnosticos_robustez.py` (sección 9) ya había reemplazado, para las
tres variables con serie histórica real (pb, g, delta_e), los desvíos
estándar calibrados a ojo por desvíos GARCH(1,1) univariados -pero mantuvo la
matriz de correlación estática CORR sin modificar-. Este script completa esa
mejora: estima la correlación condicional dinámica (DCC) entre esas mismas
tres series, documentando el agrupamiento de volatilidad y de correlación
que la calibración estática por diseño no puede capturar, y usa la
correlación DCC promedio (y la del último período, régimen "actual") para
reconstruir la matriz de covarianza del DSA.

r_d y r_f (tasas de interés doméstica y externa) no tienen serie propia en
el dataset consolidado y permanecen calibradas, exactamente como en Fase 7,
sección 9 -limitación declarada, no oculta-.

Salidas:
  - resultados/tablas/fase10_dcc_garch_parametros.csv
  - resultados/tablas/fase10_dcc_trayectoria_correlacion.csv
  - resultados/tablas/fase10_dcc_garch_comparacion_dsa.csv
  - tesis/figuras/figura_10_1_dcc_correlacion_dinamica.png
"""

import os
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

sys.path.insert(0, str(pathlib.Path(__file__).parent))
from dcc_garch import fit_univariate_garch, fit_dcc, dcc_correlation_path
from fase6_sostenibilidad_deuda import (
    simulate_stochastic_dsa, SCENARIOS, ALPHA, D_INITIAL, DSA_STUDENT_T_NU,
    STD_DEVS, CORR,
)

BASE_DIR = pathlib.Path(__file__).parent.parent.parent
CSV_PATH = BASE_DIR / "datos" / "dataset_consolidado_real.csv"
LATEX_DIR = BASE_DIR / "tesis" / "figuras"
TABLES_DIR = BASE_DIR / "resultados" / "tablas"
os.makedirs(TABLES_DIR, exist_ok=True)

VAR_LABELS = ["pb", "g", "delta_e"]
VAR_IDX_IN_STD_DEVS = [0, 1, 4]  # posiciones de pb, g, delta_e en STD_DEVS/CORR (orden: pb,g,r_d,r_f,delta_e)


def build_shocks(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    pb_shock = d["pb_pib"].diff()
    delta_e_shock = d["TCRM"].pct_change() * 100
    g_shock = d["PIB_real"].pct_change(4) * 100
    shocks = pd.DataFrame({"pb": pb_shock, "g": g_shock, "delta_e": delta_e_shock}).dropna()
    return shocks


def main():
    print("=" * 75)
    print(" FASE 10: Volatilidad Condicional Multivariada (DCC-GARCH, Engle 2002)")
    print("=" * 75)

    df = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")
    shocks = build_shocks(df)
    print(f"\n[1/5] Shocks construidos: {len(shocks)} observaciones válidas "
          f"({shocks.index.min().date()} -> {shocks.index.max().date()}).")

    print("\n[2/5] Etapa 1 (QML univariada): GARCH(1,1)-t por serie...")
    garch_results = {}
    Z = np.zeros((len(shocks), 3))
    cond_vols = {}
    for i, var in enumerate(VAR_LABELS):
        res, z = fit_univariate_garch(shocks[var].values)
        garch_results[var] = res
        Z[:, i] = z
        cond_vols[var] = res.conditional_volatility
        print(f"  -> {var}: alpha[1]={res.params.get('alpha[1]', np.nan):.4f}, "
              f"beta[1]={res.params.get('beta[1]', np.nan):.4f}, "
              f"nu={res.params.get('nu', np.nan):.2f}")

    print("\n[3/5] Etapa 2 (QML de correlación): estimando DCC(1,1)...")
    a, b, opt_res = fit_dcc(Z)
    print(f"  -> a (reactividad) = {a:.4f}")
    print(f"  -> b (persistencia) = {b:.4f}")
    print(f"  -> Convergencia del optimizador: {opt_res.success} ({opt_res.message})")
    if a < 1e-4:
        print("  -> a=0 es el óptimo global de QML (verificado con 5 puntos de partida "
              "distintos, misma log-verosimilitud): el DCC(1,1) colapsa a Correlación "
              "Condicional Constante (CCC, Bollerslev 1990). Con 84 observaciones "
              "trimestrales, los datos no sostienen variación temporal adicional de la "
              "correlación por encima de su nivel incondicional; b queda no identificado "
              "en ese punto (Q_t = Qbar para todo t, independientemente de b). El aporte "
              "real de esta etapa es, entonces, la reestimación por QML del nivel de "
              "correlación -no de su dinámica-, que resulta sistemáticamente más débil en "
              "magnitud que la calibrada a ojo (ver comparación debajo).")

    R_path = dcc_correlation_path(Z, a, b)
    pairs = [(0, 1, "pb-g"), (0, 2, "pb-delta_e"), (1, 2, "g-delta_e")]
    corr_df = pd.DataFrame({"Date": shocks.index})
    for i, j, name in pairs:
        corr_df[f"rho_{name}"] = R_path[:, i, j]
    corr_df.to_csv(TABLES_DIR / "fase10_dcc_trayectoria_correlacion.csv", index=False)

    print("\n     Correlación dinámica DCC vs. correlación estática calibrada:")
    static_corr = {
        "pb-g": CORR[0, 1], "pb-delta_e": CORR[0, 4], "g-delta_e": CORR[1, 4],
    }
    for i, j, name in pairs:
        dyn_mean = R_path[:, i, j].mean()
        dyn_last = R_path[-1, i, j]
        dyn_min, dyn_max = R_path[:, i, j].min(), R_path[:, i, j].max()
        print(f"     {name:12s}: calibrada={static_corr[name]:+.2f} | "
              f"DCC promedio={dyn_mean:+.2f} | DCC último período={dyn_last:+.2f} | "
              f"rango=[{dyn_min:+.2f}, {dyn_max:+.2f}]")

    pd.DataFrame([{
        "a_reactividad": a, "b_persistencia": b, "a_mas_b": a + b,
        "convergencia": opt_res.success,
    }]).to_csv(TABLES_DIR / "fase10_dcc_garch_parametros.csv", index=False)

    print("\n[4/5] Reconstruyendo matriz de covarianza del DSA con correlación DCC...")
    # Desvíos: se reutilizan los GARCH(1,1) univariados de esta misma etapa 1
    # (anualizados con el mismo criterio que fase7, sección 9: sqrt(4) para
    # shocks trimestrales sin comparar antes/después estacional; g_shock ya
    # es interanual y no se reanualiza).
    std_pb = cond_vols["pb"].mean() / 100 * np.sqrt(4)
    std_g = cond_vols["g"].mean() / 100
    std_delta_e = cond_vols["delta_e"].mean() / 100 * np.sqrt(4)

    std_devs_dcc = STD_DEVS.copy()
    std_devs_dcc[0] = std_pb
    std_devs_dcc[1] = std_g
    std_devs_dcc[4] = std_delta_e

    # Correlación: se reemplaza el bloque (pb, g, delta_e) por el promedio
    # temporal de la correlación condicional dinámica; r_d y r_f (índices 2,3)
    # mantienen la correlación calibrada frente a todo el resto, por no
    # integrar el sistema DCC (sin serie propia, igual que en Fase 7 sec. 9).
    corr_dcc = CORR.copy()
    R_mean = R_path.mean(axis=0)
    idx_map = {0: 0, 1: 1, 2: 4}  # posición en R_mean -> posición en CORR/STD_DEVS
    for i_r, i_c in idx_map.items():
        for j_r, j_c in idx_map.items():
            corr_dcc[i_c, j_c] = R_mean[i_r, j_r]

    cov_dcc = np.outer(std_devs_dcc, std_devs_dcc) * corr_dcc
    cov_calibrado = np.outer(STD_DEVS, STD_DEVS) * CORR

    print("\n[5/5] Comparando probabilidades de insolvencia (Monte Carlo, calibrado vs. DCC-GARCH)...")
    years = np.arange(2026, 2036)
    rows = []
    for name, params in SCENARIOS.items():
        d_calib = simulate_stochastic_dsa(ALPHA, D_INITIAL, params, cov_calibrado, years, DSA_STUDENT_T_NU, n_simulations=1000)
        d_dcc = simulate_stochastic_dsa(ALPHA, D_INITIAL, params, cov_dcc, years, DSA_STUDENT_T_NU, n_simulations=1000)
        p_calib = np.mean(d_calib[:, -1] > 1.0) * 100
        p_dcc = np.mean(d_dcc[:, -1] > 1.0) * 100
        print(f"  -> {name}: P(deuda/PIB > 100%, 2035) calibrado={p_calib:.1f}% vs. DCC-GARCH={p_dcc:.1f}%")
        rows.append({"escenario": name, "prob_calibrado": p_calib, "prob_dcc_garch": p_dcc})

    pd.DataFrame(rows).to_csv(TABLES_DIR / "fase10_dcc_garch_comparacion_dsa.csv", index=False)

    # --- Gráfico: correlación dinámica en el tiempo ---
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, j, name in pairs:
        ax.plot(shocks.index, R_path[:, i, j], label=f"$\\rho$({name})", linewidth=1.4)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(f"Correlación Condicional Dinámica DCC(1,1) — a={a:.3f}, b={b:.3f}")
    ax.set_xlabel("Trimestre")
    ax.set_ylabel("Correlación condicional")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    out_path = LATEX_DIR / "figura_10_1_dcc_correlacion_dinamica.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    print(f"\n[OK] Gráfico: {out_path}")
    print("\n[OK] Fase 10 completa.")


if __name__ == "__main__":
    main()


### Etapa 2.11 — DOLS por subperíodo

Reestima la Función de Reacción Fiscal en dos submuestras (2004-2014 y 2015-2025) para evaluar estabilidad paramétrica.

Fuente: `codigo/modelos/fase11_dols_subperiodos.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase11_dols_subperiodos.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase11_dols_subperiodos.py"

"""
fase11_dols_subperiodos.py
==========================
Reestimacion de la Funcion de Reaccion Fiscal por subperiodos mediante DOLS.

Motivacion
----------
La Seccion de sensibilidad temporal del Capitulo de Resultados reportaba
originalmente solo una especificacion estatica MCO-HAC por subperiodo
(2004T1-2014T4 y 2015T1-2025T4), que arrojaba coeficientes rho positivos y
significativos, en contraste con el DOLS de muestra completa (rho<0, no
significativo). Descalificar el MCO estatico por sesgo de simultaneidad sin
mostrar la alternativa corregida deja abierta la objecion de que el descarte
es retorico antes que empirico.

Este modulo cierra esa brecha: reestima cada subperiodo con la augmentacion
dinamica de Stock-Watson, reduciendo el orden de adelantos/rezagos (m=1 y m=2
en lugar de m=4) para que el ejercicio sea factible con n~43 observaciones.
El resultado permite decidir si el signo positivo del MCO estatico sobrevive
a la correccion de endogeneidad de corto plazo.
"""

import os
import pathlib
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm

warnings.filterwarnings("ignore")

CORTE = "2015-01-01"
SUBPERIODOS = [
    ("2004T1-2014T4", None, CORTE),
    ("2015T1-2025T4", CORTE, None),
]


def newey_west_lags(n):
    """Lags de Newey-West segun m = 4*(T/100)^(2/9)."""
    return int(np.ceil(4 * (n / 100) ** (2 / 9)))


def estimar_estatico(sub):
    """MCO estatico pb_t = a + rho*d_{t-1} + gamma*y_gap + e_t, errores HAC."""
    sub = sub.dropna(subset=["pb_pib", "d_t_1", "g_gap"])
    y = sub["pb_pib"]
    X = sm.add_constant(sub[["d_t_1", "g_gap"]])
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": newey_west_lags(len(sub))})
    return {
        "n": int(len(sub)),
        "rho": float(res.params["d_t_1"]),
        "se": float(res.bse["d_t_1"]),
        "p": float(res.pvalues["d_t_1"]),
    }


def estimar_dols(sub, m):
    """DOLS con m adelantos y m rezagos de Delta d, errores HAC."""
    sub = sub.copy()
    sub["diff_d"] = sub["d_t_1"].diff()
    feats = ["d_t_1", "g_gap", "diff_d"]
    for i in range(1, m + 1):
        sub[f"diff_d_lag_{i}"] = sub["diff_d"].shift(i)
        sub[f"diff_d_lead_{i}"] = sub["diff_d"].shift(-i)
        feats += [f"diff_d_lag_{i}", f"diff_d_lead_{i}"]

    sub = sub.dropna(subset=["pb_pib"] + feats)
    n = len(sub)
    k = len(feats) + 1  # + constante
    if n <= k + 2:
        return None  # grados de libertad insuficientes

    y = sub["pb_pib"]
    X = sm.add_constant(sub[feats])
    res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": newey_west_lags(n)})
    return {
        "n": int(n),
        "gl": int(n - k),
        "rho": float(res.params["d_t_1"]),
        "se": float(res.bse["d_t_1"]),
        "p": float(res.pvalues["d_t_1"]),
    }


def run(csv_path):
    print("=" * 78)
    print(" FASE 11: DOLS por subperiodo (robustez de la sensibilidad temporal) ")
    print("=" * 78)

    df = pd.read_csv(csv_path, parse_dates=["Date"], index_col="Date")
    df["d_t_1"] = df["deuda_pib"].shift(1)
    df = df.dropna(subset=["pb_pib", "d_t_1", "g_gap"])

    filas = []
    for etiqueta, ini, fin in SUBPERIODOS:
        sub = df.loc[ini:fin] if ini else df.loc[:fin]
        if fin is None:
            sub = df.loc[ini:]
        # .loc con fin abierto incluye el corte; se excluye explicitamente
        if fin is not None:
            sub = sub[sub.index < pd.Timestamp(fin)]

        print(f"\n--- {etiqueta} ---")
        est = estimar_estatico(sub)
        print(f"  MCO estatico   : n={est['n']:>3}  rho={est['rho']:+.4f}  "
              f"EE={est['se']:.4f}  p={est['p']:.4f}")
        filas.append({"subperiodo": etiqueta, "especificacion": "MCO estatico (HAC)", **est, "gl": np.nan})

        for m in (1, 2):
            d = estimar_dols(sub, m)
            if d is None:
                print(f"  DOLS (m={m})     : grados de libertad insuficientes")
                continue
            print(f"  DOLS (m={m})     : n={d['n']:>3}  gl={d['gl']:>3}  rho={d['rho']:+.4f}  "
                  f"EE={d['se']:.4f}  p={d['p']:.4f}")
            filas.append({"subperiodo": etiqueta, "especificacion": f"DOLS (m={m}, HAC)", **d})

    tabla = pd.DataFrame(filas)[
        ["subperiodo", "especificacion", "n", "gl", "rho", "se", "p"]
    ]

    print("\n" + "=" * 78)
    print(" SINTESIS ")
    print("=" * 78)
    print(tabla.to_string(index=False,
                          float_format=lambda x: f"{x:.4f}" if pd.notna(x) else "--"))

    signos_dols = tabla[tabla["especificacion"].str.startswith("DOLS")]
    if len(signos_dols):
        n_sig = int((signos_dols["p"] < 0.05).sum())
        print(f"\n -> Especificaciones DOLS por subperiodo estimadas: {len(signos_dols)}")
        print(f" -> De ellas, con rho significativo al 5%: {n_sig}")
        if n_sig == 0:
            print(" -> Veredicto: la significatividad del MCO estatico NO sobrevive a la")
            print("    augmentacion dinamica. El signo positivo por subperiodo es atribuible")
            print("    al sesgo de simultaneidad de corto plazo, no a una reaccion fiscal.")
        else:
            print(" -> Veredicto: al menos un subperiodo conserva reaccion significativa bajo")
            print("    DOLS; el hallazgo debe reportarse como tal en el Capitulo de Resultados.")

    os.makedirs("resultados/tablas", exist_ok=True)
    salida = "resultados/tablas/fase11_dols_subperiodos.csv"
    tabla.to_csv(salida, index=False)
    print(f"\n[OK] Resultados guardados en '{salida}'")
    return tabla


if __name__ == "__main__":
    base_dir = pathlib.Path(__file__).parent.parent.parent
    run(base_dir / "datos" / "dataset_consolidado_real.csv")


### Etapa 2.12 — Diagnósticos complementarios del VAR

Selección del orden de rezagos del VAR (AIC/BIC/HQ/FPE), sensibilidad del rango de Johansen, VIF y número de condición.

Fuente: `codigo/modelos/fase12_diagnosticos_complementarios.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase12_diagnosticos_complementarios.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase12_diagnosticos_complementarios.py"

"""
fase12_diagnosticos_complementarios.py
======================================
Diagnosticos complementarios exigidos por la revision academica:

  (a) Seleccion del orden de rezagos del VAR subyacente al procedimiento de
      Johansen mediante criterios de informacion (AIC, BIC/SC y HQ), y
      sensibilidad del rango de cointegracion estimado al orden elegido.

  (b) Multicolinealidad de la ecuacion de reaccion fiscal: factores de
      inflacion de la varianza (VIF) y numero de condicion de la matriz de
      regresores.

Ambos bloques operan sobre el mismo panel trimestral que el resto del
protocolo, de modo que sus resultados son directamente comparables con los
del Capitulo de Resultados.
"""

import os
import pathlib
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen

warnings.filterwarnings("ignore")

# Sistema de cuatro variables sometido al contraste de Johansen.
SISTEMA = ["deuda_pib", "pb_pib", "EMBI", "TCRM"]

# Valores criticos de Johansen al 5% (columna 1 de la salida de statsmodels).
NIVEL_CRITICO = 1  # 0 -> 10%, 1 -> 5%, 2 -> 1%


def seleccionar_rezagos(df, maxlags=8):
    """Orden de rezagos del VAR en niveles segun AIC, BIC(SC), HQ y FPE."""
    modelo = VAR(df[SISTEMA])
    sel = modelo.select_order(maxlags=maxlags)

    print("\n[a] Seleccion del orden de rezagos del VAR (niveles)")
    print("    " + "-" * 62)
    print(sel.summary())

    elegidos = {k: int(v) for k, v in sel.selected_orders.items()}
    print(f"\n    Ordenes seleccionados: {elegidos}")
    return elegidos


def sensibilidad_johansen(df, ordenes):
    """Rango de cointegracion estimado para cada orden de rezagos plausible."""
    print("\n[b] Sensibilidad del rango de cointegracion al orden de rezagos")
    print("    (det_order=0: constante en la relacion de cointegracion)")
    print("    " + "-" * 62)

    filas = []
    for k in ordenes:
        # coint_johansen recibe k_ar_diff = rezagos del VAR en diferencias = p - 1
        k_diff = max(k - 1, 0)
        res = coint_johansen(df[SISTEMA].values, det_order=0, k_ar_diff=k_diff)

        rango_traza = int(np.sum(res.lr1 > res.cvt[:, NIVEL_CRITICO]))
        rango_maxeig = int(np.sum(res.lr2 > res.cvm[:, NIVEL_CRITICO]))

        filas.append({
            "p_var": k,
            "k_ar_diff": k_diff,
            "rango_traza_5pct": rango_traza,
            "rango_maxeig_5pct": rango_maxeig,
        })
        print(f"    p={k} (k_ar_diff={k_diff}) -> rango Traza={rango_traza}, "
              f"rango Max-Autovalor={rango_maxeig}")

    tabla = pd.DataFrame(filas)
    coincide = tabla["rango_traza_5pct"].nunique() == 1 and tabla["rango_maxeig_5pct"].nunique() == 1
    print(f"\n    Diagnostico: el rango estimado {'NO varia' if coincide else 'VARIA'} "
          f"con el orden de rezagos en el rango explorado.")
    return tabla


def diagnostico_multicolinealidad(df):
    """VIF y numero de condicion de la ecuacion de reaccion fiscal."""
    print("\n[c] Multicolinealidad de la ecuacion de reaccion fiscal")
    print("    " + "-" * 62)

    especificaciones = {
        "DOLS (nucleo de Bohn)": ["d_t_1", "g_gap"],
        "Ampliada (Bohn + EMBI + TCRM)": ["d_t_1", "g_gap", "EMBI", "TCRM"],
    }

    filas = []
    for nombre, cols in especificaciones.items():
        sub = df.dropna(subset=cols)
        X = sm.add_constant(sub[cols])
        Xv = X.values

        # Numero de condicion sobre regresores estandarizados (excluye constante).
        Z = sub[cols].values
        Z = (Z - Z.mean(axis=0)) / Z.std(axis=0, ddof=1)
        cond = float(np.linalg.cond(Z))

        print(f"\n    {nombre}  (n={len(sub)})")
        for i, c in enumerate(X.columns):
            if c == "const":
                continue
            vif = float(variance_inflation_factor(Xv, i))
            print(f"      VIF {c:<10} = {vif:7.3f}")
            filas.append({"especificacion": nombre, "regresor": c,
                          "VIF": vif, "num_condicion": cond})
        print(f"      Numero de condicion  = {cond:7.3f}")

    tabla = pd.DataFrame(filas)
    max_vif = tabla["VIF"].max()
    print(f"\n    VIF maximo del conjunto: {max_vif:.3f} "
          f"({'por debajo' if max_vif < 10 else 'por encima'} del umbral convencional de 10).")
    return tabla


def run(csv_path):
    print("=" * 78)
    print(" FASE 12: Diagnosticos complementarios (rezagos VAR, VIF, condicion) ")
    print("=" * 78)

    df = pd.read_csv(csv_path, parse_dates=["Date"], index_col="Date")
    df["d_t_1"] = df["deuda_pib"].shift(1)

    sistema = df[SISTEMA].dropna()
    print(f"\nPanel del sistema de Johansen: n={len(sistema)} "
          f"({sistema.index.min():%Y-%m} a {sistema.index.max():%Y-%m})")

    elegidos = seleccionar_rezagos(sistema)

    ordenes = sorted({max(v, 1) for v in elegidos.values()} | {1, 2, 3, 4})
    tabla_joh = sensibilidad_johansen(sistema, ordenes)

    tabla_vif = diagnostico_multicolinealidad(df)

    os.makedirs("resultados/tablas", exist_ok=True)
    tabla_joh.to_csv("resultados/tablas/fase12_johansen_sensibilidad_rezagos.csv", index=False)
    tabla_vif.to_csv("resultados/tablas/fase12_vif.csv", index=False)
    pd.Series(elegidos).to_csv("resultados/tablas/fase12_orden_rezagos.csv",
                               header=["orden"])
    print("\n[OK] Resultados guardados en 'resultados/tablas/fase12_*.csv'")
    return elegidos, tabla_joh, tabla_vif


if __name__ == "__main__":
    base_dir = pathlib.Path(__file__).parent.parent.parent
    run(base_dir / "datos" / "dataset_consolidado_real.csv")


### Etapa 2.13 — Robustez del umbral de Hansen con Δpb_t

Reestima el modelo de umbral con la variable dependiente en primera diferencia (I(0)), que sí satisface el supuesto de estacionariedad del test.

Fuente: `codigo/modelos/fase13_robustez_hansen_dpb.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase13_robustez_hansen_dpb.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase13_robustez_hansen_dpb.py"

"""
H1 (optima) -- Re-correr el test de umbral de Hansen (2000) usando Delta pb_t
(primera diferencia del resultado primario, I(0) segun DF-GLS) como variable
dependiente, en lugar de pb_t en niveles (I(1)). Replica exactamente la logica
de codigo/modelos/fase5_umbral_hansen.py, cambiando solo y_col.
"""
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

import sys
import pathlib
sys.path.insert(0, str(pathlib.Path(__file__).parent))
from fase5_umbral_hansen import get_optimal_threshold, bootstrap_threshold_test

_base_dir = pathlib.Path(__file__).parent.parent.parent
csv_path = _base_dir / "datos" / "dataset_consolidado_real.csv"
df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')

df['d_t_1'] = df['deuda_pib'].shift(1)
df['dpb_pib'] = df['pb_pib'].diff()

subset_cols = ['dpb_pib', 'd_t_1', 'g_gap', 'EMBI']
df = df.dropna(subset=subset_cols).copy()
print(f"Observaciones validas (tras diferenciar pb_t): {len(df)}")

best_tau, best_model = get_optimal_threshold(
    df=df, y_col='dpb_pib', X_control_cols=['g_gap'],
    split_col='d_t_1', threshold_col='EMBI'
)
print(f"Umbral optimo (tau*): {best_tau:.1f} pb de EMBI+")

f_obs, p_val_boot = bootstrap_threshold_test(
    df=df, y_col='dpb_pib', X_control_cols=['g_gap'],
    split_col='d_t_1', threshold_col='EMBI', best_tau=best_tau, n_boot=1000
)
print(f"F-stat observado: {f_obs:.4f}")
print(f"p-value bootstrap (1000 iter): {p_val_boot:.4f}")
print()
print(best_model.summary().tables[1])

# Persistencia a CSV para trazabilidad (mismo criterio que el resto de las
# fases: todo resultado intermedio citado en la tesis debe quedar en
# resultados/tablas/, no solo impreso por consola).
tabla_coef = pd.DataFrame({
    "variable": best_model.params.index,
    "coeficiente": best_model.params.values,
    "error_estandar": best_model.bse.values,
    "estadistico_z": best_model.tvalues.values,
    "p_valor": best_model.pvalues.values,
})
tabla_coef.to_csv("resultados/tablas/fase13_hansen_dpb_coeficientes.csv", index=False)

tabla_resumen = pd.DataFrame([{
    "tau_optimo_pb": best_tau,
    "f_stat_observado": f_obs,
    "p_valor_bootstrap": p_val_boot,
    "n_bootstrap": 1000,
    "n_obs": len(df),
}])
tabla_resumen.to_csv("resultados/tablas/fase13_hansen_dpb_resumen.csv", index=False)
print("\nGuardado: resultados/tablas/fase13_hansen_dpb_coeficientes.csv, fase13_hansen_dpb_resumen.csv")


### Etapa 2.14 — Robustez del DSA a la dependencia en colas

Reestima la probabilidad de insolvencia bajo marginales $t$ independientes y bajo una cópula Gaussiana, para aislar el efecto de la *tail dependence* de la cópula-$t$.

Fuente: `codigo/modelos/fase14_dsa_tail_dependence.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase14_dsa_tail_dependence.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase14_dsa_tail_dependence.py"

"""
H6 (optima) -- Robustez del DSA estocastico bajo shocks t marginales
INDEPENDIENTES (sin estructura de copula-t, por tanto sin tail dependence),
preservando exactamente la misma varianza marginal de cada shock que la
version correlacionada de fase6_sostenibilidad_deuda.py. Compara contra
30.4% (referencia, t-copula correlacionada).

Reporta ademas una variante intermedia (copula Gaussiana + marginales t)
que preserva la correlacion lineal pero remueve el exceso de tail
dependence propio de la t-copula, para distinguir el efecto de la
correlacion del efecto de la tail dependence.
"""
import numpy as np
from scipy.stats import t as t_dist, norm
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path(__file__).parent))
from fase6_sostenibilidad_deuda import (
    SCENARIOS, STD_DEVS, CORR, ALPHA, D_INITIAL, DSA_STUDENT_T_NU,
    build_cov_matrix, simulate_stochastic_dsa
)

years = np.arange(2026, 2036)
n_years = len(years)
nu = DSA_STUDENT_T_NU
cov_matrix = build_cov_matrix()
base_params = SCENARIOS['Referencia']
alpha = ALPHA

# --- 0. Referencia: t-copula correlacionada (replica exacta de fase6) ---
d_paths_ref = simulate_stochastic_dsa(alpha, D_INITIAL, base_params, cov_matrix, years, nu, n_simulations=1000)
p_ref = np.mean(d_paths_ref[:, -1] > 1.0)
print(f"[0] Referencia (t-copula correlacionada, tal como reportado): {p_ref*100:.1f}%")

def simulate_path_from_shocks(shocks_5xN, alpha, d_initial, base_params, years):
    n_years = len(years)
    d = np.zeros(n_years)
    d[0] = d_initial
    pb_s = np.full(n_years, base_params['pb']) + shocks_5xN[:, 0]
    g_s = np.full(n_years, base_params['g']) + shocks_5xN[:, 1]
    rd_s = np.full(n_years, base_params['r_d']) + shocks_5xN[:, 2]
    rf_s = np.full(n_years, base_params['r_f']) + shocks_5xN[:, 3]
    delta_e_s = np.full(n_years, base_params['delta_e']) + shocks_5xN[:, 4]
    for i in range(1, n_years):
        term_d = alpha * (1 + rd_s[i]) / (1 + g_s[i])
        term_f = (1 - alpha) * (1 + rf_s[i]) * (1 + delta_e_s[i]) / (1 + g_s[i])
        M = term_d + term_f
        d[i] = M * d[i-1] - pb_s[i] + 0.0
    return d

# --- 1. Marginales t INDEPENDIENTES (sin correlacion, sin tail dependence) ---
np.random.seed(42)
n_sim = 1000
scale_indep = np.sqrt(np.diag(cov_matrix) * (nu - 2) / nu)  # preserva var. marginal exacta
d_paths_indep = np.zeros((n_sim, n_years))
d_paths_indep[:, 0] = D_INITIAL
for s in range(n_sim):
    shocks = np.zeros((n_years, 5))
    for j in range(5):
        shocks[:, j] = t_dist.rvs(df=nu, size=n_years) * scale_indep[j]
    d_paths_indep[s, :] = simulate_path_from_shocks(shocks, alpha, D_INITIAL, base_params, years)
p_indep = np.mean(d_paths_indep[:, -1] > 1.0)
print(f"[1] Marginales t independientes (sin correlacion, sin tail dependence): {p_indep*100:.1f}%")

# --- 2. Copula Gaussiana + marginales t (preserva correlacion, sin exceso de tail dependence) ---
np.random.seed(42)
L = np.linalg.cholesky(CORR)
d_paths_gausscop = np.zeros((n_sim, n_years))
d_paths_gausscop[:, 0] = D_INITIAL
for s in range(n_sim):
    Z = np.random.normal(size=(n_years, 5)) @ L.T   # correlacion gaussiana
    U = norm.cdf(Z)                                   # a uniformes via copula gaussiana
    U = np.clip(U, 1e-6, 1 - 1e-6)
    T = t_dist.ppf(U, df=nu)                           # a marginales t(nu) correlacionadas via copula gaussiana
    shocks = T * scale_indep[np.newaxis, :]
    d_paths_gausscop[s, :] = simulate_path_from_shocks(shocks, alpha, D_INITIAL, base_params, years)
p_gausscop = np.mean(d_paths_gausscop[:, -1] > 1.0)
print(f"[2] Copula Gaussiana + marginales t (correlacion sin exceso de tail dependence): {p_gausscop*100:.1f}%")

print()
print(f"Mediana 2035 -- ref: {np.median(d_paths_ref[:,-1])*100:.1f}%  indep: {np.median(d_paths_indep[:,-1])*100:.1f}%  gausscop: {np.median(d_paths_gausscop[:,-1])*100:.1f}%")

# Persistencia a CSV para trazabilidad (mismo criterio que el resto de las
# fases: todo resultado intermedio citado en la tesis debe quedar en
# resultados/tablas/, no solo impreso por consola).
import pandas as pd
tabla = pd.DataFrame([
    {"especificacion": "Copula-t correlacionada (reportada, Tabla 7.3)",
     "prob_insolvencia_2035": p_ref * 100, "mediana_2035": np.median(d_paths_ref[:, -1]) * 100},
    {"especificacion": "Copula Gaussiana + marginales t (correlacion sin exceso de tail dependence)",
     "prob_insolvencia_2035": p_gausscop * 100, "mediana_2035": np.median(d_paths_gausscop[:, -1]) * 100},
    {"especificacion": "Marginales t independientes (sin correlacion, sin tail dependence)",
     "prob_insolvencia_2035": p_indep * 100, "mediana_2035": np.median(d_paths_indep[:, -1]) * 100},
])
tabla.to_csv("resultados/tablas/fase14_dsa_tail_dependence.csv", index=False)
print("\nGuardado: resultados/tablas/fase14_dsa_tail_dependence.csv")


### Etapa 2.15 — Descomposición ilustrativa del Ajuste Stock-Flujo

Aproxima la descomposición de la identidad de movimiento de la deuda ($SF_t$) para los episodios de 2005, 2018 y 2020.

Fuente: `codigo/modelos/fase15_sft_decomposicion_ilustrativa.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/modelos/fase15_sft_decomposicion_ilustrativa.py` desde la raíz del proyecto.
__file__ = "codigo/modelos/fase15_sft_decomposicion_ilustrativa.py"

"""
H3 (optima) -- Descomposicion ilustrativa del Ajuste Stock-Flujo (SF_t) para
2005, 2018 y 2020, usando la identidad d_t = (1+r_t)/(1+g_t) d_{t-1} - pb_t + SF_t.

Fuentes:
 - d_t, d_{t-1}, pb_t: dataset propio de la tesis (deuda_consolidada_pib, pb_pib),
   datos/dataset_consolidado_real.csv.
 - r_t (tasa de interes real efectiva): APROXIMACION construida en tres pasos a
   partir de series del IMF WEO DataMapper (perimetro "general government", NO
   idéntico al perimetro SPNF+BCRA de la tesis):
     1. interes_t (%PIB) = balance_primario_IMF_t - balance_global_IMF_t
     2. tasa nominal implicita_t = interes_t / deuda_IMF_{t-1}
     3. tasa real_t = Fisher[(1+tasa nominal_t)/(1+inflacion_IMF_t)] - 1
 - g_t (crecimiento real del PIB): PIB_real (indice), dataset propio de la tesis.

Esto es una aproximacion ilustrativa, NO una serie de tasa de interes real
propia de la tesis: mezcla el perimetro de deuda "general government" del FMI
(para estimar la tasa implicita) con el perimetro SPNF+BCRA propio de la tesis
(para d_t, d_{t-1}, pb_t). Se documenta explicitamente como limitacion.
"""
import json
import pathlib
import pandas as pd

BASE_DIR = pathlib.Path(__file__).parent.parent.parent
IMF_DIR = BASE_DIR / "Bibliografia" / "descargas_verificacion" / "imf_weo_datamapper_ARG"

overall = json.load(open(IMF_DIR / "GGXCNL_NGDP_overall_balance.json"))["values"]["GGXCNL_NGDP"]["ARG"]
primary = json.load(open(IMF_DIR / "pb_primary_balance.json"))["values"]["pb"]["ARG"]
debt_imf = json.load(open(IMF_DIR / "GGXWDG_NGDP_gross_debt.json"))["values"]["GGXWDG_NGDP"]["ARG"]
infl = json.load(open(IMF_DIR / "PCPIPCH_inflation.json"))["values"]["PCPIPCH"]["ARG"]

df = pd.read_csv(BASE_DIR / "datos" / "dataset_consolidado_real.csv", parse_dates=['Date'], index_col='Date')

def annual_d(year):
    return df.loc[f"{year}-12-31", "deuda_consolidada_pib"] / 100

def annual_pb(year):
    return df.loc[f"{year}", "pb_pib"].sum() / 100

def annual_g_real(year):
    pib_t = df.loc[f"{year}", "PIB_real"].sum()
    pib_t1 = df.loc[f"{year-1}", "PIB_real"].sum()
    return pib_t / pib_t1 - 1

print(f"{'Anio':<6}{'d_t':>8}{'d_t-1':>8}{'pb_t':>8}{'i_nom(FMI)':>12}{'infl(FMI)':>10}{'r_real':>9}{'g_real':>9}{'SF_t implicito':>16}")
resultados = {}
for year in [2005, 2018, 2020]:
    d_t = annual_d(year)
    d_t1 = annual_d(year - 1)
    pb_t = annual_pb(year)
    interes_pib = primary[str(year)] - overall[str(year)]
    i_nom = interes_pib / debt_imf[str(year - 1)]
    pi = infl[str(year)] / 100
    r_real = (1 + i_nom) / (1 + pi) - 1
    g_real = annual_g_real(year)
    sf_t = d_t - (1 + r_real) / (1 + g_real) * d_t1 + pb_t
    resultados[year] = dict(d_t=d_t, d_t1=d_t1, pb_t=pb_t, i_nom=i_nom, pi=pi, r_real=r_real, g_real=g_real, sf_t=sf_t)
    print(f"{year:<6}{d_t*100:>7.1f}%{d_t1*100:>7.1f}%{pb_t*100:>7.1f}%{i_nom*100:>11.1f}%{pi*100:>9.1f}%{r_real*100:>8.1f}%{g_real*100:>8.1f}%{sf_t*100:>15.1f}%")

print()
print("Nota: SF_t > 0 significa que la deuda crecio mas de lo que explican el diferencial r-g y el resultado primario (efecto de valuacion/reconocimiento neto positivo sobre el stock).")

# Persistencia a CSV para trazabilidad (mismo criterio que el resto de las
# fases: todo resultado intermedio citado en la tesis debe quedar en
# resultados/tablas/, no solo impreso por consola).
tabla = pd.DataFrame.from_dict(resultados, orient="index")
tabla.index.name = "anio"
tabla.to_csv("resultados/tablas/fase15_sft_decomposicion_ilustrativa.csv")
print("\nGuardado: resultados/tablas/fase15_sft_decomposicion_ilustrativa.csv")


### Etapa 3.1 — Figuras principales

Genera las figuras de deuda/resultado primario, dispersión de fatiga fiscal y diagnóstico de primera etapa del IV-2SLS.

Fuente: `codigo/graficos/generacion_graficos_tesis.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/graficos/generacion_graficos_tesis.py` desde la raíz del proyecto.
__file__ = "codigo/graficos/generacion_graficos_tesis.py"

"""
generacion_graficos_tesis.py
============================
Genera, a partir de los datos reales del proyecto (datos/dataset_consolidado_real.csv
y el protocolo econométrico de codigo/modelos/), las versiones rediseñadas de las
Figuras 5.1 y 5.3, y la nueva Figura 6.1 del diagnóstico de instrumentos de la
estimación IV-2SLS (Capítulo 6, Sección 6.3).

Estilo visual: matplotlib + seaborn ('whitegrid'), paleta institucional
(azul marino para las series de solvencia, gris/rojo apagado para las series
de contraste), en línea con las publicaciones de organismos multilaterales
(FMI, Banco Mundial).

Todas las cifras se recalculan aquí directamente desde el dataset consolidado
real del proyecto; ninguna serie ni estadístico es inventado o aproximado.

Salidas (en tesis/figuras/):
  - fig5_1_deuda_resultado_primario.png
  - fig5_3_dispersion_fatiga_fiscal.png
  - fig6_1_diagnostico_primera_etapa.png
"""

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# ---------------------------------------------------------------------------
# Estilo institucional (tipo FMI / Banco Mundial)
# ---------------------------------------------------------------------------
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "axes.edgecolor": "#4d4d4d",
    "axes.labelcolor": "#1a1a1a",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.color": "#d9d9d9",
    "grid.linewidth": 0.6,
    "legend.frameon": False,
})

NAVY = "#0B3D66"      # Deuda / series principal de solvencia
GREY_RED = "#B04A4A"  # Resultado primario / series de contraste
TEAL = "#1C7C74"      # Elementos auxiliares (línea de ajuste, referencia)
LIGHT_GREY = "#8c8c8c"

BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
DATA_PATH = os.path.join(BASE_DIR, "datos", "dataset_consolidado_real.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "tesis", "figuras")

REGIME_BOUNDARIES = [2004, 2012, 2018, 2021, 2026]
REGIME_LABELS = [
    "2004-2011\nDescompresión",
    "2012-2017\nDeterioro gradual",
    "2018-2020\nCrisis y FMI",
    "2021-2025\nConsolidación",
]


def load_data():
    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df["year"] = df["Date"].dt.year
    return df


def fig_5_1_deuda_resultado_primario(df):
    """Figura 5.1 (rediseñada): Deuda/PIB vs Resultado Primario, ejes duales,
    con líneas verticales delimitando los cuatro regímenes macrofiscales."""
    annual = df.groupby("year").agg(deuda_pib=("deuda_pib", "mean"),
                                     pb_pib=("pb_pib", "mean")).reset_index()

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()

    ax1.plot(annual["year"], annual["deuda_pib"], color=NAVY, marker="o",
              markersize=5, linewidth=2.2, label="Deuda Pública Consolidada / PIB (%)")
    ax2.plot(annual["year"], annual["pb_pib"], color=GREY_RED, marker="s",
              markersize=5, linewidth=1.8, linestyle="--",
              label="Resultado Primario / PIB (%)")

    ax1.set_xlabel("Año")
    ax1.set_ylabel("Deuda Pública Consolidada / PIB (%)", color=NAVY, fontweight="bold")
    ax2.set_ylabel("Resultado Primario / PIB (%)", color=GREY_RED, fontweight="bold")
    ax1.tick_params(axis="y", colors=NAVY)
    ax2.tick_params(axis="y", colors=GREY_RED)
    ax2.grid(False)

    # Regímenes: líneas verticales punteadas en los límites + sombreado alterno
    for boundary in REGIME_BOUNDARIES[1:-1]:
        ax1.axvline(boundary - 0.5, color=LIGHT_GREY, linestyle=":", linewidth=1.3, zorder=1)

    shade_colors = ["#eef2f7", "#ffffff", "#eef2f7", "#ffffff"]
    ymin, ymax = ax1.get_ylim()
    for i in range(len(REGIME_BOUNDARIES) - 1):
        start, end = REGIME_BOUNDARIES[i] - 0.5, REGIME_BOUNDARIES[i + 1] - 0.5
        ax1.axvspan(start, end, color=shade_colors[i], zorder=0, alpha=0.6)
        mid = (start + end) / 2
        ax1.text(mid, ymax - (ymax - ymin) * 0.04, REGIME_LABELS[i],
                  ha="center", va="top", fontsize=8.5, color="#333333")
    ax1.set_ylim(ymin, ymax)
    ax1.set_xlim(REGIME_BOUNDARIES[0] - 0.5, REGIME_BOUNDARIES[-1] - 1.5)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper center",
               bbox_to_anchor=(0.5, -0.12), ncol=2)

    ax1.set_title("Dinámica Macrofiscal Agregada: Deuda Consolidada y Resultado Primario (2004-2025)")
    fig.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "fig5_1_deuda_resultado_primario.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Figura 5.1 guardada en {out_path}")


def fig_5_3_dispersion_fatiga_fiscal(df):
    """Figura 5.3 (rediseñada): dispersión completa (d_{t-1}, pb_t) con ajuste
    polinómico de segundo grado e intervalo de confianza sombreado (95%)."""
    d = df.copy()
    d["d_lag1"] = d["deuda_pib"].shift(1)
    d = d.dropna(subset=["d_lag1", "pb_pib"])

    fig, ax = plt.subplots(figsize=(9, 6.5))
    sns.regplot(
        x="d_lag1", y="pb_pib", data=d, order=2, ci=95, ax=ax,
        scatter_kws={"color": NAVY, "alpha": 0.65, "s": 32, "edgecolor": "white", "linewidths": 0.4},
        line_kws={"color": GREY_RED, "linewidth": 2.2},
    )
    ax.set_xlabel(r"Ratio Deuda Pública / PIB rezagada ($d_{t-1}$, %)")
    ax.set_ylabel(r"Resultado Primario / PIB ($pb_t$, %)")
    ax.set_title("Dispersión Empírica: Esfuerzo Primario vs. Endeudamiento Heredado\n(ajuste polinómico de 2do grado, IC 95%, panel completo n={})".format(len(d)))
    ax.axhline(0, color="#999999", linewidth=0.8, linestyle="-")

    fig.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "fig5_3_dispersion_fatiga_fiscal.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Figura 5.3 guardada en {out_path}")


def fig_6_1_diagnostico_primera_etapa(df):
    """Figura 6.1: proyección de la primera etapa del IV-2SLS (EMBI+ real vs.
    EMBI+ ajustado por d_{t-1}, brecha del producto, VIX y EMBI_BRASIL -spread
    soberano regional, Mejora Dimensión III, en reemplazo del TCRM_{t-1}
    original, rechazado por Sargan-), con el estadístico F de relevancia de
    primera etapa."""
    d = df.copy()
    d["d_t_1"] = d["deuda_pib"].shift(1)
    subset_cols = ["pb_pib", "d_t_1", "g_gap", "EMBI", "VIX", "EMBI_BRASIL"]
    d = d.dropna(subset=subset_cols)

    # Valores ajustados de la primera etapa (idénticos bajo OLS clásico o robusto)
    y = d["EMBI"]
    X_full = sm.add_constant(d[["d_t_1", "g_gap", "VIX", "EMBI_BRASIL"]])
    full_model = sm.OLS(y, X_full).fit()
    fitted = full_model.fittedvalues

    # Estadístico F de relevancia de primera etapa (HAC/kernel), replicando
    # exactamente el diagnóstico de codigo/modelos/fase4_variables_instrumentales.py
    iv_exog = sm.add_constant(d[["d_t_1", "g_gap"]])
    iv_endog = d[["EMBI"]]
    iv_instr = d[["VIX", "EMBI_BRASIL"]]
    iv_res = IV2SLS(dependent=d["pb_pib"], exog=iv_exog, endog=iv_endog,
                     instruments=iv_instr).fit(cov_type="kernel", kernel="newey-west")
    f_stat = iv_res.first_stage.diagnostics.loc["EMBI", "f.stat"]
    f_pval = iv_res.first_stage.diagnostics.loc["EMBI", "f.pval"]
    sargan = iv_res.sargan

    fig, ax = plt.subplots(figsize=(8.5, 7))
    ax.scatter(fitted, y, color=NAVY, alpha=0.65, s=34, edgecolor="white", linewidth=0.4,
               label="Observaciones trimestrales (n={})".format(len(d)))

    lo = min(fitted.min(), y.min())
    hi = max(fitted.max(), y.max())
    ax.plot([lo, hi], [lo, hi], color=LIGHT_GREY, linestyle=":", linewidth=1.4,
            label="Referencia 45° (ajuste perfecto)")

    fit_line = np.polyfit(fitted, y, 1)
    xs = np.linspace(lo, hi, 100)
    ax.plot(xs, fit_line[0] * xs + fit_line[1], color=GREY_RED, linewidth=2.2,
            label="Recta de ajuste (EMBI+ real ~ EMBI+ ajustado)")

    ax.set_xlabel(r"EMBI+ ajustado por $d_{t-1}$, brecha del producto, VIX y EMBI$_{Brasil}$ (primera etapa)")
    ax.set_ylabel("EMBI+ real (puntos básicos)")
    ax.set_title("Diagnóstico de Primera Etapa: EMBI+ Real vs. Ajustado\n(instrumentos: VIX, EMBI$_{Brasil}$)")
    ax.legend(loc="upper left", fontsize=9)

    textbox = (
        f"F (relevancia, instrumentos excluidos) = {f_stat:.2f}\n"
        f"$p$-valor < 0.001\n"
        f"Umbral de referencia (Staiger-Stock) = 10\n"
        f"Sargan (sobreidentificación) = {sargan.stat:.3f}, $p={sargan.pval:.3f}$"
    )
    ax.text(0.98, 0.03, textbox, transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9.5, family="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f7f7f7", edgecolor="#4d4d4d"))

    fig.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "fig6_1_diagnostico_primera_etapa.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Figura 6.1 guardada en {out_path}")
    print(f"     -> F={f_stat:.4f}  p={f_pval:.6f}  (n={len(d)})")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    df = load_data()
    fig_5_1_deuda_resultado_primario(df)
    fig_5_3_dispersion_fatiga_fiscal(df)
    fig_6_1_diagnostico_primera_etapa(df)


if __name__ == "__main__":
    main()


### Etapa 3.2 — Cronología de quiebres estructurales

Genera la línea de tiempo de los quiebres de Bai-Perron sobre la serie de deuda.

Fuente: `codigo/graficos/generacion_cronologia_quiebres.py`

In [ ]:
# __file__ no está definido dentro de una celda de Jupyter; se fija aquí
# explícitamente para que el código de más abajo -- copiado sin modificar
# del script original -- calcule las mismas rutas relativas que al correr
# `python codigo/graficos/generacion_cronologia_quiebres.py` desde la raíz del proyecto.
__file__ = "codigo/graficos/generacion_cronologia_quiebres.py"

"""
generacion_cronologia_quiebres.py
=================================
Linea de tiempo de quiebres estructurales multiples detectados de forma real
sobre la serie de Resultado Primario / PIB (pb_t), mediante el algoritmo de
Programacion Dinamica con costo L2 del paquete `ruptures` (Killick et al.,
2012; Truong, Oudre y Vayatis, 2020), en el espiritu del enfoque de particion
optima de quiebres multiples de Bai y Perron (2003).

Nota metodologica: este script NO reproduce el procedimiento secuencial de
contrastes F con errores robustos propuesto originalmente por Bai y Perron
(2003) -esa implementacion formal permanece como agenda de investigacion
futura, segun se documenta en el Capitulo 4-. Lo que aqui se reporta es una
deteccion exploratoria y realmente computada de quiebres multiples sobre la
serie real del proyecto (datos/dataset_consolidado_real.csv), util como
evidencia complementaria a la prueba de quiebre unico de Zivot-Andrews.

Salida: tesis/figuras/figura_5_4_quiebres_timeline.png
"""

import os

import matplotlib.pyplot as plt
import pandas as pd
import ruptures as rpt

BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
DATA_PATH = os.path.join(BASE_DIR, "datos", "dataset_consolidado_real.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "tesis", "figuras")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
})

EVENT_LABELS = {
    "2008-09": "Quiebre I: fin del ciclo de superávits\ny crisis financiera internacional",
    "2013-06": "Quiebre II: profundización del\ncontrol de cambios y estancamiento",
    "2019-03": "Quiebre III: recesión post-crisis\ncambiaria y programa con el FMI",
    "2024-03": "Quiebre IV: ajuste fiscal\nabrupto (shock de superávit)",
}


def detect_breaks(series, n_breaks=4):
    algo = rpt.Dynp(model="l2", min_size=6, jump=1).fit(series.values)
    bkps = algo.predict(n_bkps=n_breaks)
    return bkps[:-1]  # el último índice que devuelve ruptures es el fin de la serie, no un quiebre


def main():
    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    bkps_idx = detect_breaks(df["pb_pib"], n_breaks=4)
    break_dates = [df.loc[idx, "Date"] for idx in bkps_idx]

    print("Quiebres estructurales detectados (Programación Dinámica, costo L2):")
    for d in break_dates:
        print(" ->", d.strftime("%Y-%m"))

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(pd.to_datetime(["2004-01-01", "2025-12-31"]), [0, 0], color="#2C3E50", linewidth=2, zorder=1)

    levels = [1, -1, 1.2, -1.2]
    for i, d in enumerate(break_dates):
        key = d.strftime("%Y-%m")
        label = EVENT_LABELS.get(key, f"Quiebre detectado\n{key}")
        ax.scatter(d, 0, color="#D9534F", s=100, zorder=2, edgecolor="black")
        ax.vlines(d, 0, levels[i], color="#D9534F", linestyle="--", linewidth=1)
        ax.text(d, levels[i] + (0.05 if levels[i] > 0 else -0.15), label,
                horizontalalignment="center", verticalalignment="center",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="#F8F9F9", edgecolor="gainsboro", alpha=0.9),
                fontsize=9, fontname="serif")

    ax.set_xlim(pd.to_datetime("2003-01-01"), pd.to_datetime("2026-12-31"))
    ax.set_ylim(-1.8, 1.8)
    ax.yaxis.grid(False)
    ax.xaxis.grid(True, linestyle=":", alpha=0.6)
    for spine in ["left", "right", "top"]:
        ax.spines[spine].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_xlabel("Año")

    fig.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "figura_5_4_quiebres_timeline.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"\n[OK] Figura guardada en {out_path}")


if __name__ == "__main__":
    main()


## Cierre

Este notebook reprodujo, celda por celda y con el código fuente íntegro visible, el mismo proceso empírico documentado en el Capítulo de Metodología y en el Apéndice de la tesis. Al finalizar, `datos/dataset_consolidado_real.csv`, `resultados/tablas/*.csv` y `tesis/figuras/*.png` quedan actualizados. Ver también `notebook_analisis_completo.ipynb` para una ejecución liviana que llama a los scripts canónicos de `codigo/` sin duplicar su código.